# Process Experimental Data

This notebook parses the messy PCIbex results file `results_prod.csv`, which contains raw experimental results, and produces an event-level dataframe. We also parse the eye-tracking data files and merge them with the behavioral data.

## Setup

### Libraries

In [1]:
# Imports and paths
import re
import os
from glob import glob
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
import plotly.express as px

ROOT = Path(r"c:\\Users\\parti\\Projects\\hungarian-focus-exhaustivity")
# Run the notebook first with sesults_prod, then you can use results_prod_runids.csv
raw_file = ROOT / "results_prod.csv"
plots = ROOT / "plots"
plots.mkdir(exist_ok=True)

print("Will parse:", raw_file)

Will parse: c:\Users\parti\Projects\hungarian-focus-exhaustivity\results_prod.csv


### Themes

In [2]:
# Nord-themed Plotly template (light, printer-friendly, nord colorway)
import plotly.io as pio
import plotly.graph_objects as go

nord = ['#5e81ac', '#a3be8c', '#ebcb8b', '#d19a66', '#bf616a', "#b97ec9",
        '#4c566a', '#8fbcbb', '#88c0d0', '#81a1c1']
dark = ['#2e3440', '#3b4252', '#434c5e', '#4c566a']
snow = ['#2e3440', '#3b4252', '#4c566a']  # darker text on white for print
transparent = 'rgba(0,0,0,0)'

# Global serif stack (Times New Roman first)
SERIF_FAMILY = "Times New Roman, Times, Georgia, serif"

# -----------------------------
# Global font size control (edit these)
# -----------------------------
BASE_FONT_SIZE = 16   # <- change this to resize fonts everywhere
FONT_SCALE = 1.0      # <- optional multiplier (e.g., 1.15 for larger print)

# Derived sizes (keeps a consistent hierarchy)
FONT_MAIN = int(round(BASE_FONT_SIZE * FONT_SCALE))
FONT_TITLE = int(round((BASE_FONT_SIZE + 4) * FONT_SCALE))
FONT_LEGEND = int(round(BASE_FONT_SIZE * FONT_SCALE))
FONT_AXIS_TITLE = int(round(BASE_FONT_SIZE * FONT_SCALE))
FONT_TICKS = int(round((BASE_FONT_SIZE - 2) * FONT_SCALE))
FONT_HOVER = int(round((BASE_FONT_SIZE - 2) * FONT_SCALE))
FONT_ANNOT = int(round((BASE_FONT_SIZE + 2) * FONT_SCALE))

nord_template = go.layout.Template()

# general layout – white background, printer friendly
nord_template.layout.paper_bgcolor = "white"
nord_template.layout.plot_bgcolor = "white"
nord_template.layout.colorway = nord
nord_template.layout.font = dict(color=snow[0], family=SERIF_FAMILY, size=FONT_MAIN)
nord_template.layout.title = dict(font=dict(color=snow[0], family=SERIF_FAMILY, size=FONT_TITLE))
nord_template.layout.legend = dict(
    font=dict(color=snow[0], family=SERIF_FAMILY, size=FONT_LEGEND),
    title=dict(font=dict(color=snow[0], family=SERIF_FAMILY, size=FONT_LEGEND)),
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="rgba(0,0,0,0.1)",
    borderwidth=0
)
nord_template.layout.hoverlabel = dict(
    font=dict(color=snow[0], family=SERIF_FAMILY, size=FONT_HOVER)
)

# axis defaults – light gridlines, high contrast axes
axis_defaults = dict(
    title=dict(font=dict(color=snow[0], family=SERIF_FAMILY, size=FONT_AXIS_TITLE)),
    tickfont=dict(color=snow[0], family=SERIF_FAMILY, size=FONT_TICKS),
    gridcolor='rgba(0,0,0,0.08)',
    zerolinecolor='rgba(0,0,0,0.12)',
    linecolor='rgba(0,0,0,0.4)'
)
nord_template.layout.xaxis = axis_defaults
nord_template.layout.yaxis = axis_defaults

# annotation defaults (facet titles etc.)
nord_template.layout.annotationdefaults = dict(
    font=dict(color=snow[1], family=SERIF_FAMILY, size=FONT_ANNOT),  # Subplot titles
    bgcolor="rgba(255,255,255,0.0)",
)

# thinner default lines/markers for print
nord_template.data.scatter = [go.Scatter(
    line=dict(width=1.5),
    marker=dict(size=5)
)]

base_layout = dict(
    template='nord_light_paper',
    width=1000,
    height=400,
    margin=dict(l=40, r=20, t=60, b=40),
)

# register template
pio.templates['nord_light_paper'] = nord_template
# optional: make it default
pio.templates.default = 'nord_light_paper'

## Load data

In [3]:
# Read file, split comments vs data
raw_lines = raw_file.read_text(encoding="utf-8").splitlines()

# # Replace specific IDs so that the same participant doing different groups is distinguishable
# for i, line in enumerate(raw_lines):
#     if i > 5000:
#         raw_lines[i] = line.replace("6108da57e362f96a3ee32a88", "6108da57e362f96a3ee32a88_2")

# for i, line in enumerate(raw_lines):
#     if i > 5000:
#         raw_lines[i] = line.replace("5dade76a4860f70017f70ec5", "5dade76a4860f70017f70ec5_2")
        
# for i, line in enumerate(raw_lines):
#     if i > 39000:
#         raw_lines[i] = line.replace("6150053fddd36a6892b8f13c", "6150053fddd36a6892b8f13c_2")

for i, line in enumerate(raw_lines):
    if i > 54675:
        raw_lines[i] = line.replace("60cb4cd6477b2ff7c1adaea4", "60cb4cd6477b2ff7c1adaea4_3")
        
for i, line in enumerate(raw_lines):
    if i > 46872:
        raw_lines[i] = line.replace("5f4e8bbd350d2a08d6175762", "5f4e8bbd350d2a08d6175762_2")

# NOTE: manual ID rewrites removed. We will detect repeated runs by ResultsTime
# and create run-specific participant ids later in the parsing pipeline.

header_comments = []
rows = []
for line in raw_lines:
    if line.startswith("#"):
        header_comments.append(line)
    elif line.strip():
        rows.append(line)

print(f"Comment lines: {len(header_comments)} | Data rows: {len(rows)}")
rows[:3]

Comment lines: 3792 | Data rows: 58780


['1758558839,5ea96059c7f8c9bb71d4ecdb67ea496e,PennController,0,0,welcome,NULL,PennController,0,_Trial_,Start,1758556961557,prolific_id,NULL',
 '1758558839,5ea96059c7f8c9bb71d4ecdb67ea496e,PennController,0,0,welcome,NULL,PennController,0,_Header_,Start,1758556961557,prolific_id,NULL',
 '1758558839,5ea96059c7f8c9bb71d4ecdb67ea496e,PennController,0,0,welcome,NULL,PennController,0,_Header_,End,1758556961557,prolific_id,NULL']

## Data processing

In [4]:
# Define the base schema described by comments just before each block
base_cols = [
    "ResultsTime", "MD5", "Controller", "Order", "Inner", "Label",
    "LatinSquare", "PennElementType", "PennElementName", "Parameter",
    "Value", "EventTime", "prolific_id",
]

def extract_trial_fields_from_comments(comments):
    # We expect something like a line that mentions these names.
    allowed = ["participant_id", "group", "no", "item", "exp", "condition", "cb", "left", "right", "target"]
    # accept common variants (participant-id, participant id, participantid, prolific_id etc.)
    pattern = re.compile(r"(prolific[_\-\s]?id|participant[_\-\s]?id|participantid|group|no|item|exp|condition|cb|left|right|target)", re.I)

    best_line = None
    max_hits = 0
    # prefer the comment line that contains the most known tokens
    for c in comments:
        hits = len(pattern.findall(c))
        if hits > max_hits:
            best_line = c
            max_hits = hits

    # if we found a useful header line, try to parse explicit comma-separated tokens after a colon
    if best_line and max_hits >= 6:
        text = best_line.split(":", 1)[-1]
        tokens = [t.strip() for t in re.split(r"[,\t]+", text) if t.strip()]

        def norm(s):
            s2 = re.sub(r"[^A-Za-z0-9_]", "", s).lower()
            if s2 == "participantid":
                s2 = "participant_id"
            if s2 == "prolificid":
                s2 = "prolific_id"
            return s2

        names = []
        for t in tokens:
            n = norm(t)
            if n in allowed and n not in names:
                names.append(n)

        # fallback: if explicit tokens not present, use the order of regex matches in the line
        if not names:
            for m in pattern.finditer(best_line):
                n = norm(m.group(0))
                if n in allowed and n not in names:
                    names.append(n)

        # Preserve canonical order relative to allowed list
        ordered = [name for name in allowed if name in names]
        # accept extraction if we found at least participant_id + several others
        if "participant_id" in ordered and len(ordered) >= 6:
            return ordered

    # Final fallback: canonical full trial ordering (updated to include exp and target)
    return ["participant_id", "group", "no", "item", "exp", "condition", "cb", "left", "right", "target"]

TRIAL_FIELDS = extract_trial_fields_from_comments(header_comments)
print("Using trial fields:", TRIAL_FIELDS)

# Parse each data row into a record using base_cols + TRIAL_FIELDS depending on Label

def parse_row_to_record(line: str):
    parts = [p.strip() for p in line.split(",")]
    rec = {}
    base_vals = parts[:len(base_cols)]
    extra_vals = parts[len(base_cols):]

    for k, v in zip(base_cols, base_vals):
        rec[k] = v

    # If the extra values accidentally include a literal header token as the first element
    # (some CSV dumps include the header name), drop it so alignment works.
    if extra_vals and str(extra_vals[0]).lower() in {f.lower() for f in TRIAL_FIELDS + ["prolific_id"]}:
        # drop single leading header-like token
        extra_vals = extra_vals[1:]

    label = rec.get("Label")
    if label in ("practice", "experiment"):
        n = min(len(extra_vals), len(TRIAL_FIELDS))
        for k, v in zip(TRIAL_FIELDS[:n], extra_vals[:n]):
            rec[k] = v
    elif label in ("participant_data",):
        if extra_vals:
            # participant_data frequently stores id in first extra token (after possible header token)
            rec["participant_id"] = extra_vals[0]

    return rec

records = [parse_row_to_record(l) for l in rows]
raw_df = pd.DataFrame.from_records(records)

# Cast some known numeric columns where possible
for c in ["ResultsTime", "Order", "Inner", "EventTime", "no", "item"]:
    if c in raw_df.columns:
        raw_df[c] = pd.to_numeric(raw_df[c], errors="coerce")

# Human-readable timestamps
raw_df["results_time"] = pd.to_datetime(raw_df["ResultsTime"], unit="s", utc=True)

# Disambiguate repeated runs by the same participant using results_time
# Strategy: for each participant-key (participant_id, prolific_id or MD5) sort by results_time
# and start a new session when the gap to the previous results_time exceeds THRESH seconds.
# This produces a run-specific id like '<id>_2', '<id>_3' for subsequent runs.
try:
    KEY_CANDIDATES = ["participant_id", "prolific_id", "MD5"]
    key_col = next((c for c in KEY_CANDIDATES if c in raw_df.columns), None)
    if key_col is not None:
        # sort in-place for deterministic session detection
        raw_df = raw_df.sort_values([key_col, "results_time"]).reset_index(drop=True)
        # compute gap to previous results_time per participant key
        raw_df["_prev_results_time"] = raw_df.groupby(key_col)["results_time"].shift(1)
        raw_df["_gap_s"] = (raw_df["results_time"] - raw_df["_prev_results_time"]).dt.total_seconds().fillna(0)
        # threshold in seconds to start a new run (1 hour by default)
        THRESH = 3600
        raw_df["_new_session"] = (raw_df["_gap_s"] > THRESH).astype(int)
        # cumulative session index per participant (1-based)
        raw_df["_session_idx"] = raw_df.groupby(key_col)["_new_session"].cumsum() + 1
        # build a run-specific participant id (append _{idx} for idx>1)
        def _make_run_id(r):
            base = r.get(key_col)
            if pd.isna(base):
                return base
            idx = int(r.get("_session_idx", 1))
            return f"{base}" if idx <= 1 else f"{base}_{idx}"
        raw_df["participant_run_id"] = raw_df.apply(_make_run_id, axis=1)
        # adopt participant_run_id as the canonical participant_id used downstream
        raw_df["participant_id"] = raw_df["participant_run_id"]
    else:
        # no suitable key column found; leave participant_id untouched
        pass
except Exception:
    # If anything goes wrong here, fall back to leaving ids unchanged
    import traceback; traceback.print_exc()

# IMPORTANT: Keep EventTime both as numeric milliseconds and as a proper timestamp (UTC)
# - EventTime_ms: numeric milliseconds for computations (diffs, means)
# - event_time: Pandas Timestamp (UTC) for readability
raw_df["event_time_ms"] = raw_df["EventTime"]
raw_df["event_time"] = pd.to_datetime(raw_df["event_time_ms"], unit="ms", utc=True)

################################################

# Helper: derive per-trial fields now that we named extras explicitly in raw_df
# We keep a light decoder only to ensure participant_id is present and to normalize types.

EXPECTED_FIELDS = ["participant_id", "group", "no", "item", "exp", "condition", "cb", "left", "right", "target"]

# Make sure that item and no are integers
for col in ["no", "item"]:
    if col in raw_df.columns:
        # Use pandas nullable Int64 type to keep NaNs and force integer dtype
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce").astype('Int64')
        
# Ensure all expected columns exist even if missing in some rows
for col in EXPECTED_FIELDS:
    if col not in raw_df.columns:
        raw_df[col] = None

# Build df and forward fill participant_id/group as before
df = raw_df.copy()

# Forward-fill participant_id only within blocks of identical participant_id (no cross-over)
# (keep as-is: do not fill across participants)
if "participant_id" in df.columns:
    # nothing to do beyond keeping column present; participant ids should appear in rows where provided

    # If you need to forward-fill within contiguous blocks uncomment:
    # df['participant_id'] = df['participant_id'].ffill()

    pass

# Backward-fill group only within blocks of identical participant_id (no cross-over)
if "group" in df.columns and "participant_id" in df.columns:
    # normalize empty/NULL then backfill per participant using transform to avoid groupby.apply deprecation
    df["group"] = df["group"].replace({"": None, "NULL": None})
    df["group"] = df.groupby("participant_id")["group"].transform(lambda s: s.bfill())

# Derive block-type flags
# df["is_practice"] = df["Label"].eq("practice")
# df["is_experiment"] = df["Label"].eq("experiment")

# For convenience: also include a local-time copy if desired (commented)
# df["results_timestamp_local"] = df["results_timestamp"].dt.tz_convert("Europe/Budapest")

# Drop Controller column
df.drop(columns=["ResultsTime", "Controller", "Inner", "LatinSquare", "EventTime", "prolific_id"], inplace=True)


# Drop rows where the Parameter column starts and ends with '_'
df = df[~df["Parameter"].str.match(r"^_.*_$", na=False)]

# Remove rows where PennElementType is "Canvas"
df = df[df["PennElementType"] != "Canvas"]

# Add simple elapsed time between events: current row EventTime_ms minus previous row EventTime_ms
if 'elapsed_ms' in df.columns:
    df.drop(columns=['elapsed_ms'], inplace=True)
df['elapsed_ms'] = df['event_time_ms'].diff()
df.drop(columns=['event_time_ms'], inplace=True)

# Keep only rows where label is practice or experiment
df = df[df["Label"].isin(["practice", "experiment"])]

# In Values column rename right_canvas_practice to right_canvas
df["Value"] = df["Value"].replace({"right_canvas_practice": "right_canvas"})
df["Value"] = df["Value"].replace({"left_canvas_practice": "left_canvas"})

# Rename Label to label and Order to order
df.rename(columns={"Label": "label", "Order": "trial"}, inplace=True)

# Fix mistakes in stimuli data ##########################
# Change condition of sentences no 74 to other-directed-x
df.loc[df['no'] == 74, 'condition'] = 'other-directed-x'
df

Using trial fields: ['participant_id', 'group', 'no', 'item', 'exp', 'condition', 'cb', 'left', 'right', 'target']


,MD5,trial,label,PennElementType,PennElementName,Parameter,Value,participant_id,group,no,...,right,target,results_time,_prev_results_time,_gap_s,_new_session,_session_idx,participant_run_id,event_time,elapsed_ms
10,0815250ff86d7adf38545faa5f20e73f,6,practice,EyeTracker,tracker,calibration,57,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:03:57.938000+00:00,285407.0
11,0815250ff86d7adf38545faa5f20e73f,6,practice,EyeTracker,tracker,Filename,httpsfarmpcibexnetpzCPVqO/437257d2-8424-bb79-8...,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:19.788000+00:00,21850.0
13,0815250ff86d7adf38545faa5f20e73f,6,practice,Key,r0,PressedKey,,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:06.652000+00:00,-13136.0
14,0815250ff86d7adf38545faa5f20e73f,6,practice,Key,r1,PressedKey,,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:08.312000+00:00,1660.0
15,0815250ff86d7adf38545faa5f20e73f,6,practice,Key,r2,PressedKey,,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:09.482000+00:00,1170.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57018,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r4,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:30.260000+00:00,353.0
57019,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r5,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:30.613000+00:00,353.0
57020,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r6,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:31.125000+00:00,512.0
57021,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r7,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:31.583000+00:00,458.0


In [5]:
# Write a backed-up copy of the raw results with run-specific participant ids
# (creates results_prod_runids.csv and a backup of the original file)
try:
    import shutil
    out_suffix = '_runids'
    out_path = raw_file.with_name(raw_file.stem + out_suffix + raw_file.suffix)
    backup_path = raw_file.with_name(raw_file.name + '.bak')
    shutil.copy2(raw_file, backup_path)

    # Reconstruct rows from raw_df using the canonical base columns + trial fields
    cols_order = base_cols + TRIAL_FIELDS
    out_lines = []
    # preserve header comment lines exactly as read earlier
    out_lines.extend(header_comments)

    # ensure raw_df iteration order matches original sort (if you prefer original file order
    # you can sort by ResultsTime or index)
    for _, r in raw_df.iterrows():
        parts = []
        for c in cols_order:
            v = r.get(c, '')
            # write empty string for NaN/None
            if pd.isna(v):
                parts.append('')
            else:
                parts.append(str(v))
        out_lines.append(','.join(parts))

    out_text = '\n'.join(out_lines)
    out_path.write_text(out_text, encoding='utf-8')
    print('Wrote updated results to', out_path)
    print('Original file backed up to', backup_path)
except Exception as e:
    print('Failed writing updated results:', e)
    import traceback; traceback.print_exc()

Wrote updated results to c:\Users\parti\Projects\hungarian-focus-exhaustivity\results_prod_runids.csv
Original file backed up to c:\Users\parti\Projects\hungarian-focus-exhaustivity\results_prod.csv.bak


In [6]:
df

,MD5,trial,label,PennElementType,PennElementName,Parameter,Value,participant_id,group,no,...,right,target,results_time,_prev_results_time,_gap_s,_new_session,_session_idx,participant_run_id,event_time,elapsed_ms
10,0815250ff86d7adf38545faa5f20e73f,6,practice,EyeTracker,tracker,calibration,57,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:03:57.938000+00:00,285407.0
11,0815250ff86d7adf38545faa5f20e73f,6,practice,EyeTracker,tracker,Filename,httpsfarmpcibexnetpzCPVqO/437257d2-8424-bb79-8...,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:19.788000+00:00,21850.0
13,0815250ff86d7adf38545faa5f20e73f,6,practice,Key,r0,PressedKey,,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:06.652000+00:00,-13136.0
14,0815250ff86d7adf38545faa5f20e73f,6,practice,Key,r1,PressedKey,,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:08.312000+00:00,1660.0
15,0815250ff86d7adf38545faa5f20e73f,6,practice,Key,r2,PressedKey,,5909f8b6bc185d00011aa307,b,990,...,B,A,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:04:09.482000+00:00,1170.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57018,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r4,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:30.260000+00:00,353.0
57019,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r5,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:30.613000+00:00,353.0
57020,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r6,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:31.125000+00:00,512.0
57021,bfc1095ef3646815b79cdf0fecd8ba35,49,experiment,Key,r7,PressedKey,,parti_test_01,c,21,...,7b,7b,2025-10-02 03:02:37+00:00,2025-10-02 03:02:37+00:00,0.0,0,1.0,parti_test_01,2025-10-02 03:02:31.583000+00:00,458.0


### Participants

In [7]:
def extract_participant_info(events_df, fallback_df=None):
    # Helper: return first non-empty normalized value for a column
    def pick_first(df_like, col):
        if df_like is None or col not in df_like.columns:
            return None
        s = df_like[col].replace({"": None, "NULL": None}).dropna().astype(str)
        return s.iloc[0] if len(s) > 0 else None

    # Extract participant_id, group, results_time (prefer events_df, fallback to fallback_df)
    pid = pick_first(events_df, "participant_id") or pick_first(fallback_df, "participant_id")
    group = pick_first(events_df, "group") or pick_first(fallback_df, "group")
    results_time = pick_first(events_df, "results_time") or pick_first(fallback_df, "results_time")

    # Exact requirement: Value where PennElementType == 'EyeTracker' and Parameter == 'Filename'
    def find_et_filename(df_like):
        if df_like is None or not {"PennElementType", "Parameter", "Value"}.issubset(df_like.columns):
            return None
        pet = df_like["PennElementType"].astype(str).str.lower()
        par = df_like["Parameter"].astype(str).str.lower()
        mask = pet.eq("eyetracker") & par.eq("filename")
        vals = df_like.loc[mask, "Value"].replace({"": None}).dropna().astype(str)
        return vals.iloc[0] if len(vals) > 0 else None

    et = find_et_filename(events_df) or find_et_filename(fallback_df)

    return pd.DataFrame([
        {
            "participant_id": pid,
            "group": group,
            "eyetracker_filename": et,
            "results_time": results_time,
        }
    ])

# Build participants_df for all participants (uses raw_df as fallback to find filename)
parts = []
df_nonnull = df[df['participant_id'].notna()]
for pid, g in df_nonnull.groupby('participant_id', sort=False):
    parts.append(extract_participant_info(g, fallback_df=raw_df))
participants_df = pd.concat(parts, ignore_index=True)

# Remove row with "parti_test_01"
participants_df = participants_df[participants_df['participant_id'] != 'parti_test_01']

participants_df

,participant_id,group,eyetracker_filename,results_time
0,5909f8b6bc185d00011aa307,b,httpsfarmpcibexnetpzCPVqO/437257d2-8424-bb79-8...,2025-12-08 13:22:10+00:00
1,59679c319febf80001d53655,a,httpsfarmpcibexnetpzCPVqO/0971ec97-7338-712f-4...,2025-10-08 11:45:02+00:00
2,5b93d1913dca6000012c5fdc,c,httpsfarmpcibexnetpzCPVqO/c7f73ec8-c254-ee9d-f...,2025-10-06 21:02:00+00:00
3,5bbc9e8c70f8df0001c0e060,a,httpsfarmpcibexnetpzCPVqO/00fb9fda-16ec-9c32-0...,2025-12-05 21:06:40+00:00
4,5c5c785fc9735b00010ced0b,a,httpsfarmpcibexnetpzCPVqO/de88e6e7-31f1-0112-1...,2025-10-07 09:30:23+00:00
...,...,...,...,...
73,6932c07f41479348b6b5d9fb_2,b,httpsfarmpcibexnetpzCPVqO/c973bae9-8c45-b7f8-e...,2025-12-11 13:09:24+00:00
74,69346587dfdffc1057144d32,b,httpsfarmpcibexnetpzCPVqO/5d7553c2-94a2-7e43-7...,2025-12-08 12:01:23+00:00
75,69346587dfdffc1057144d32_2,c,httpsfarmpcibexnetpzCPVqO/8edae79b-dead-14a8-a...,2025-12-10 10:32:31+00:00
76,693c768c138df103c91b6c22,c,httpsfarmpcibexnetpzCPVqO/36999898-e046-f51d-2...,2025-12-15 07:38:03+00:00


In [8]:
# count unique participants as for participant_id BEFORE the underscore run suffixes
participants_df['base_participant_id'] = participants_df['participant_id'].str.split('_').str[0]
num_unique_participants = participants_df['base_participant_id'].nunique()
num_unique_participants

52

In [9]:
# Checking
pid = "5f046bf88d2c186cc10c6ad0"
import re
pattern = re.escape(pid)
mask = participants_df['participant_id'].astype(str).str.contains(pattern, na=False)
participants_df[mask]

,participant_id,group,eyetracker_filename,results_time,base_participant_id
15,5f046bf88d2c186cc10c6ad0,c,httpsfarmpcibexnetpzCPVqO/c0118ee8-441f-f798-6...,2025-12-12 17:54:42+00:00,5f046bf88d2c186cc10c6ad0


### Remove participants

In [10]:
# Remove certain participants
# Too long times: 642b35c70771761602e9c3ae
# No ET data: 641379405684937e6fad9f1b
# Failed attention checks: 5f046bf88d2c186cc10c6ad0
# Duplicate test: 5f4c042383588080d02e61a3_2, 67698e94c727a4a942390c57_3, 5dade76a4860f70017f70ec5_2

list_of_participants_to_remove = ['parti_test_01', '642b35c70771761602e9c3ae', '641379405684937e6fad9f1b', '5f4c042383588080d02e61a3_2', '67698e94c727a4a942390c57_3', '5dade76a4860f70017f70ec5_2', '5f046bf88d2c186cc10c6ad0']

# Remove them from both dataframes
participants_df = participants_df[~participants_df['participant_id'].isin(list_of_participants_to_remove)]
df = df[~df['participant_id'].isin(list_of_participants_to_remove)]

# Count no of unique participants after removal
participants_df['base_participant_id'] = participants_df['participant_id'].str.split('_').str[0]
num_unique_participants_after_removal = participants_df['base_participant_id'].nunique()
num_unique_participants_after_removal

C:\Users\parti\AppData\Local\Temp\ipykernel_15204\480181518.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  participants_df['base_participant_id'] = participants_df['participant_id'].str.split('_').str[0]


49

### Anonymize IDs

In [11]:
# Get a list of all valid participants, and map them to anonymized ids
valid_participants = df['participant_id'].unique()
participant_id_map = {pid: f"participant_{i+1:03d}" for i, pid in enumerate(valid_participants)}
participant_id_map

{'5909f8b6bc185d00011aa307': 'participant_001',
 '59679c319febf80001d53655': 'participant_002',
 '5b93d1913dca6000012c5fdc': 'participant_003',
 '5bbc9e8c70f8df0001c0e060': 'participant_004',
 '5c5c785fc9735b00010ced0b': 'participant_005',
 '5cb9d45ac3eb6a00123c226e': 'participant_006',
 '5cb9d45ac3eb6a00123c226e_2': 'participant_007',
 '5cb9d45ac3eb6a00123c226e_3': 'participant_008',
 '5d4fe6a2ffbcf800019d5e54': 'participant_009',
 '5dade76a4860f70017f70ec5': 'participant_010',
 '5e3b29dc87243b34bde5abfa': 'participant_011',
 '5e3b29dc87243b34bde5abfa_2': 'participant_012',
 '5e3b29dc87243b34bde5abfa_3': 'participant_013',
 '5ee75f3d1a88450293a38aeb': 'participant_014',
 '5f3013e31c8a690aacb02c31': 'participant_015',
 '5f3013e31c8a690aacb02c31_2': 'participant_016',
 '5f3013e31c8a690aacb02c31_3': 'participant_017',
 '5f338ba6ea047119dbd6e49e': 'participant_018',
 '5f3ce934a12769b771503625': 'participant_019',
 '5f4c042383588080d02e61a3': 'participant_020',
 '5f4e8bbd350d2a08d6175762':

In [12]:
# Anonymize participant ids in df and participants_df using the participant_id_map
df = df.copy()
participants_df = participants_df.copy()

# map and preserve any ids that are not present in the map
df.loc[:, 'participant_id'] = df['participant_id'].map(participant_id_map).fillna(df['participant_id'])
participants_df.loc[:, 'participant_id'] = participants_df['participant_id'].map(participant_id_map).fillna(participants_df['participant_id'])

participants_df

,participant_id,group,eyetracker_filename,results_time,base_participant_id
0,participant_001,b,httpsfarmpcibexnetpzCPVqO/437257d2-8424-bb79-8...,2025-12-08 13:22:10+00:00,5909f8b6bc185d00011aa307
1,participant_002,a,httpsfarmpcibexnetpzCPVqO/0971ec97-7338-712f-4...,2025-10-08 11:45:02+00:00,59679c319febf80001d53655
2,participant_003,c,httpsfarmpcibexnetpzCPVqO/c7f73ec8-c254-ee9d-f...,2025-10-06 21:02:00+00:00,5b93d1913dca6000012c5fdc
3,participant_004,a,httpsfarmpcibexnetpzCPVqO/00fb9fda-16ec-9c32-0...,2025-12-05 21:06:40+00:00,5bbc9e8c70f8df0001c0e060
4,participant_005,a,httpsfarmpcibexnetpzCPVqO/de88e6e7-31f1-0112-1...,2025-10-07 09:30:23+00:00,5c5c785fc9735b00010ced0b
...,...,...,...,...,...
73,participant_068,b,httpsfarmpcibexnetpzCPVqO/c973bae9-8c45-b7f8-e...,2025-12-11 13:09:24+00:00,6932c07f41479348b6b5d9fb
74,participant_069,b,httpsfarmpcibexnetpzCPVqO/5d7553c2-94a2-7e43-7...,2025-12-08 12:01:23+00:00,69346587dfdffc1057144d32
75,participant_070,c,httpsfarmpcibexnetpzCPVqO/8edae79b-dead-14a8-a...,2025-12-10 10:32:31+00:00,69346587dfdffc1057144d32
76,participant_071,c,httpsfarmpcibexnetpzCPVqO/36999898-e046-f51d-2...,2025-12-15 07:38:03+00:00,693c768c138df103c91b6c22


In [13]:
# Value counts for group
participants_df['group'].value_counts(dropna=False)

group
a    25
c    24
b    23
Name: count, dtype: int64

## Events

In [14]:
# Filter for experiment/practice trials only
events_df = df[df['label'].isin(['experiment', 'practice'])].copy()

# Rename conditions: remane "exclusive" to "exhaustive", and rename "focus" to "unmodified" in events_df
events_df['condition'] = events_df['condition'].replace({
    'exclusive': 'exhaustive',
    'focus': 'unmodified',
})

# Sort for deterministic grouping
events_df = events_df.sort_values(['participant_id', 'no', 'item'])
events_df


,MD5,trial,label,PennElementType,PennElementName,Parameter,Value,participant_id,group,no,...,right,target,results_time,_prev_results_time,_gap_s,_new_session,_session_idx,participant_run_id,event_time,elapsed_ms
445,0815250ff86d7adf38545faa5f20e73f,34,experiment,EyeTracker,tracker,calibration,58,participant_001,b,2,...,1a,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:14:52.814000+00:00,4029.0
446,0815250ff86d7adf38545faa5f20e73f,34,experiment,EyeTracker,tracker,Filename,httpsfarmpcibexnetpzCPVqO/437257d2-8424-bb79-8...,participant_001,b,2,...,1a,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:11.229000+00:00,18415.0
448,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r0,PressedKey,,participant_001,b,2,...,1a,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:03.614000+00:00,-7615.0
449,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r1,PressedKey,,participant_001,b,2,...,1a,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:04.091000+00:00,477.0
450,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r2,PressedKey,,participant_001,b,2,...,1a,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:04.535000+00:00,444.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55626,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,r5,PressedKey,,participant_072,a,992,...,F,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:02.685000+00:00,826.0
55627,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,r6,PressedKey,,participant_072,a,992,...,F,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:04.445000+00:00,1760.0
55628,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,r7,PressedKey,,participant_072,a,992,...,F,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:06.256000+00:00,1811.0
55629,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,question,PressedKey,,participant_072,a,992,...,F,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:09.485000+00:00,3229.0


In [15]:
# Region reading times: r1..r7
region_names = [f"r{i}" for i in range(1, 8)]
is_region = events_df['PennElementName'].str.lower().isin(region_names) & events_df['Parameter'].str.lower().eq('pressedkey')
regions = events_df[is_region].copy()
regions['region_idx'] = regions['PennElementName'].str.extract(r'r(\d)')[0].astype(int)
regions

,MD5,trial,label,PennElementType,PennElementName,Parameter,Value,participant_id,group,no,...,target,results_time,_prev_results_time,_gap_s,_new_session,_session_idx,participant_run_id,event_time,elapsed_ms,region_idx
449,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r1,PressedKey,,participant_001,b,2,...,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:04.091000+00:00,477.0,1
450,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r2,PressedKey,,participant_001,b,2,...,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:04.535000+00:00,444.0,2
451,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r3,PressedKey,,participant_001,b,2,...,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:04.963000+00:00,428.0,3
452,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r4,PressedKey,,participant_001,b,2,...,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:05.343000+00:00,380.0,4
453,0815250ff86d7adf38545faa5f20e73f,34,experiment,Key,r5,PressedKey,,participant_001,b,2,...,,2025-12-08 13:22:10+00:00,2025-12-08 13:22:10+00:00,0.0,0,1.0,5909f8b6bc185d00011aa307,2025-12-08 13:15:05.752000+00:00,409.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55624,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,r3,PressedKey,,participant_072,a,992,...,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:00.960000+00:00,896.0,3
55625,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,r4,PressedKey,,participant_072,a,992,...,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:01.859000+00:00,899.0,4
55626,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,r5,PressedKey,,participant_072,a,992,...,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:02.685000+00:00,826.0,5
55627,7426c99a528e9ce8f1ed0b7234deca9d,8,practice,Key,r6,PressedKey,,participant_072,a,992,...,E,2025-12-13 07:55:48+00:00,2025-12-13 07:55:48+00:00,0.0,0,1.0,693d10ff23f42886b9051c5f,2025-12-13 07:28:04.445000+00:00,1760.0,6


In [16]:
# Build a full trial index to ensure all trials are present
trial_index_cols = ['participant_id', 'group', 'trial', 'label', 'no', 'item', 'exp', 'condition', 'cb', 'left', 'right', 'target']

# Ensure trial_index_cols exist in events_df before selecting (EXPECTED_FIELDS earlier guarantees presence)
trial_index = events_df.drop_duplicates(subset=[c for c in trial_index_cols if c in events_df.columns])[[c for c in trial_index_cols if c in events_df.columns]].sort_values([c for c in trial_index_cols if c in events_df.columns])
trial_index

,participant_id,group,trial,label,no,item,exp,condition,cb,left,right,target
10,participant_001,b,6,practice,990,1,,practice,n,A,B,A
26,participant_001,b,7,practice,991,2,,practice,y,D,C,C
42,participant_001,b,8,practice,992,3,,practice,n,E,F,E
59,participant_001,b,10,experiment,95,29,2,self-directed,n,földműves,munkás,munkás
75,participant_001,b,11,experiment,121,36,,attention-check,n,vacsora,reggeli,vacsora
...,...,...,...,...,...,...,...,...,...,...,...,...
56229,participant_072,a,46,experiment,61,20,2,self-directed-x,n,rendező,programozó,programozó
56249,participant_072,a,47,experiment,24,8,1,contrastive,y,8b,8a,8b
56270,participant_072,a,48,experiment,109,32,2,self-directed-x,n,politikus,újságíró,újságíró
56290,participant_072,a,49,experiment,19,7,1,exhaustive,n,7a,7b,7a


In [17]:
# Compute region RTs per trial (item) using average of elapsed_ms
region_df = regions.pivot_table(
    index=[c for c in trial_index_cols if c in regions.columns],
    columns='region_idx',
    values='elapsed_ms',
    aggfunc='mean',
    fill_value=np.nan
)

# Reindex to include any trials that had no region rows
region_df = region_df.reindex(trial_index.set_index([c for c in trial_index_cols if c in trial_index.columns]).index, fill_value=np.nan)

# Rename numeric region columns to r1..r7
region_df = region_df.rename(columns={i: f"r{int(i)}" for i in region_df.columns})
region_df

region_idx                                                                                                  r1  \
participant_id  group trial label      no  item exp condition       cb left      right      target               
participant_001 b     6     practice   990 1        practice        n  A         B          A           1660.0   
                      7     practice   991 2        practice        y  D         C          C            907.0   
                      8     practice   992 3        practice        n  E         F          E            619.0   
                      10    experiment 95  29   2   self-directed   n  földműves munkás     munkás       733.0   
                      11    experiment 121 36       attention-check n  vacsora   reggeli    vacsora     1350.0   
...                                                                                                        ...   
participant_072 a     46    experiment 61  20   2   self-directed-x n  rendező   programozó programozó   646.0   
                      47    experiment 24  8    1   contrastive     y  8b        8a         8b           478.0   
                      48    experiment 109 32   2   self-directed-x n  politikus újságíró   újságíró     627.0   
                      49    experiment 19  7    1   exhaustive      n  7a        7b         7a           600.0   
                      50    experiment 6   2    1   contrastive     y  2b        2a         2b           498.0   

region_idx                                                                                                  r2  \
participant_id  group trial label      no  item exp condition       cb left      right      target               
participant_001 b     6     practice   990 1        practice        n  A         B          A           1170.0   
                      7     practice   991 2        practice        y  D         C          C            757.0   
                      8     practice   992 3        practice        n  E         F          E            533.0   
                      10    experiment 95  29   2   self-directed   n  földműves munkás     munkás       590.0   
                      11    experiment 121 36       attention-check n  vacsora   reggeli    vacsora      900.0   
...                                                                                                        ...   
participant_072 a     46    experiment 61  20   2   self-directed-x n  rendező   programozó programozó   378.0   
                      47    experiment 24  8    1   contrastive     y  8b        8a         8b           363.0   
                      48    experiment 109 32   2   self-directed-x n  politikus újságíró   újságíró     370.0   
                      49    experiment 19  7    1   exhaustive      n  7a        7b         7a           383.0   
                      50    experiment 6   2    1   contrastive     y  2b        2a         2b           448.0   

region_idx                                                                                                 r3  \
participant_id  group trial label      no  item exp condition       cb left      right      target              
participant_001 b     6     practice   990 1        practice        n  A         B          A           983.0   
                      7     practice   991 2        practice        y  D         C          C           730.0   
                      8     practice   992 3        practice        n  E         F          E           569.0   
                      10    experiment 95  29   2   self-directed   n  földműves munkás     munkás      585.0   
                      11    experiment 121 36       attention-check n  vacsora   reggeli    vacsora     753.0   
...                                                                                                       ...   
participant_072 a     46    experiment 61  20   2   self-directed-x n  rendező   programozó programozó  347.0   
                 

In [18]:
# Question RT: average elapsed_ms for question keypress
is_question = events_df['PennElementName'].str.lower().eq('question') & events_df['Parameter'].str.lower().eq('pressedkey')
questions = events_df[is_question].copy()
questions = questions.sort_values(['participant_id', 'no', 'item'])
questions['question_rt'] = questions['elapsed_ms']
question_rt = questions.groupby([c for c in trial_index_cols if c in questions.columns])['question_rt'].mean()

# Choice RT and value: average elapsed_ms for choice selection
is_choice = events_df['PennElementType'].str.lower().eq('selector') & events_df['Parameter'].str.lower().eq('selection')
choices = events_df[is_choice].copy()
choices = choices.sort_values(['MD5', 'participant_id', 'no', 'item'])
choices['choice_rt'] = choices['elapsed_ms']
choices_value = choices.groupby([c for c in trial_index_cols if c in choices.columns])['Value'].first()
choice_rt = choices.groupby([c for c in trial_index_cols if c in choices.columns])['choice_rt'].mean()

# Assemble events_df DataFrame
events_df = region_df.copy()
events_df['question_rt'] = question_rt
events_df['choice_rt'] = choice_rt
events_df['choice'] = choices_value
events_df = events_df.reset_index()

# Rename choice values to 'left'/'right' only
events_df['choice'] = (
    events_df['choice']
    .astype(str)
    .str.lower()
    .replace({'left_canvas': 'left', 'right_canvas': 'right'})
)

# Determine chosen_type based on choice and left/right stimuli
def _suffix_type(val):
    """Return 'A' or 'B' if val ends with 'a' or 'b' (in any case), else None."""
    if not isinstance(val, str):
        return None
    val = val.strip()
    if val.lower().endswith('a'):
        return 'A'
    if val.lower().endswith('b'):
        return 'B'
    return None

def _chosen_type(row):
    choice = str(row.get('choice')).strip().lower() if pd.notna(row.get('choice')) else None
    if choice == 'left':
        return _suffix_type(row.get('left'))
    if choice == 'right':
        return _suffix_type(row.get('right'))
    return None

mask_exp1 = events_df['exp'].astype(str).eq('1')
events_df['chosen_type'] = pd.Series(pd.NA, index=events_df.index, dtype='string')
events_df.loc[mask_exp1, 'chosen_type'] = (
    events_df.loc[mask_exp1].apply(_chosen_type, axis=1).astype('string')
)
events_df

region_idx,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,r2,r3,r4,r5,r6,r7,question_rt,choice_rt,choice,chosen_type
0,participant_001,b,6,practice,990,1,,practice,n,A,...,1170.0,983.0,980.0,867.0,904.0,1096.0,NaN,5473.0,left,<NA>
1,participant_001,b,7,practice,991,2,,practice,y,D,...,757.0,730.0,685.0,696.0,958.0,681.0,NaN,5679.0,right,<NA>
2,participant_001,b,8,practice,992,3,,practice,n,E,...,533.0,569.0,631.0,566.0,695.0,673.0,3719.0,4664.0,left,<NA>
3,participant_001,b,10,experiment,95,29,2,self-directed,n,földműves,...,590.0,585.0,662.0,715.0,NaN,888.0,5412.0,4220.0,right,<NA>
4,participant_001,b,11,experiment,121,36,,attention-check,n,vacsora,...,900.0,753.0,630.0,793.0,689.0,924.0,2360.0,3787.0,left,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,participant_072,a,46,experiment,61,20,2,self-directed-x,n,rendező,...,378.0,347.0,470.0,380.0,NaN,433.0,5391.0,3020.0,left,<NA>
3140,participant_072,a,47,experiment,24,8,1,contrastive,y,8b,...,363.0,394.0,368.0,407.0,411.0,429.0,NaN,4005.0,right,A
3141,participant_072,a,48,experiment,109,32,2,self-directed-x,n,politikus,...,370.0,418.0,430.0,399.0,NaN,475.0,1864.0,2609.0,right,<NA>
3142,participant_072,a,49,experiment,19,7,1,exhaustive,n,7a,...,383.0,321.0,305.0,294.0,275.0,315.0,NaN,3990.0,left,A


In [19]:
################################################################
# Subtract fixed offsets (keep units consistent with elapsed_ms)
events_df['question_rt'] = events_df['question_rt'] - 1000
events_df['choice_rt'] = events_df['choice_rt'] - 3000

# Sort by participant and trial for deterministic ordering
sort_cols = ['participant_id', 'trial']
events_df = events_df.sort_values(by=sort_cols, ascending=[True] * len(sort_cols))

# events_df = events_df.reset_index()
events_df

region_idx,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,r2,r3,r4,r5,r6,r7,question_rt,choice_rt,choice,chosen_type
0,participant_001,b,6,practice,990,1,,practice,n,A,...,1170.0,983.0,980.0,867.0,904.0,1096.0,NaN,2473.0,left,<NA>
1,participant_001,b,7,practice,991,2,,practice,y,D,...,757.0,730.0,685.0,696.0,958.0,681.0,NaN,2679.0,right,<NA>
2,participant_001,b,8,practice,992,3,,practice,n,E,...,533.0,569.0,631.0,566.0,695.0,673.0,2719.0,1664.0,left,<NA>
3,participant_001,b,10,experiment,95,29,2,self-directed,n,földműves,...,590.0,585.0,662.0,715.0,NaN,888.0,4412.0,1220.0,right,<NA>
4,participant_001,b,11,experiment,121,36,,attention-check,n,vacsora,...,900.0,753.0,630.0,793.0,689.0,924.0,1360.0,787.0,left,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,participant_072,a,46,experiment,61,20,2,self-directed-x,n,rendező,...,378.0,347.0,470.0,380.0,NaN,433.0,4391.0,20.0,left,<NA>
3140,participant_072,a,47,experiment,24,8,1,contrastive,y,8b,...,363.0,394.0,368.0,407.0,411.0,429.0,NaN,1005.0,right,A
3141,participant_072,a,48,experiment,109,32,2,self-directed-x,n,politikus,...,370.0,418.0,430.0,399.0,NaN,475.0,864.0,-391.0,right,<NA>
3142,participant_072,a,49,experiment,19,7,1,exhaustive,n,7a,...,383.0,321.0,305.0,294.0,275.0,315.0,NaN,990.0,left,A


### Attention checks

In [20]:
# Filter events_df for attention-check condition
attention_check_rows = events_df[events_df['condition'] == 'attention-check']
# Show rows where choice does not match target
failed_attention_checks = attention_check_rows[attention_check_rows['choice'] != attention_check_rows['target']]
failed_attention_checks

# Refined attention check: match choice to the column (left or right) that matches target
def check_attention(row):
    if row['choice'] == 'left':
        return row['target'] == row['left']
    elif row['choice'] == 'right':
        return row['target'] == row['right']
    return False

attention_check_rows = events_df[events_df['condition'] == 'attention-check'].copy()
attention_check_rows['attention_pass'] = attention_check_rows.apply(check_attention, axis=1)

# Show rows where attention check failed
failed_attention_checks = attention_check_rows[~attention_check_rows['attention_pass']]
failed_attention_checks

region_idx,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,r3,r4,r5,r6,r7,question_rt,choice_rt,choice,chosen_type,attention_pass
1215,participant_028,a,44,experiment,111,33,,attention-check,n,burgonya,...,206.0,201.0,346.0,NaN,371.0,NaN,1958.0,right,<NA>,False
1953,participant_045,a,41,experiment,120,36,,attention-check,y,reggeli,...,441.0,496.0,754.0,811.0,1227.0,7190.0,13764.0,left,<NA>,False
3137,participant_072,a,44,experiment,111,33,,attention-check,n,burgonya,...,420.0,368.0,413.0,NaN,494.0,NaN,13469.0,right,<NA>,False


### Mismatch

In [21]:
# Find rows where target is available and does not match the chosen side's value
def choice_matches_target(row):
    if pd.isna(row['choice']) or pd.isna(row['target']):
        return True  # skip if missing
    if row['choice'] == 'left':
        return row['target'] == row['left']
    elif row['choice'] == 'right':
        return row['target'] == row['right']
    return False

mismatch_rows = events_df[
    events_df['target'].notna() &
    events_df['choice'].notna() &
    (~events_df.apply(choice_matches_target, axis=1))
].copy()

# Filter mismatch_rows to exclude any rows where 'target' is missing or empty string
mismatch_rows = mismatch_rows[mismatch_rows['target'].notna() & (mismatch_rows['target'] != '')].copy()
mismatch_rows

# Add a new column 'mismatch' to the events_df to indicate these items where the choice does not match the target
events_df['mismatch'] = False
events_df.loc[mismatch_rows.index, 'mismatch'] = True

# Print how many items I have for each condition
print(events_df['mismatch'].value_counts())
events_df

mismatch
False    2716
True      428
Name: count, dtype: int64


region_idx,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,r3,r4,r5,r6,r7,question_rt,choice_rt,choice,chosen_type,mismatch
0,participant_001,b,6,practice,990,1,,practice,n,A,...,983.0,980.0,867.0,904.0,1096.0,NaN,2473.0,left,<NA>,False
1,participant_001,b,7,practice,991,2,,practice,y,D,...,730.0,685.0,696.0,958.0,681.0,NaN,2679.0,right,<NA>,False
2,participant_001,b,8,practice,992,3,,practice,n,E,...,569.0,631.0,566.0,695.0,673.0,2719.0,1664.0,left,<NA>,False
3,participant_001,b,10,experiment,95,29,2,self-directed,n,földműves,...,585.0,662.0,715.0,NaN,888.0,4412.0,1220.0,right,<NA>,False
4,participant_001,b,11,experiment,121,36,,attention-check,n,vacsora,...,753.0,630.0,793.0,689.0,924.0,1360.0,787.0,left,<NA>,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,participant_072,a,46,experiment,61,20,2,self-directed-x,n,rendező,...,347.0,470.0,380.0,NaN,433.0,4391.0,20.0,left,<NA>,True
3140,participant_072,a,47,experiment,24,8,1,contrastive,y,8b,...,394.0,368.0,407.0,411.0,429.0,NaN,1005.0,right,A,True
3141,participant_072,a,48,experiment,109,32,2,self-directed-x,n,politikus,...,418.0,430.0,399.0,NaN,475.0,864.0,-391.0,right,<NA>,False
3142,participant_072,a,49,experiment,19,7,1,exhaustive,n,7a,...,321.0,305.0,294.0,275.0,315.0,NaN,990.0,left,A,False


## Choosing experiment

In [22]:
# Remove all rows where exp is 2 from trial_index
events_df = events_df[events_df['exp'] != '2'].copy()
events_df

# Reorder by custom condition order: practice, attention-check, exclusive, focus, contrastive
custom_order = ['practice', 'attention-check', 'exhaustive', 'unmodified', 'contrastive']
existing = events_df['condition'].dropna().astype(str).unique().tolist()
remaining = [c for c in existing if c not in custom_order]
full_order = custom_order + remaining

# Make ordered categorical and then sort by it (plus any secondary keys).
events_df['condition'] = pd.Categorical(events_df['condition'].astype(str), categories=full_order, ordered=True)
events_df = events_df.sort_values(by=['participant_id', 'no'], na_position='last').reset_index(drop=True)
events_df

region_idx,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,r3,r4,r5,r6,r7,question_rt,choice_rt,choice,chosen_type,mismatch
0,participant_001,b,34,experiment,2,1,1,unmodified,y,1b,...,428.0,380.0,409.0,412.0,469.0,NaN,1595.0,right,A,False
1,participant_001,b,26,experiment,4,2,1,exhaustive,y,2b,...,525.0,502.0,513.0,549.0,865.0,NaN,1288.0,right,A,False
2,participant_001,b,49,experiment,9,3,1,contrastive,n,3a,...,466.0,404.0,424.0,557.0,701.0,NaN,3059.0,left,A,True
3,participant_001,b,25,experiment,11,4,1,unmodified,n,4a,...,557.0,609.0,522.0,761.0,780.0,NaN,1406.0,left,A,False
4,participant_001,b,13,experiment,13,5,1,exhaustive,n,5a,...,558.0,476.0,450.0,584.0,731.0,NaN,2774.0,left,A,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1795,participant_072,a,20,experiment,117,35,,attention-check,n,pelikán,...,576.0,538.0,553.0,437.0,679.0,2759.0,-165.0,left,<NA>,False
1796,participant_072,a,41,experiment,120,36,,attention-check,y,reggeli,...,392.0,426.0,429.0,717.0,459.0,992.0,-337.0,right,<NA>,False
1797,participant_072,a,6,practice,990,1,,practice,n,A,...,973.0,829.0,891.0,842.0,920.0,NaN,2041.0,left,<NA>,False
1798,participant_072,a,7,practice,991,2,,practice,y,D,...,753.0,737.0,769.0,755.0,771.0,NaN,1720.0,right,<NA>,False


## Expectations

In [23]:
import plotly.express as px

# Drop 'attention_check' and 'practice' conditions for this plot
events_df = events_df[~events_df['condition'].isin(['attention-check', 'practice'])].copy()

# Calculate mismatch ratio per condition
mismatch_counts = events_df.groupby('condition', observed=True)['mismatch'].agg(['sum', 'count'])
mismatch_counts['mismatch_ratio'] = mismatch_counts['sum'] / mismatch_counts['count']

# Prepare data for donut chart
donut_data = []
for cond, row in mismatch_counts.iterrows():
    donut_data.append({'condition': cond, 'type': 'Mismatch', 'count': row['sum']})
    donut_data.append({'condition': cond, 'type': 'Match', 'count': row['count'] - row['sum']})

donut_df = pd.DataFrame(donut_data)

fig = px.pie(
    donut_df,
    names='type',
    values='count',
    color='type',
    facet_col='condition',
    hole=0.5,
    title='Expectation Mismatch Ratio per Condition',
    color_discrete_map={'Mismatch': '#bf616a', 'Match': '#4c566a'},
)

fig.for_each_annotation(
    lambda a: a.update(
        text=a.text.split("=")[-1],
        y=-0,          # move under donuts
        yanchor="top",
    )
)

# Set template (or set pio.templates.default earlier)
fig.update_layout(**base_layout, legend=dict(orientation='h', y=-0.1, x=0.5, xanchor='center'))
fig.update_xaxes(title_standoff=0, automargin=False)
fig.update_yaxes(title_standoff=0, automargin=False)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

# Save as png and html
fig.write_image(plots / "mismatch_per_condition.png", scale=3)
fig.write_html(plots / "mismatch_per_condition.html", include_plotlyjs='cdn')

## Reading Times

In [24]:
# In the events_df dataframe, create a new column between r7 and question_rt 'sentence_rt' which is the sum of all the region rts (r1 to r7)
region_cols = [f"r{i}" for i in range(1, 8)]
events_df['sentence_rt'] = events_df[region_cols].sum(axis=1)  

# Move this new sentence_rt column to be between r7 and question_rt
cols = events_df.columns.tolist()
cols.insert(cols.index('question_rt'), cols.pop(cols.index('sentence_rt')))
events_df = events_df[cols]

events_df

region_idx,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,r4,r5,r6,r7,sentence_rt,question_rt,choice_rt,choice,chosen_type,mismatch
0,participant_001,b,34,experiment,2,1,1,unmodified,y,1b,...,380.0,409.0,412.0,469.0,3019.0,NaN,1595.0,right,A,False
1,participant_001,b,26,experiment,4,2,1,exhaustive,y,2b,...,502.0,513.0,549.0,865.0,4152.0,NaN,1288.0,right,A,False
2,participant_001,b,49,experiment,9,3,1,contrastive,n,3a,...,404.0,424.0,557.0,701.0,3485.0,NaN,3059.0,left,A,True
3,participant_001,b,25,experiment,11,4,1,unmodified,n,4a,...,609.0,522.0,761.0,780.0,4396.0,NaN,1406.0,left,A,False
4,participant_001,b,13,experiment,13,5,1,exhaustive,n,5a,...,476.0,450.0,584.0,731.0,4098.0,NaN,2774.0,left,A,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1788,participant_072,a,31,experiment,42,14,1,contrastive,y,14b,...,446.0,368.0,568.0,2080.0,5883.0,NaN,2088.0,left,B,False
1789,participant_072,a,29,experiment,44,15,1,unmodified,y,15b,...,941.0,416.0,416.0,470.0,3709.0,NaN,603.0,right,A,False
1790,participant_072,a,40,experiment,46,16,1,exhaustive,y,16b,...,403.0,387.0,361.0,394.0,2767.0,NaN,465.0,right,A,False
1791,participant_072,a,34,experiment,51,17,1,contrastive,n,17a,...,536.0,371.0,402.0,425.0,3117.0,NaN,2626.0,right,B,False


In [25]:
# # Replot: every item as an individual line, faceted by condition, color by participant
# import plotly.express as px

# # Rebuild melted if needed (one row per participant × item × region)
# region_cols = [c for c in events_df.columns if re.match(r"r\d+$", c)]
# plot_df = events_df.melt(
#     id_vars=['participant_id', 'no', 'condition'],
#     value_vars=region_cols,
#     var_name='region',
#     value_name='reading_time'
# ).dropna(subset=['reading_time'])

# # Ensure region order
# full_region_order = [f"r{i}" for i in range(1, 8)]
# plot_df['region'] = pd.Categorical(plot_df['region'], categories=full_region_order, ordered=True)

# # Create a unique line id per item (so each item is its own trace) and keep color = participant
# plot_df['pid_item'] = plot_df['participant_id'].astype(str) + ' | no ' + plot_df['no'].astype(str)

# fig = px.line(
#     plot_df,
#     x='region',
#     y='reading_time',
#     color='participant_id',        # color by participant
#     line_group='pid_item',         # each item -> separate connected line
#     facet_col='condition',
#     # facet_col_wrap=2,              # <- two columns of facets
#     category_orders={'region': full_region_order},
#     markers=True,
#     template='nord_light_paper',       
#     title='Reading times per item (each item = one line), faceted by condition, colored by participant',
#     labels={'reading_time': 'Reading time (ms)', 'region': 'Region'}
# )

# fig.update_traces(mode='lines+markers', marker={'size':5}, opacity=0.75, hovertemplate=None)
# fig.update_layout(base_layout, legend_title_text='Participant')
# fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # simplify facet labels
# fig.show()

# # Save image as png and html
# fig.write_image(plots / "reading_times_raw.png", scale=3)
# fig.write_html(plots / "reading_times_raw.html", include_plotlyjs='cdn')

In [26]:
# Replot: every item as an individual line, faceted by condition, color by participant
import plotly.express as px
import numpy as np

# Rebuild melted if needed (one row per participant × item × region)
region_cols = [c for c in events_df.columns if re.match(r"r\d+$", c)]
plot_df = events_df.melt(
    id_vars=['participant_id', 'no', 'condition'],
    value_vars=region_cols,
    var_name='region',
    value_name='reading_time'
).dropna(subset=['reading_time'])

# Ensure region order
full_region_order = [f"r{i}" for i in range(1, 8)]
plot_df['region'] = pd.Categorical(plot_df['region'], categories=full_region_order, ordered=True)

# Create a unique line id per item (so each item is its own trace) and keep color = participant
plot_df['pid_item'] = plot_df['participant_id'].astype(str) + ' | no ' + plot_df['no'].astype(str)

# Winsorize reading_time at mean + 2*SD within each condition × region
wins_stats = (
    plot_df
    .groupby(['condition', 'region'], observed=True)['reading_time']
    .agg(mu='mean', sd='std')
    .reset_index()
)

plot_df = plot_df.merge(wins_stats, on=['condition', 'region'], how='left')
plot_df['upper_cap'] = plot_df['mu'] + 2 * plot_df['sd']
plot_df['reading_time_wins'] = np.where(
    plot_df['upper_cap'].notna(),
    np.minimum(plot_df['reading_time'], plot_df['upper_cap']),
    plot_df['reading_time']
)

# Optional check
n_capped = int((plot_df['reading_time_wins'] < plot_df['reading_time']).sum())
print(f"Capped points: {n_capped}")

fig = px.line(
    plot_df,
    x='region',
    y='reading_time_wins',
    color='participant_id',        # color by participant
    line_group='pid_item',         # each item -> separate connected line
    facet_col='condition',
    # facet_col_wrap=2,              # <- two columns of facets
    category_orders={'region': full_region_order},
    markers=True,
    template='nord_light_paper',
    title='Reading times per item (winsorized at mean + 2SD), faceted by condition, colored by participant',
    labels={'reading_time_wins': 'Reading time (ms)', 'region': 'Region'}
)

fig.update_traces(mode='lines+markers', marker={'size':5}, opacity=0.75, hovertemplate=None)
fig.update_layout(base_layout, legend_title_text='Participant')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # simplify facet labels
fig.show()

# Save image as png and html
fig.write_image(plots / "reading_times_winsorized.png", scale=3)
fig.write_html(plots / "reading_times_winsorized.html", include_plotlyjs='cdn')

Capped points: 391


In [27]:
# Remove practice and attention checks
events_df = events_df[events_df['exp'].astype(str).eq('1')].copy()
events_df

region_idx,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,r4,r5,r6,r7,sentence_rt,question_rt,choice_rt,choice,chosen_type,mismatch
0,participant_001,b,34,experiment,2,1,1,unmodified,y,1b,...,380.0,409.0,412.0,469.0,3019.0,NaN,1595.0,right,A,False
1,participant_001,b,26,experiment,4,2,1,exhaustive,y,2b,...,502.0,513.0,549.0,865.0,4152.0,NaN,1288.0,right,A,False
2,participant_001,b,49,experiment,9,3,1,contrastive,n,3a,...,404.0,424.0,557.0,701.0,3485.0,NaN,3059.0,left,A,True
3,participant_001,b,25,experiment,11,4,1,unmodified,n,4a,...,609.0,522.0,761.0,780.0,4396.0,NaN,1406.0,left,A,False
4,participant_001,b,13,experiment,13,5,1,exhaustive,n,5a,...,476.0,450.0,584.0,731.0,4098.0,NaN,2774.0,left,A,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1788,participant_072,a,31,experiment,42,14,1,contrastive,y,14b,...,446.0,368.0,568.0,2080.0,5883.0,NaN,2088.0,left,B,False
1789,participant_072,a,29,experiment,44,15,1,unmodified,y,15b,...,941.0,416.0,416.0,470.0,3709.0,NaN,603.0,right,A,False
1790,participant_072,a,40,experiment,46,16,1,exhaustive,y,16b,...,403.0,387.0,361.0,394.0,2767.0,NaN,465.0,right,A,False
1791,participant_072,a,34,experiment,51,17,1,contrastive,n,17a,...,536.0,371.0,402.0,425.0,3117.0,NaN,2626.0,right,B,False


In [28]:
import plotly.express as px
import numpy as np

# Melt events_df to one row per participant × item × region
region_cols = [c for c in events_df.columns if re.match(r"r\d+$", c)]
melted = (
    events_df
    .melt(
        id_vars=['participant_id', 'item', 'condition'],
        value_vars=region_cols,
        var_name='region',
        value_name='reading_time'
    )
    .dropna(subset=['reading_time'])
)

# -----------------------------
# SAME 2SD winsorization (upper cap): mean + 2*SD
# -----------------------------
# Region RTs: cap within condition × region
reg_stats = (
    melted
    .groupby(['condition', 'region'], observed=True)['reading_time']
    .agg(mu='mean', sd='std')
    .reset_index()
)
melted = melted.merge(reg_stats, on=['condition', 'region'], how='left')
melted['upper_cap'] = melted['mu'] + 2 * melted['sd']
melted['reading_time_wins'] = np.where(
    melted['upper_cap'].notna(),
    np.minimum(melted['reading_time'], melted['upper_cap']),
    melted['reading_time']
)

# Question/choice RTs: cap within condition
events_w = events_df.copy()

if 'question_rt' in events_w.columns:
    q_stats = (
        events_w.dropna(subset=['question_rt'])
        .groupby('condition', observed=True)['question_rt']
        .agg(mu='mean', sd='std')
        .reset_index()
    )
    q_stats['upper_cap_q'] = q_stats['mu'] + 2 * q_stats['sd']
    events_w = events_w.merge(q_stats[['condition', 'upper_cap_q']], on='condition', how='left')
    events_w['question_rt_wins'] = np.where(
        events_w['upper_cap_q'].notna(),
        np.minimum(events_w['question_rt'], events_w['upper_cap_q']),
        events_w['question_rt']
    )
else:
    events_w['question_rt_wins'] = np.nan

if 'choice_rt' in events_w.columns:
    c_stats = (
        events_w.dropna(subset=['choice_rt'])
        .groupby('condition', observed=True)['choice_rt']
        .agg(mu='mean', sd='std')
        .reset_index()
    )
    c_stats['upper_cap_c'] = c_stats['mu'] + 2 * c_stats['sd']
    events_w = events_w.merge(c_stats[['condition', 'upper_cap_c']], on='condition', how='left')
    events_w['choice_rt_wins'] = np.where(
        events_w['upper_cap_c'].notna(),
        np.minimum(events_w['choice_rt'], events_w['upper_cap_c']),
        events_w['choice_rt']
    )
else:
    events_w['choice_rt_wins'] = np.nan

# Optional check
n_capped_regions = int((melted['reading_time_wins'] < melted['reading_time']).sum())
n_capped_q = int(((events_w['question_rt_wins'] < events_w['question_rt']) & events_w['question_rt'].notna()).sum()) if 'question_rt' in events_w.columns else 0
n_capped_c = int(((events_w['choice_rt_wins'] < events_w['choice_rt']) & events_w['choice_rt'].notna()).sum()) if 'choice_rt' in events_w.columns else 0
print(f"Capped points -> regions: {n_capped_regions}, question: {n_capped_q}, choice: {n_capped_c}")

# Per-participant (across items) mean and std for each region × condition
pp_region_stats = (
    melted
    .groupby(['participant_id', 'condition', 'region'], as_index=False, observed=True)
    .reading_time_wins.agg(reading_time_mean='mean', reading_time_std='std')
)

# Per-participant question/choice mean+std across items (winsorized, if present)
extras_pp = []
if 'question_rt_wins' in events_w.columns:
    q_pp = (
        events_w
        .dropna(subset=['question_rt_wins'])
        .groupby(['participant_id', 'condition'], as_index=False, observed=True)
        .question_rt_wins.agg(reading_time_mean='mean', reading_time_std='std')
    )
    q_pp['region'] = 'question'
    extras_pp.append(q_pp[['participant_id', 'condition', 'region', 'reading_time_mean', 'reading_time_std']])

if 'choice_rt_wins' in events_w.columns:
    c_pp = (
        events_w
        .dropna(subset=['choice_rt_wins'])
        .groupby(['participant_id', 'condition'], as_index=False, observed=True)
        .choice_rt_wins.agg(reading_time_mean='mean', reading_time_std='std')
    )
    c_pp['region'] = 'choice'
    extras_pp.append(c_pp[['participant_id', 'condition', 'region', 'reading_time_mean', 'reading_time_std']])

if extras_pp:
    pp_region_stats = pd.concat([pp_region_stats] + extras_pp, ignore_index=True, sort=False)

# Only show present conditions
present_conditions = pp_region_stats['condition'].dropna().unique().tolist()

# Aggregate across participants
present_regions = pp_region_stats['region'].unique().tolist()
full_region_order = [f"r{i}" for i in range(1, 8)]
present_regions = [r for r in full_region_order if r in present_regions] + [r for r in ['question', 'choice'] if r in pp_region_stats['region'].unique()]

agg_plot = (
    pp_region_stats
    .groupby(['region', 'condition'], as_index=False, observed=True)
    .reading_time_mean.agg(mean_reading_time='mean', std_reading_time='std')
)

# Fallback std (also from winsorized values)
std_items = (
    melted
    .groupby(['condition', 'region'], observed=True)['reading_time_wins']
    .std()
    .reset_index()
    .rename(columns={'reading_time_wins': 'std_items'})
)

if 'question_rt_wins' in events_w.columns:
    std_q = (
        events_w
        .dropna(subset=['question_rt_wins'])
        .groupby('condition', observed=True)['question_rt_wins']
        .std()
        .reset_index()
        .rename(columns={'question_rt_wins': 'std_items'})
    )
    std_q['region'] = 'question'
    std_items = pd.concat([std_items, std_q[['condition', 'region', 'std_items']]], ignore_index=True)

if 'choice_rt_wins' in events_w.columns:
    std_c = (
        events_w
        .dropna(subset=['choice_rt_wins'])
        .groupby('condition', observed=True)['choice_rt_wins']
        .std()
        .reset_index()
        .rename(columns={'choice_rt_wins': 'std_items'})
    )
    std_c['region'] = 'choice'
    std_items = pd.concat([std_items, std_c[['condition', 'region', 'std_items']]], ignore_index=True)

agg_plot = agg_plot.merge(std_items, on=['condition', 'region'], how='left')
agg_plot['std_reading_time'] = agg_plot['std_reading_time'].fillna(agg_plot['std_items'])
agg_plot.drop(columns=['std_items'], inplace=True)

# Keep only present regions and set categorical ordering
agg_plot = agg_plot[agg_plot['region'].isin(present_regions) & agg_plot['condition'].isin(present_conditions)].copy()
agg_plot['region'] = pd.Categorical(agg_plot['region'], categories=present_regions, ordered=True)
agg_plot['condition'] = pd.Categorical(agg_plot['condition'], categories=present_conditions, ordered=True)

# Plot 1: Regions
region_only = agg_plot[agg_plot['region'].isin(full_region_order)]
fig1 = px.line(
    region_only,
    x='region',
    y='mean_reading_time',
    error_y='std_reading_time',
    color='condition',
    facet_col='condition',
    category_orders={'region': full_region_order, 'condition': present_conditions},
    title='Mean Reading Times per Region (winsorized at mean + 2SD)<br>(Error Bars = Across-Participant Std Dev, Faceted by Condition)',
    labels={'mean_reading_time': 'Mean Reading Time (ms)', 'region': 'Region'}
)
fig1.update_traces(connectgaps=True, mode='lines+markers', marker={'size': 6}, line={'width': 2}, opacity=0.85)
fig1.update_yaxes(title_text='Mean reading time (ms)')
fig1.update_xaxes(title_text='Region')
fig1.update_layout(legend_title_text='Condition')
fig1.show()

# Plot 3: Decision RTs
choice_only = agg_plot[agg_plot['region'] == 'choice']
fig3 = px.bar(
    choice_only,
    x='condition',
    y='mean_reading_time',
    error_y='std_reading_time',
    color='condition',
    category_orders={'condition': present_conditions},
    title='Mean Decision RT by Condition (winsorized at mean + 2SD; Error Bars = Std Dev)',
    labels={'mean_reading_time': 'Mean Decision RT (ms)', 'condition': 'Condition'}
)
fig3.update_layout(legend_title_text='Condition')
fig3.show()

Capped points -> regions: 391, question: 0, choice: 32


In [29]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# Use winsorized aggregate if you kept a separate variable; otherwise use agg_plot
plot_source = agg_plot_wins if 'agg_plot_wins' in globals() else agg_plot.copy()

# Determine three_conditions (prefer canonical)
preferred = ['exhaustive', 'unmodified', 'contrastive']
present_conditions = plot_source['condition'].dropna().astype(str).unique().tolist()
three_conditions = [c for c in preferred if c in present_conditions]
if len(three_conditions) < 3:
    three_conditions = present_conditions[:3]

# Regions (exclude 'choice' from region facets)
full_region_order = [f"r{i}" for i in range(1, 8)]
region_order = full_region_order[:]

# Data frames
region_df = plot_source[
    plot_source['region'].isin(region_order) &
    plot_source['condition'].isin(three_conditions)
].copy()
region_df['region'] = pd.Categorical(region_df['region'], categories=region_order, ordered=True)

choice_df = plot_source[plot_source['region'].astype(str) == 'choice'].copy()
choice_df = choice_df[choice_df['condition'].isin(three_conditions)]

# Build color map
palette = nord
color_map = {cond: palette[i % len(palette)] for i, cond in enumerate(three_conditions)}

# Global y-range with error bars
all_y = pd.concat([
    region_df['mean_reading_time'] + region_df['std_reading_time'].fillna(0),
    choice_df['mean_reading_time'] + choice_df['std_reading_time'].fillna(0)
]).dropna()

ymin = float(pd.concat([
    region_df['mean_reading_time'] - region_df['std_reading_time'].fillna(0),
    choice_df['mean_reading_time'] - choice_df['std_reading_time'].fillna(0)
]).min()) if not all_y.isnull().all() else 0.0

ymax = float(all_y.max()) if not all_y.isnull().all() else 1.0
ypad = (ymax - ymin) * 0.15 if (ymax - ymin) > 0 else 10.0
ymin = max(0.0, ymin - ypad)
ymax = ymax + ypad

# Build subplots
n_cols = len(three_conditions) + 1
subplot_titles = [f"{c}" for c in three_conditions] + ['choice']
fig = make_subplots(rows=1, cols=n_cols, shared_yaxes=True, subplot_titles=subplot_titles)

# Region lines
for ci, cond in enumerate(three_conditions, start=1):
    d = region_df[region_df['condition'].astype(str) == cond].sort_values('region')
    if d.empty:
        continue

    cond_color = color_map.get(cond)
    fig.add_trace(
        go.Scatter(
            x=d['region'].astype(str),
            y=d['mean_reading_time'],
            mode='lines+markers',
            name=cond,
            error_y=dict(type='data', array=d['std_reading_time'].fillna(0).to_numpy()),
            marker=dict(size=6, color=cond_color),
            line=dict(width=2, color=cond_color),
        ),
        row=1, col=ci
    )
    fig.update_xaxes(categoryorder='array', categoryarray=region_order, row=1, col=ci)
    fig.update_yaxes(title_text='Mean reading time (ms)', row=1, col=ci, range=[ymin, ymax])

# Choice bars
choice_col = n_cols
if not choice_df.empty:
    choice_df['condition'] = pd.Categorical(choice_df['condition'].astype(str), categories=three_conditions, ordered=True)

    for cond in three_conditions:
        d = choice_df[choice_df['condition'].astype(str) == cond]
        if d.empty:
            continue

        cond_color = color_map.get(cond)
        fig.add_trace(
            go.Bar(
                x=[cond],
                y=[d['mean_reading_time'].iloc[0]],
                name=cond,
                marker_color=cond_color,
                error_y=dict(type='data', array=[float(d['std_reading_time'].fillna(0).iloc[0])], color=snow[2]),
            ),
            row=1, col=choice_col
        )

    fig.update_xaxes(title_text='Condition', row=1, col=choice_col)
    fig.update_yaxes(title_text='Mean reading time (ms)', row=1, col=choice_col, range=[ymin, ymax])

# Layout
fig.update_layout(
    base_layout,
    template='nord_light_paper',
    showlegend=False,
    # title='Mean Reading Times (RTs): Sentence Regions and Choices (winsorized at mean + 2SD)',
)

# fig.update_yaxes(automargin=True, title_standoff=10)

fig.show()

# Save
fig.write_image(plots / "reading_times.png", scale=3)
fig.write_html(plots / "reading_times.html", include_plotlyjs='cdn')

In [30]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd

# Use winsorized aggregate if you kept a separate variable; otherwise use agg_plot
plot_source = agg_plot_wins if 'agg_plot_wins' in globals() else agg_plot.copy()

# Determine three_conditions (prefer canonical)
preferred = ['exhaustive', 'unmodified', 'contrastive']
present_conditions = plot_source['condition'].dropna().astype(str).unique().tolist()
three_conditions = [c for c in preferred if c in present_conditions]
if len(three_conditions) < 3:
    three_conditions = present_conditions[:3]

# Regions (exclude 'choice' from region facets)
full_region_order = [f"r{i}" for i in range(1, 8)]
region_order = full_region_order[:]

# Data frames
region_df = plot_source[
    plot_source['region'].isin(region_order) &
    plot_source['condition'].isin(three_conditions)
].copy()
region_df['region'] = pd.Categorical(region_df['region'], categories=region_order, ordered=True)

choice_df = plot_source[plot_source['region'].astype(str) == 'choice'].copy()
choice_df = choice_df[choice_df['condition'].isin(three_conditions)].copy()

# Build color map
palette = nord
color_map = {cond: palette[i % len(palette)] for i, cond in enumerate(three_conditions)}

# Global y-range with error bars
all_y = pd.concat([
    region_df['mean_reading_time'] + region_df['std_reading_time'].fillna(0),
    choice_df['mean_reading_time'] + choice_df['std_reading_time'].fillna(0)
]).dropna()

ymin = float(pd.concat([
    region_df['mean_reading_time'] - region_df['std_reading_time'].fillna(0),
    choice_df['mean_reading_time'] - choice_df['std_reading_time'].fillna(0)
]).min()) if not all_y.isnull().all() else 0.0

ymax = float(all_y.max()) if not all_y.isnull().all() else 1.0
ypad = (ymax - ymin) * 0.15 if (ymax - ymin) > 0 else 10.0
ymin = max(0.0, ymin - ypad)
ymax = ymax + ypad

# --- NEW: 2-panel layout ---
# Panel 1: Regions r1-r7, all three conditions on the same axes (color distinguishes conditions)
# Panel 2: Choice RTs as bars (still by condition)
fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=["Regions (r1–r7)", "choice"],
)

# Region lines (all conditions in the same subplot)
for cond in three_conditions:
    d = region_df[region_df['condition'].astype(str) == cond].sort_values('region')
    if d.empty:
        continue

    cond_color = color_map.get(cond)
    fig.add_trace(
        go.Scatter(
            x=d['region'].astype(str),
            y=d['mean_reading_time'],
            mode='lines+markers',
            name=cond,
            error_y=dict(type='data', array=d['std_reading_time'].fillna(0).to_numpy()),
            marker=dict(size=7, color=cond_color),
            line=dict(width=2, color=cond_color),
        ),
        row=1,
        col=1,
    )

fig.update_xaxes(categoryorder='array', categoryarray=region_order, row=1, col=1)
fig.update_yaxes(title_text='Mean reading time (ms)', row=1, col=1, range=[ymin, ymax])

# Choice bars
if not choice_df.empty:
    choice_df['condition'] = pd.Categorical(choice_df['condition'].astype(str), categories=three_conditions, ordered=True)
    choice_df = choice_df.sort_values('condition')

    y = []
    err = []
    x = []
    colors = []
    for cond in three_conditions:
        d = choice_df[choice_df['condition'].astype(str) == cond]
        if d.empty:
            continue
        x.append(cond)
        y.append(float(d['mean_reading_time'].iloc[0]))
        err.append(float(d['std_reading_time'].fillna(0).iloc[0]))
        colors.append(color_map.get(cond))

    fig.add_trace(
        go.Bar(
            x=x,
            y=y,
            marker_color=colors,
            error_y=dict(type='data', array=err, color=snow[2]),
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.update_xaxes(title_text='Condition', row=1, col=2)
fig.update_yaxes(title_text='Mean reading time (ms)', row=1, col=2, range=[ymin, ymax])

# Layout
fig.update_layout(
    base_layout,
    template='nord_light_paper',
    showlegend=True,
)

fig.show()

# Save
fig.write_image(plots / "reading_times_regions_combined.png", scale=3)
fig.write_html(plots / "reading_times_regions_combined.html", include_plotlyjs='cdn')


In [31]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

# Conditions to compare
conds = ['exhaustive', 'unmodified', 'contrastive']

# Prefer winsorized choice RT if available
rt_col = 'choice_rt_wins' if 'choice_rt_wins' in events_df.columns else 'choice_rt'

# Keep only needed rows
d = events_df.copy()
d = d[d['condition'].astype(str).isin(conds)]
d = d[d[rt_col].notna()].copy()

# Participant-level mean RT per condition (repeated-measures structure)
pp = (
    d.groupby(['participant_id', 'condition'], observed=True)[rt_col]
    .mean()
    .reset_index()
)

# Wide format and keep only complete participants (all 3 conditions present)
wide = pp.pivot(index='participant_id', columns='condition', values=rt_col)
wide = wide.reindex(columns=conds).dropna()

print(f"Using RT column: {rt_col}")
print(f"Complete participants included: {wide.shape[0]}")

# Omnibus test: Friedman (non-parametric repeated-measures)
stat, p = friedmanchisquare(wide[conds[0]], wide[conds[1]], wide[conds[2]])
print(f"\nFriedman test: chi2={stat:.4f}, p={p:.6f}")

if p < 0.05:
    print("=> Significant overall difference across conditions.")
else:
    print("=> No significant overall difference across conditions.")

# Pairwise post-hoc: Wilcoxon signed-rank + Holm correction
pairs = list(combinations(conds, 2))
results = []
for a, b in pairs:
    w_stat, p_raw = wilcoxon(wide[a], wide[b], zero_method='wilcox')
    results.append({'pair': f'{a} vs {b}', 'W': w_stat, 'p_raw': p_raw})

post = pd.DataFrame(results).sort_values('p_raw').reset_index(drop=True)

# Holm correction
m = len(post)
holm_p = []
for i, p_raw in enumerate(post['p_raw'], start=1):
    holm_p.append(min(1.0, p_raw * (m - i + 1)))
# enforce monotonicity
for i in range(1, len(holm_p)):
    holm_p[i] = max(holm_p[i], holm_p[i - 1])

post['p_holm'] = holm_p
post['significant_0.05'] = post['p_holm'] < 0.05

print("\nPairwise Wilcoxon (Holm-corrected):")
print(post)

Using RT column: choice_rt
Complete participants included: 72

Friedman test: chi2=4.3611, p=0.112979
=> No significant overall difference across conditions.

Pairwise Wilcoxon (Holm-corrected):
                        pair       W     p_raw    p_holm  significant_0.05
0  exhaustive vs contrastive   783.5  0.002911  0.008732              True
1  unmodified vs contrastive   961.5  0.047915  0.095829             False
2   exhaustive vs unmodified  1157.0  0.378298  0.378298             False


In [32]:
import re
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

def holm_adjust(p_values: list[float]) -> list[float]:
    """Holm step-down adjustment (controls FWER)."""
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=float)
    prev = 0.0
    for rank, idx in enumerate(order, start=1):
        adj = min(1.0, float(p_values[idx]) * (m - rank + 1))
        prev = max(prev, adj)
        adjusted[idx] = prev
    return adjusted.tolist()

def paired_stats_table(wide: pd.DataFrame, conds: list[str], label: str) -> tuple[dict, pd.DataFrame]:
    """Return omnibus + pairwise results for a wide (pp x condition) table."""
    wide = wide.reindex(columns=conds)
    wide = wide.dropna()
    n = int(wide.shape[0])
    if n == 0:
        return ({'label': label, 'n_complete': 0, 'test': None, 'stat': np.nan, 'p': np.nan}, pd.DataFrame())

    if len(conds) == 2:
        a, b = conds
        w_stat, p_raw = wilcoxon(wide[a], wide[b], zero_method='wilcox')
        pairwise = pd.DataFrame([
            {'label': label, 'pair': f'{a} vs {b}', 'n': n, 'W': float(w_stat), 'p_raw': float(p_raw), 'p_holm_within_label': float(p_raw)}
        ])
        omnibus = {'label': label, 'n_complete': n, 'test': 'wilcoxon', 'stat': float(w_stat), 'p': float(p_raw)}
        return omnibus, pairwise

    # 3+ conditions: Friedman omnibus + Wilcoxon pairwise
    vals = [wide[c] for c in conds]
    stat, p = friedmanchisquare(*vals)
    pair_rows = []
    for a, b in combinations(conds, 2):
        w_stat, p_raw = wilcoxon(wide[a], wide[b], zero_method='wilcox')
        pair_rows.append({'label': label, 'pair': f'{a} vs {b}', 'n': n, 'W': float(w_stat), 'p_raw': float(p_raw)})
    pairwise = pd.DataFrame(pair_rows)
    if not pairwise.empty:
        pairwise = pairwise.sort_values('p_raw').reset_index(drop=True)
        pairwise['p_holm_within_label'] = holm_adjust(pairwise['p_raw'].tolist())
    omnibus = {'label': label, 'n_complete': n, 'test': 'friedman', 'stat': float(stat), 'p': float(p)}
    return omnibus, pairwise

# -----------------------------
# Region RTs: r1-r7 (participant-level repeated measures)
# -----------------------------
preferred_conds = ['exhaustive', 'unmodified', 'contrastive']
present_conds = [c for c in preferred_conds if c in events_df['condition'].astype(str).unique().tolist()]
if len(present_conds) < 2:
    raise ValueError(f"Need at least 2 conditions present; found: {present_conds}")

# Build a long table (participant × item × region) if not already available
if 'melted' in globals() and isinstance(melted, pd.DataFrame) and {'participant_id', 'item', 'condition', 'region'}.issubset(set(melted.columns)):
    reg_long = melted.copy()
else:
    region_cols = [c for c in events_df.columns if re.match(r"r\d+$", str(c))]
    reg_long = (
        events_df
        .melt(
            id_vars=['participant_id', 'item', 'condition'],
            value_vars=region_cols,
            var_name='region',
            value_name='reading_time',
        )
        .dropna(subset=['reading_time'])
    )

value_col = 'reading_time_wins' if 'reading_time_wins' in reg_long.columns else 'reading_time'
regions_to_test = [f"r{i}" for i in range(1, 8) if f"r{i}" in reg_long['region'].astype(str).unique().tolist()]

# Participant means per region × condition
pp_reg = (
    reg_long
    .loc[reg_long['condition'].astype(str).isin(present_conds) & reg_long['region'].astype(str).isin(regions_to_test)]
    .groupby(['participant_id', 'condition', 'region'], as_index=False, observed=True)[value_col]
    .mean()
    .rename(columns={value_col: 'rt'})
 )

omnibus_rows = []
pairwise_all = []
for region in regions_to_test:
    wide = (
        pp_reg[pp_reg['region'].astype(str).eq(region)]
        .pivot(index='participant_id', columns='condition', values='rt')
    )
    omnibus, pairwise = paired_stats_table(wide, present_conds, label=region)
    omnibus_rows.append(omnibus)
    if not pairwise.empty:
        pairwise_all.append(pairwise)

omnibus_df = pd.DataFrame(omnibus_rows)
if not omnibus_df.empty and omnibus_df['p'].notna().any():
    omnibus_df['p_holm_across_regions'] = holm_adjust(omnibus_df['p'].fillna(1.0).tolist())
    omnibus_df['significant_0.05'] = omnibus_df['p_holm_across_regions'] < 0.05
    omnibus_df = omnibus_df.sort_values(['p_holm_across_regions', 'p'], na_position='last').reset_index(drop=True)

pairwise_df = pd.concat(pairwise_all, ignore_index=True) if pairwise_all else pd.DataFrame()

print("Region reading times (per-participant means; repeated-measures nonparametric tests)")
print(f"Using conditions: {present_conds}")
print(f"Using column: {value_col}")
display(omnibus_df)
if not pairwise_df.empty:
    display(pairwise_df.sort_values(['label', 'p_holm_within_label', 'p_raw']).reset_index(drop=True))
else:
    print("No pairwise comparisons computed (insufficient data).")

# -----------------------------
# Choice RT: repeated-measures across conditions (mirrors the region approach)
# -----------------------------
rt_col = 'choice_rt_wins' if 'choice_rt_wins' in events_df.columns else 'choice_rt'
d = events_df.copy()
d = d[d['condition'].astype(str).isin(present_conds)]
d = d[d[rt_col].notna()].copy()
pp_choice = (
    d.groupby(['participant_id', 'condition'], observed=True)[rt_col]
    .mean()
    .reset_index()
)
wide_choice = pp_choice.pivot(index='participant_id', columns='condition', values=rt_col)
choice_omnibus, choice_pairwise = paired_stats_table(wide_choice, present_conds, label='choice_rt')

print("\nChoice RT (per-participant means; repeated-measures nonparametric tests)")
print(f"Using RT column: {rt_col}")
display(pd.DataFrame([choice_omnibus]))
if not choice_pairwise.empty:
    display(choice_pairwise.sort_values('p_holm_within_label').reset_index(drop=True))

Region reading times (per-participant means; repeated-measures nonparametric tests)
Using conditions: ['exhaustive', 'unmodified', 'contrastive']
Using column: reading_time_wins


,label,n_complete,test,stat,p,p_holm_across_regions,significant_0.05
0,r5,72,friedman,15.527778,0.000425,0.002974,True
1,r7,72,friedman,13.000000,0.001503,0.009021,True
2,r2,72,friedman,8.777778,0.012415,0.062073,False
3,r6,72,friedman,4.861111,0.087988,0.351952,False
4,r3,72,friedman,4.527778,0.103945,0.351952,False
5,r1,72,friedman,4.361111,0.112979,0.351952,False
6,r4,72,friedman,4.111111,0.128022,0.351952,False


,label,pair,n,W,p_raw,p_holm_within_label
0,r1,exhaustive vs contrastive,72,841.5,0.008013,0.024038
1,r1,unmodified vs contrastive,72,1102.5,0.235277,0.470553
2,r1,exhaustive vs unmodified,72,1135.0,0.315141,0.470553
3,r2,unmodified vs contrastive,72,772.0,0.002354,0.007061
4,r2,exhaustive vs unmodified,72,931.5,0.031835,0.063669
5,r2,exhaustive vs contrastive,72,1308.0,0.973140,0.973140
6,r3,exhaustive vs contrastive,72,833.0,0.006950,0.020850
7,r3,unmodified vs contrastive,72,977.0,0.058604,0.117208
8,r3,exhaustive vs unmodified,72,1133.0,0.309762,0.309762
9,r4,unmodified vs contrastive,72,825.5,0.006119,0.018358



Choice RT (per-participant means; repeated-measures nonparametric tests)
Using RT column: choice_rt


,label,n_complete,test,stat,p
0,choice_rt,72,friedman,4.361111,0.112979


,label,pair,n,W,p_raw,p_holm_within_label
0,choice_rt,exhaustive vs contrastive,72,783.5,0.002911,0.008732
1,choice_rt,unmodified vs contrastive,72,961.5,0.047915,0.095829
2,choice_rt,exhaustive vs unmodified,72,1157.0,0.378298,0.378298


In [33]:
print(omnibus_df)

  label  n_complete      test       stat         p  p_holm_across_regions  \
0    r5          72  friedman  15.527778  0.000425               0.002974   
1    r7          72  friedman  13.000000  0.001503               0.009021   
2    r2          72  friedman   8.777778  0.012415               0.062073   
3    r6          72  friedman   4.861111  0.087988               0.351952   
4    r3          72  friedman   4.527778  0.103945               0.351952   
5    r1          72  friedman   4.361111  0.112979               0.351952   
6    r4          72  friedman   4.111111  0.128022               0.351952   

   significant_0.05  
0              True  
1              True  
2             False  
3             False  
4             False  
5             False  
6             False  


In [34]:
omnibus_df

,label,n_complete,test,stat,p,p_holm_across_regions,significant_0.05
0,r5,72,friedman,15.527778,0.000425,0.002974,True
1,r7,72,friedman,13.000000,0.001503,0.009021,True
2,r2,72,friedman,8.777778,0.012415,0.062073,False
3,r6,72,friedman,4.861111,0.087988,0.351952,False
4,r3,72,friedman,4.527778,0.103945,0.351952,False
5,r1,72,friedman,4.361111,0.112979,0.351952,False
6,r4,72,friedman,4.111111,0.128022,0.351952,False


In [35]:
# --- Whole-sentence reading times: sum r1-r7 and test condition differences ---

import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon

# Fallbacks in case the earlier helper cell has not been run
if 'holm_adjust' not in globals():
    def holm_adjust(p_values: list[float]) -> list[float]:
        m = len(p_values)
        order = np.argsort(p_values)
        adjusted = np.empty(m, dtype=float)
        prev = 0.0
        for rank, idx in enumerate(order, start=1):
            adj = min(1.0, float(p_values[idx]) * (m - rank + 1))
            prev = max(prev, adj)
            adjusted[idx] = prev
        return adjusted.tolist()

if 'paired_stats_table' not in globals():
    def paired_stats_table(wide: pd.DataFrame, conds: list[str], label: str):
        wide = wide.reindex(columns=conds).dropna()
        n = int(wide.shape[0])

        if n == 0:
            return (
                {'label': label, 'n_complete': 0, 'test': None, 'stat': np.nan, 'p': np.nan},
                pd.DataFrame()
            )

        if len(conds) == 2:
            a, b = conds
            w_stat, p_raw = wilcoxon(wide[a], wide[b], zero_method='wilcox')
            pairwise = pd.DataFrame([{
                'label': label,
                'pair': f'{a} vs {b}',
                'n': n,
                'W': float(w_stat),
                'p_raw': float(p_raw),
                'p_holm_within_label': float(p_raw),
            }])
            omnibus = {'label': label, 'n_complete': n, 'test': 'wilcoxon', 'stat': float(w_stat), 'p': float(p_raw)}
            return omnibus, pairwise

        stat, p = friedmanchisquare(*[wide[c] for c in conds])

        pair_rows = []
        for a, b in combinations(conds, 2):
            w_stat, p_raw = wilcoxon(wide[a], wide[b], zero_method='wilcox')
            pair_rows.append({
                'label': label,
                'pair': f'{a} vs {b}',
                'n': n,
                'W': float(w_stat),
                'p_raw': float(p_raw),
            })

        pairwise = pd.DataFrame(pair_rows)
        if not pairwise.empty:
            pairwise = pairwise.sort_values('p_raw').reset_index(drop=True)
            pairwise['p_holm_within_label'] = holm_adjust(pairwise['p_raw'].tolist())

        omnibus = {'label': label, 'n_complete': n, 'test': 'friedman', 'stat': float(stat), 'p': float(p)}
        return omnibus, pairwise

# Conditions to compare
preferred_conds = ['exhaustive', 'unmodified', 'contrastive']
present_conds = [c for c in preferred_conds if c in events_df['condition'].astype(str).unique().tolist()]
if len(present_conds) < 2:
    raise ValueError(f"Need at least 2 conditions present; found: {present_conds}")

# Sentence RT = sum of r1-r7
region_cols = [f"r{i}" for i in range(1, 8) if f"r{i}" in events_df.columns]
if len(region_cols) == 0:
    raise ValueError("No region columns r1-r7 found in events_df.")

sentence_df = events_df.copy()
sentence_df[region_cols] = sentence_df[region_cols].apply(pd.to_numeric, errors='coerce')

# Require all present regions for a valid whole-sentence RT
sentence_df['sentence_rt'] = sentence_df[region_cols].sum(axis=1, min_count=len(region_cols))

# Keep only tested conditions and non-missing sentence RTs
sentence_df = sentence_df[
    sentence_df['condition'].astype(str).isin(present_conds) &
    sentence_df['sentence_rt'].notna()
].copy()

# Participant-level mean whole-sentence RT per condition
pp_sentence = (
    sentence_df
    .groupby(['participant_id', 'condition'], observed=True)['sentence_rt']
    .mean()
    .reset_index()
)

wide_sentence = pp_sentence.pivot(index='participant_id', columns='condition', values='sentence_rt')

sentence_omnibus, sentence_pairwise = paired_stats_table(
    wide_sentence,
    present_conds,
    label='whole_sentence_rt'
)

# Descriptives
sentence_desc = (
    pp_sentence
    .groupby('condition', observed=True)['sentence_rt']
    .agg(mean='mean', sd='std', n='size')
    .reset_index()
)

print("Whole-sentence reading times (sum of r1-r7)")
print(f"Using conditions: {present_conds}")
print(f"Using regions: {region_cols}")

display(sentence_desc)
display(pd.DataFrame([sentence_omnibus]))

if not sentence_pairwise.empty:
    display(sentence_pairwise.sort_values('p_holm_within_label').reset_index(drop=True))
else:
    print("No pairwise comparisons computed.")

Whole-sentence reading times (sum of r1-r7)
Using conditions: ['exhaustive', 'unmodified', 'contrastive']
Using regions: ['r1', 'r2', 'r3', 'r4', 'r5', 'r6', 'r7']


,condition,mean,sd,n
0,exhaustive,3422.712963,1182.694485,72
1,unmodified,3639.675926,1317.030060,72
2,contrastive,3564.185185,1277.077494,72


,label,n_complete,test,stat,p
0,whole_sentence_rt,72,friedman,3.583333,0.166682


,label,pair,n,W,p_raw,p_holm_within_label
0,whole_sentence_rt,exhaustive vs unmodified,72,879.0,0.014643,0.043929
1,whole_sentence_rt,exhaustive vs contrastive,72,913.5,0.024609,0.049218
2,whole_sentence_rt,unmodified vs contrastive,72,1050.0,0.138477,0.138477


In [36]:
# --- Region reading times by condition, split by choice (A vs B) ---
import plotly.express as px

region_cols = [f"r{i}" for i in range(1, 8) if f"r{i}" in events_df.columns]
if not region_cols:
    raise ValueError("No region columns r1-r7 found in events_df.")

rt_choice_df = (
    events_df
    .loc[
        events_df['condition'].astype(str).isin(present_conds)
        & events_df['chosen_type'].astype(str).isin(['A', 'B'])
    , ['participant_id', 'condition', 'chosen_type'] + region_cols]
    .copy()
)

rt_choice_long = (
    rt_choice_df
    .melt(
        id_vars=['participant_id', 'condition', 'chosen_type'],
        value_vars=region_cols,
        var_name='region',
        value_name='reading_time',
    )
    .dropna(subset=['reading_time'])
)

rt_choice_long['reading_time'] = pd.to_numeric(rt_choice_long['reading_time'], errors='coerce')
rt_choice_long = rt_choice_long[rt_choice_long['reading_time'] > 0].copy()
rt_choice_long['log_rt'] = np.log(rt_choice_long['reading_time'])

rt_choice_long['region'] = pd.Categorical(
    rt_choice_long['region'],
    categories=region_cols,
    ordered=True,
)

pp_rt_choice = (
    rt_choice_long
    .groupby(['participant_id', 'chosen_type', 'condition', 'region'], observed=True)['log_rt']
    .mean()
    .reset_index()
)

rt_choice_plot = (
    pp_rt_choice
    .groupby(['chosen_type', 'condition', 'region'], observed=True)['log_rt']
    .agg(mean_log_rt='mean', sem_log_rt='sem', n='size')
    .reset_index()
)

cond_color_map = COND_COLORS if 'COND_COLORS' in globals() else {
    'exhaustive': '#5e81ac',
    'unmodified': '#a3be8c',
    'contrastive': '#ebcb8b',
}

fig = px.line(
    rt_choice_plot,
    x='region',
    y='mean_log_rt',
    color='condition',
    facet_col='chosen_type',
    markers=True,
    error_y='sem_log_rt',
    category_orders={
        'region': region_cols,
        'condition': present_conds,
        'chosen_type': ['A', 'B'],
    },
    color_discrete_map=cond_color_map,
    labels={
        'region': 'Region',
        'mean_log_rt': 'log RT',
        'condition': 'Condition',
        'chosen_type': 'Choice',
    },
    title='Region Reading Times by Condition, Split by Choice<br><sup>Mean log RT across participants; error bars show SEM</sup>',
)

fig.for_each_annotation(
    lambda a: a.update(text=f"Chose {a.text.split('=')[-1]}")
)

fig.update_layout(
    template='nord_light_paper' if 'nord_template' in globals() else 'plotly_white',
    height=500,
    width=1100,
    legend=dict(orientation='h', x=0.5, xanchor='center', y=1.12, yanchor='bottom'),
    margin=dict(l=60, r=30, t=90, b=50),
)
fig.update_xaxes(title_text='Region')
fig.update_yaxes(title_text='log RT')

fig.show()

fig.write_html(plots / 'region_log_rt_by_condition_split_by_choice.html', include_plotlyjs='cdn')
try:
    fig.write_image(plots / 'region_log_rt_by_condition_split_by_choice.png', scale=3)
except Exception:
    pass

In [38]:
# --- Whole-sentence reading times by condition, split by choice (A vs B) ---

if 'sentence_rt' not in events_df.columns:
    region_cols = [f"r{i}" for i in range(1, 8) if f"r{i}" in events_df.columns]
    if not region_cols:
        raise ValueError("Neither `sentence_rt` nor region columns r1-r7 were found in events_df.")
    events_df['sentence_rt'] = events_df[region_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=len(region_cols))

sentence_choice_df = (
    events_df
    .loc[
        events_df['condition'].astype(str).isin(present_conds)
        & events_df['chosen_type'].astype(str).isin(['A', 'B']),
        ['participant_id', 'condition', 'chosen_type', 'sentence_rt']
    ]
    .copy()
)

sentence_choice_df['sentence_rt'] = pd.to_numeric(sentence_choice_df['sentence_rt'], errors='coerce')
sentence_choice_df = sentence_choice_df[sentence_choice_df['sentence_rt'] > 0].copy()
sentence_choice_df['log_sentence_rt'] = np.log(sentence_choice_df['sentence_rt'])

# Same aggregation logic as before: participant-level mean per condition × choice
pp_sentence_choice = (
    sentence_choice_df
    .groupby(['participant_id', 'chosen_type', 'condition'], observed=True)['log_sentence_rt']
    .mean()
    .reset_index()
)

cond_color_map = COND_COLORS if 'COND_COLORS' in globals() else {
    'exhaustive': '#5e81ac',
    'unmodified': '#a3be8c',
    'contrastive': '#ebcb8b',
}

fig = px.box(
    pp_sentence_choice,
    x='condition',
    y='log_sentence_rt',
    color='condition',
    facet_col='chosen_type',
    points='all',
    category_orders={
        'condition': present_conds,
        'chosen_type': ['A', 'B'],
    },
    color_discrete_map=cond_color_map,
    labels={
        'condition': 'Condition',
        'log_sentence_rt': 'log sentence RT',
        'chosen_type': 'Choice',
    },
    title='Whole-Sentence Reading Times by Condition, Split by Choice<br><sup>Participant-level mean log sentence RTs</sup>',
)

fig.for_each_annotation(
    lambda a: a.update(text=f"Chose {a.text.split('=')[-1]}")
)

fig.update_traces(
    jitter=0.25,
    pointpos=0,
    marker=dict(size=6, opacity=0.7),
    line=dict(width=1.5),
)

fig.update_layout(
    template='nord_light_paper' if 'nord_template' in globals() else 'plotly_white',
    height=500,
    width=1000,
    margin=dict(l=60, r=30, t=90, b=50),
    showlegend=False,
)

fig.update_xaxes(title_text='Condition')
fig.update_yaxes(title_text='log sentence RT')

fig.show()

fig.write_html(plots / 'sentence_log_rt_by_condition_split_by_choice.html', include_plotlyjs='cdn')
try:
    fig.write_image(plots / 'sentence_log_rt_by_condition_split_by_choice.png', scale=3)
except Exception:
    pass

## Choices

In [87]:
# Choice ratios: per-condition donut using chosen_type column
choice_plot_df = (
    events_df
    .loc[~events_df['condition'].isin(['practice', 'attention-check'])]
    .assign(chosen_type=events_df['chosen_type'].fillna('None'))
)

choice_counts = (
    choice_plot_df
    .groupby(['condition', 'chosen_type'], observed=True)
    .size()
    .reset_index(name='count')
)

fig = px.pie(
    choice_counts,
    names='chosen_type',
    values='count',
    color='chosen_type',
    facet_col='condition',
    hole=0.5,
    title='Choice of Exclusive (A) or Contrastive (B) types of stimuli',
    color_discrete_map={'A': '#5e81ac', 'B': '#ebcb8b', 'None': '#4c566a'}
)

# show both percent and raw count on the wedge and in hover
fig.update_traces(
    textinfo='percent+label',
    # texttemplate="%{label}<br>%{percent} <br>(%{value})",
    texttemplate="%{percent}<br>(%{value})",
    hovertemplate="%{label}: %{value} (%{percent})<extra></extra>"
)

fig.update_layout(
    **base_layout,
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.075, yanchor='top')
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1], y=0, yanchor='top'))
fig.show()

fig.write_image(plots / "choice_type_per_condition.png", scale=3)
fig.write_html(plots / "choice_type_per_condition.html", include_plotlyjs='cdn')

In [88]:
# --- Choices: plot choice proportions + significance ---
# We treat choices as repeated-measures: per participant, compute the proportion of "A" choices
# within each condition (excluding missing / None), then run the same omnibus+pairwise tests.

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ALPHA = 0.05


def _p_to_stars(p: float) -> str:
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


preferred_conds = ["exhaustive", "unmodified", "contrastive"]
conds = [c for c in preferred_conds if c in events_df["condition"].astype(str).unique().tolist()]

if "chosen_type" not in events_df.columns:
    print("No `chosen_type` column found; cannot compute choice proportions.")
else:
    d = events_df.loc[events_df["condition"].astype(str).isin(conds)].copy()
    d["chosen_type"] = d["chosen_type"].astype(str)

    # Keep only actual A/B choices (drop None / nan)
    d = d.loc[d["chosen_type"].isin(["A", "B"])].copy()
    if d.empty:
        print("No A/B choices found after filtering.")
    else:
        d["is_A"] = (d["chosen_type"] == "A").astype(float)

        pp_choice_prop = (
            d.groupby(["participant_id", "condition"], observed=True)["is_A"]
            .mean()
            .reset_index()
            .rename(columns={"is_A": "prop_A"})
        )

        wide_prop = pp_choice_prop.pivot(index="participant_id", columns="condition", values="prop_A")
        choice_prop_omni, choice_prop_pairwise = paired_stats_table(wide_prop, conds, label="choice_prop_A")

        # Summary for plotting
        summ = (
            pp_choice_prop
            .groupby("condition", observed=True)["prop_A"]
            .agg(mean="mean", sem="sem", n="size")
            .reindex(conds)
            .reset_index()
        )

        template_to_use = nord_template if "nord_template" in globals() else "plotly_white"
        cond_color_map = COND_COLORS if "COND_COLORS" in globals() else {}

        fig = make_subplots(
            rows=2,
            cols=1,
            vertical_spacing=0.15,
            subplot_titles=(
                "Proportion of A choices by condition (mean ± SEM across participants)",
                "Pairwise tests on per-participant proportions (Holm-corrected)",
            ),
        )

        # Row 1: mean proportions
        fig.add_trace(
            go.Scatter(
                x=summ["condition"].astype(str).tolist(),
                y=summ["mean"].tolist(),
                mode="lines+markers",
                name="prop(A)",
                marker=dict(color="#5e81ac"),
                line=dict(color="#5e81ac"),
                error_y=dict(type="data", array=summ["sem"].fillna(0).tolist(), visible=True),
                hovertemplate="Condition=%{x}<br>Mean prop(A)=%{y:.3f}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=1,
        )
        fig.update_yaxes(title_text="prop(A)", range=[0, 1], row=1, col=1)

        # Row 2: pairwise bars (-log10 p)
        if choice_prop_pairwise is None or len(choice_prop_pairwise) == 0:
            fig.add_annotation(
                text="No pairwise comparisons available (insufficient complete cases).",
                xref="paper",
                yref="paper",
                x=0.5,
                y=0.1,
                showarrow=False,
                row=2,
                col=1,
            )
        else:
            pw = choice_prop_pairwise.copy()
            pcol = "p_holm_within_label" if "p_holm_within_label" in pw.columns else ("p_holm" if "p_holm" in pw.columns else "p_raw")
            pw[pcol] = pd.to_numeric(pw[pcol], errors="coerce")
            pw["neglog10p"] = -np.log10(pw[pcol].clip(lower=1e-300))
            pw["stars"] = pw[pcol].map(_p_to_stars)

            # ensure consistent ordering of pairs
            pair_order = []
            for i in range(len(conds)):
                for j in range(i + 1, len(conds)):
                    pair_order.append(f"{conds[i]} vs {conds[j]}")
            if "pair" in pw.columns:
                pw["pair"] = pd.Categorical(pw["pair"].astype(str), categories=pair_order, ordered=True)
                pw = pw.sort_values("pair")
                x = pw["pair"].astype(str).tolist()
            else:
                x = list(range(len(pw)))

            fig.add_trace(
                go.Bar(
                    x=x,
                    y=pw["neglog10p"].tolist(),
                    text=pw["stars"].tolist(),
                    textposition="outside",
                    name="pairwise",
                    marker=dict(color="#a3be8c"),
                    hovertemplate="Pair=%{x}<br>-log10(p)=%{y:.2f}<extra></extra>",
                    showlegend=False,
                ),
                row=2,
                col=1,
            )
            fig.add_hline(
                y=-np.log10(ALPHA),
                line_dash="dash",
                line_color="#4c566a",
                row=2,
                col=1,
            )
            fig.update_yaxes(title_text="-log10(p)", row=2, col=1)

        fig.update_layout(
            template=template_to_use,
            height=700,
            title=(
                "Choices: A-choice proportion by condition + significance"
                "<br><sup>Computed per participant; stars mark p<.05; dashed line is alpha=.05 on -log10 scale</sup>"
            ),
            margin=dict(l=70, r=20, t=90, b=60),
        )

        fig.show()

        # quick tables for reference
        display(pd.DataFrame([choice_prop_omni]))
        if choice_prop_pairwise is not None and len(choice_prop_pairwise) > 0:
            display(choice_prop_pairwise)


,label,n_complete,test,stat,p
0,choice_prop_A,72,friedman,88.822335,5.157925e-20


,label,pair,n,W,p_raw,p_holm_within_label
0,choice_prop_A,unmodified vs contrastive,72,26.0,2.791936e-11,8.375809e-11
1,choice_prop_A,exhaustive vs contrastive,72,21.5,8.789563e-11,1.757913e-10
2,choice_prop_A,exhaustive vs unmodified,72,69.0,4.564353e-01,4.564353e-01


In [89]:
# Choice ratios: per-condition donut using chosen_type column
choice_plot_df = (
    events_df
    .loc[~events_df['condition'].isin(['practice', 'attention-check'])]
    .assign(chosen_type=events_df['chosen_type'].fillna('None'))
)

choice_counts = (
    choice_plot_df
    .groupby(['condition', 'chosen_type'], observed=True)
    .size()
    .reset_index(name='count')
)

fig = px.pie(
    choice_counts,
    names='chosen_type',
    values='count',
    color='chosen_type',
    facet_col='condition',
    hole=0.5,
    title=None,
    color_discrete_map={'A': '#5e81ac', 'B': '#ebcb8b', 'None': '#4c566a'}
)

# show both percent and raw count on the wedge and in hover
fig.update_traces(
    textinfo='percent+label',
    # texttemplate="%{label}<br>%{percent} <br>(%{value})",
    texttemplate="%{percent}<br>(%{value})",
    hovertemplate="%{label}: %{value} (%{percent})<extra></extra>"
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1], y=0, yanchor='top'))

base_font_size = (fig.layout.font.size or 14) + 4
legend_font_size = (fig.layout.legend.font.size if fig.layout.legend and fig.layout.legend.font and fig.layout.legend.font.size else base_font_size) + 4
title_font_size = (fig.layout.title.font.size or base_font_size) + 4

fig.update_layout(
    **base_layout,
    font=dict(size=base_font_size),
    title=dict(text=None, font=dict(size=title_font_size)),
    legend=dict(
        orientation='v',
        x=1,
        xanchor='left',
        y=1,
        yanchor='top',
        font=dict(size=legend_font_size)
    )
)

# Increase width to 1200
fig.update_layout(width=1200)

if fig.layout.annotations:
    annot_font_size = (fig.layout.annotations[0].font.size or base_font_size) + 4
    fig.update_annotations(font=dict(size=annot_font_size))

trace_font_size = (fig.data[0].textfont.size if fig.data and getattr(fig.data[0], 'textfont', None) and fig.data[0].textfont.size else base_font_size) + 4
fig.update_traces(textfont=dict(size=trace_font_size))

fig.show()

fig.write_image(plots / "choice_type_per_condition.png", scale=3)
fig.write_html(plots / "choice_type_per_condition.html", include_plotlyjs='cdn')

In [90]:
# Choice ratios: per-condition donut using chosen_type column
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

choice_plot_df = (
    events_df
    .loc[~events_df['condition'].isin(['practice', 'attention-check'])]
    .assign(chosen_type=events_df['chosen_type'].fillna('None'))
)

choice_counts = (
    choice_plot_df
    .groupby(['condition', 'chosen_type'], observed=True)
    .size()
    .reset_index(name='count')
)

fig = px.pie(
    choice_counts,
    names='chosen_type',
    values='count',
    color='chosen_type',
    facet_col='condition',
    hole=0.5,
    title=None,
    color_discrete_map={'A': '#5e81ac', 'B': '#ebcb8b', 'None': '#4c566a'}
)

# show both percent and raw count on the wedge and in hover
fig.update_traces(
    textinfo='percent+label',
    # texttemplate="%{label}<br>%{percent} <br>(%{value})",
    texttemplate="%{percent}<br>(%{value})",
    hovertemplate="%{label}: %{value} (%{percent})<extra></extra>"
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1], y=0, yanchor='top'))

base_font_size = (fig.layout.font.size or 14) + 4
legend_font_size = (
    fig.layout.legend.font.size
    if fig.layout.legend and fig.layout.legend.font and fig.layout.legend.font.size
    else base_font_size
) + 4
title_font_size = (fig.layout.title.font.size or base_font_size) + 4

# Base layout (keep legend definition, but we'll overwrite its position below)
fig.update_layout(
    **base_layout,
    font=dict(size=base_font_size),
    title=dict(text=None, font=dict(size=title_font_size)),
    legend=dict(
        orientation='v',
        font=dict(size=legend_font_size),
    )
)

# Increase width to 1200
# fig.update_layout(width=1200)

if fig.layout.annotations:
    annot_font_size = (fig.layout.annotations[0].font.size or base_font_size) + 4
    fig.update_annotations(font=dict(size=annot_font_size))

trace_font_size = (
    fig.data[0].textfont.size
    if fig.data and getattr(fig.data[0], 'textfont', None) and fig.data[0].textfont.size
    else base_font_size
) + 4
fig.update_traces(textfont=dict(size=trace_font_size))

# -----------------------------
# Put the legend inside the *middle* donut (middle facet)
# -----------------------------
centers = []
for tr in fig.data:
    if isinstance(tr, go.Pie) and tr.domain is not None and tr.domain.x and tr.domain.y:
        x0, x1 = tr.domain.x
        y0, y1 = tr.domain.y
        centers.append(((x0 + x1) / 2.0, (y0 + y1) / 2.0))

if centers:
    # unique x-centers correspond to facet columns; pick the middle one
    xs = sorted({round(x, 6) for x, _ in centers})
    mid_x = xs[len(xs) // 2]

    # y-center for the traces in that middle facet (usually just one)
    ys_mid = [y for x, y in centers if round(x, 6) == mid_x]
    mid_y = float(np.mean(ys_mid)) if ys_mid else 0.5

    fig.update_layout(
        legend=dict(
            x=mid_x,
            y=mid_y,
            xanchor='center',
            yanchor='middle',
            orientation='v',
            bgcolor='rgba(255,255,255,0.75)',
            bordercolor='rgba(0,0,0,0)',
            borderwidth=1,
            itemsizing='constant',
        )
    )

fig.show()

fig.write_image(plots / "choice_type_per_condition.png", scale=3)
fig.write_html(plots / "choice_type_per_condition.html", include_plotlyjs='cdn')

## Eye-tracking

### Download ET data

mondo1 server is unreachable as of 2026.02.02.

In [91]:
# # # DO NOT REMOVE: this function downloads eyetracking data files per participant

# # Download and save eyetracking data files per participant
# def download_and_save_eyetracking_data():
#     out_dir = 'eyetracking_data'
#     eturl = "https://mondo1.dreamhosters.com/script.php?experiment="
#     for _, row in participants_df.iterrows():
#         et_filename = row['eyetracker_filename']
#         participant_id = row['participant_id']
#         if pd.notnull(et_filename) and pd.notnull(participant_id):
#             et_file = eturl + et_filename
#             try:
#                 r = requests.get(et_file, timeout=15)
#                 r.raise_for_status()
#                 df_et = pd.read_csv(StringIO(r.text))
#                 df_et.to_csv(f"{out_dir}/{participant_id}.csv", index=False)
#             except Exception as e:
#                 print(f"Failed for {participant_id}: {e}")
#     print("Done!")
# # Example usage:
# download_and_save_eyetracking_data()

In [92]:
# # DO NOT REMOVE: this function fixes eyetracking csv trial numbering

# # Fix eyetracking csv trial numbering
# et_dir = os.path.join(ROOT, 'eyetracking_data')

# for csv_file in glob(os.path.join(et_dir, '*.csv')):
#     df = pd.read_csv(csv_file)
#     # Apply fix_trial_numbering WITHOUT changing row order
#     trial_col = 'trial' if 'trial' in df.columns else [c for c in df.columns if 'trial' in c][0]
#     # Map 9,10,11 to 6,7,8
#     mapping = {9: 6, 10: 7, 11: 8}
#     df[trial_col] = df[trial_col].replace(mapping)
#     # Find the index of the last occurrence of 8
#     last_mapped_idx = df[df[trial_col] == 8].index.max()
#     if last_mapped_idx is not None and last_mapped_idx + 1 < len(df):
#         next_trial = 10
#         prev_trial_val = None
#         for i in range(last_mapped_idx + 1, len(df)):
#             current_trial_val = df.at[i, trial_col]
#             if prev_trial_val is not None and current_trial_val != prev_trial_val:
#                 next_trial += 1
#             df.at[i, trial_col] = next_trial
#             prev_trial_val = current_trial_val
#     df.to_csv(csv_file, index=False)
# print("All eyetracking files fixed and overwritten.")

## Parse ET data

In [93]:
# Eye tracking data
et_dir = os.path.join(ROOT, 'eyetracking_data')

def parse_eyetracking_file(filepath):
    df = pd.read_csv(filepath)  # read one participant CSV
    df.columns = [c.lower() for c in df.columns]  # normalize column names
    trial_col = 'trial' if 'trial' in df.columns else [c for c in df.columns if 'trial' in c][0]  # find trial column
    if trial_col != 'trial':
        df = df.rename(columns={trial_col: 'trial'})  # ensure consistent column name
    return df

def load_eyetracking_directory(et_directory, skip_ids=None):
    records = []  # collected per-participant frames
    skip_ids = set(skip_ids or [])  # convert skip list to set
    for csv_file in glob(os.path.join(et_directory, '*.csv')):
        participant_id = os.path.splitext(os.path.basename(csv_file))[0]  # derive participant id from filename
        if participant_id in skip_ids:
            continue  # skip flagged participants
        df = parse_eyetracking_file(csv_file)  # load participant samples
        df['participant_id'] = participant_id  # keep participant id in rows
        records.append(df)
    if not records:
        return pd.DataFrame()  # handle empty directory case
    return pd.concat(records, ignore_index=True)  # combine all samples

raw_et = load_eyetracking_directory(et_dir, list_of_participants_to_remove)  # load all usable samples

if 'unnamed: 4' in raw_et.columns:
    raw_et = raw_et.drop(columns=['unnamed: 4'])  # drop stray export column

# Anonymize participant IDs in eyetracking dataframe using participant_id_map
raw_et['participant_id'] = raw_et['participant_id'].map(participant_id_map)
raw_et

,trial,times,_left_canvas,_right_canvas,participant_id
0,6,0,0,1,participant_001
1,6,25,0,1,participant_001
2,6,51,0,1,participant_001
3,6,84,1,0,participant_001
4,6,117,0,1,participant_001
...,...,...,...,...,...
132393,50,649,0,1,participant_072
132394,50,693,0,1,participant_072
132395,50,735,0,1,participant_072
132396,50,776,0,1,participant_072


## Cleaning and classifying fixations

In [94]:
def samples_to_aoi_episodes(samples: pd.DataFrame,
                            allow_neutral_bridge: bool = True,
                            max_neutral_bridge_ms: float | None = None,
                            min_sample_duration_ms: float | None = None,
                            include_neutral_episodes: bool = False) -> pd.DataFrame:
    """
    Collapse raw eyetracking samples into contiguous AOI episodes, optionally
    merging same-side episodes separated only by a neutral (no AOI) gap.

    Input columns required:
        participant_id, trial, times, _left_canvas, _right_canvas

    Output columns:
        participant_id, trial, AOI, start_time, end_time, duration_ms

    Parameters:
        allow_neutral_bridge (bool): If True, a neutral stretch flanked by the
            same AOI (left or right) is absorbed and the two sides merge.
        max_neutral_bridge_ms (float|None): Optional upper limit on the total
            duration (ms) of a neutral stretch that can be bridged. None = no limit.
        min_sample_duration_ms (float|None): If set, drop any EPISODES whose total
            duration is < this threshold, applied as the LAST step (after calculations).
        include_neutral_episodes (bool): If True, keep neutral (no AOI) episodes
            in the returned DataFrame as AOI='neutral'. If False, drop them.
    """
    need = {'participant_id', 'trial', 'times', '_left_canvas', '_right_canvas'}
    missing = need - set(samples.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    df = samples.copy()
    df = df.sort_values(['participant_id', 'trial', 'times']).reset_index(drop=True)

    # Helper: per-group (pid×trial) sample durations from timestamps
    def _durations(series: pd.Series) -> pd.Series:
        arr = series.to_numpy(dtype=float)
        n = arr.size
        if n == 0:
            return pd.Series([], index=series.index, dtype=float)
        if n == 1:
            return pd.Series([0.0], index=series.index, dtype=float)
        diffs = np.diff(arr, prepend=arr[0])
        if n > 1:
            diffs[0] = diffs[1]
            diffs[-1] = diffs[-2]
        diffs = np.clip(diffs, a_min=0.0, a_max=None)
        return pd.Series(diffs, index=series.index, dtype=float)

    # 1) Compute durations (no dropping here)
    df['sample_duration_ms'] = (
        df.groupby(['participant_id', 'trial'], sort=False)['times']
          .transform(_durations)
    )

    # 2) AOI classification
    df['AOI'] = np.where(
        (df['_left_canvas'] == 1) & (df['_right_canvas'] == 0), 'left',
        np.where(
            (df['_right_canvas'] == 1) & (df['_left_canvas'] == 0), 'right',
            None
        )
    )

    # 3) Optional neutral bridging
    if allow_neutral_bridge:
        for (pid, tr), idx in df.groupby(['participant_id', 'trial'], sort=False).groups.items():
            g = df.loc[idx]
            if g.empty:
                continue
            mask = g['AOI'].isna().to_numpy()
            if not mask.any():
                continue
            i = 0
            n = len(g)
            while i < n:
                if not mask[i]:
                    i += 1
                    continue
                # neutral run [start..end]
                start = i
                while i + 1 < n and mask[i + 1]:
                    i += 1
                end = i

                prev_i = start - 1
                next_i = end + 1
                if prev_i >= 0 and next_i < n:
                    prev_aoi = g['AOI'].iloc[prev_i]
                    next_aoi = g['AOI'].iloc[next_i]
                    if (prev_aoi is not None) and (next_aoi is not None) and (prev_aoi == next_aoi):
                        neutral_duration = float(g['sample_duration_ms'].iloc[start:end+1].sum())
                        if (max_neutral_bridge_ms is None) or (neutral_duration <= float(max_neutral_bridge_ms)):
                            # Bridge: assign neutral samples to the flanking AOI
                            bridge_idx = g.index[start:end+1]
                            df.loc[bridge_idx, 'AOI'] = prev_aoi
                i += 1

    # 4) Recalculate sample durations AFTER bridging
    df['sample_duration_ms'] = (
        df.groupby(['participant_id', 'trial'], sort=False)['times']
          .transform(_durations)
    )
    
    # 5) Mark episode boundaries. Neutral samples are contiguous and will be
    #    collapsed into a single episode by this segmentation.
    is_new_group = (
        (df['participant_id'] != df['participant_id'].shift()) |
        (df['trial'] != df['trial'].shift())
    )
    # Treat neutral (None/NaN) as the same value when detecting changes
    aoi_cmp = df['AOI'].fillna('__neutral__')
    prev_cmp = aoi_cmp.shift().fillna('__neutral__')
    is_break = is_new_group | (aoi_cmp != prev_cmp)
    episode_id = is_break.cumsum()

    rows = []
    for (pid, tr, eid), g in df.groupby(['participant_id', 'trial', episode_id], sort=False):
        aoi = g['AOI'].iloc[0]
        if aoi is None:
            if not include_neutral_episodes:
                continue
            out_aoi = 'neutral'
        else:
            out_aoi = aoi

        start_time = int(g['times'].iloc[0])
        duration_ms = float(g['sample_duration_ms'].sum())
        # Derive end_time from start_time + duration to avoid start==end artifacts
        end_time = int(start_time + round(duration_ms))

        rows.append({
            'participant_id': pid,
            'trial': tr,
            'AOI': out_aoi,
            'start_time': start_time,
            'end_time': end_time,
            'duration_ms': duration_ms
        })

    episodes_df = pd.DataFrame(rows)

    # 6) FINAL STEP: drop too-short episodes (after all calculations)
    if min_sample_duration_ms is not None and not episodes_df.empty:
        thr = float(min_sample_duration_ms)
        episodes_df = episodes_df[episodes_df['duration_ms'] >= thr].copy()

    # 7) Post-pass: coalesce adjacent neutral episodes within pid×trial
    if include_neutral_episodes and not episodes_df.empty:
        episodes_df = episodes_df.sort_values(['participant_id', 'trial', 'start_time'], kind='mergesort')

        def _coalesce_group(g: pd.DataFrame) -> pd.DataFrame:
            rows = []
            cur = None
            for r in g.itertuples(index=False):
                d = {
                    'participant_id': r.participant_id,
                    'trial': r.trial,
                    'AOI': r.AOI,
                    'start_time': int(r.start_time),
                    'end_time': int(r.end_time),
                    'duration_ms': float(r.duration_ms),
                }
                if cur is None:
                    cur = d
                    continue
                if cur['AOI'] == 'neutral' and d['AOI'] == 'neutral':
                    # merge consecutive neutrals
                    cur['duration_ms'] += d['duration_ms']
                    # end_time will be recomputed below from start_time + duration_ms
                else:
                    rows.append(cur)
                    cur = d
            if cur is not None:
                rows.append(cur)
            out = pd.DataFrame(rows)
            if not out.empty:
                out['duration_ms'] = out['duration_ms'].astype(float)
                out['start_time'] = out['start_time'].astype(int)
                out['end_time'] = (out['start_time'] + out['duration_ms'].round().astype(int)).astype(int)
            return out

        episodes_df = (
            episodes_df
            .groupby(['participant_id', 'trial'], sort=False, group_keys=False)
            .apply(_coalesce_group, include_groups=True)  # keep grouping cols and silence warning
            .reset_index(drop=True)
        )

    return episodes_df

# Example usage:
episodes = samples_to_aoi_episodes(
    raw_et[['participant_id','trial','times','_left_canvas','_right_canvas']],
    allow_neutral_bridge=True,         # do not bridge neutral gaps
    max_neutral_bridge_ms=50,            # no bridging allowed 
    min_sample_duration_ms=60,          # drop episodes < 80 ms at the very end
    include_neutral_episodes=False      # drop neutral episodes
)
print(episodes.head(20))

episodes

     participant_id  trial    AOI  start_time  end_time  duration_ms
0   participant_001      6  right           0        76         76.0
2   participant_001      6  right         117       418        301.0
3   participant_001      6   left         419      1238        819.0
6   participant_001      6  right        1337      1403         66.0
7   participant_001      6   left        1404      1957        553.0
9   participant_001      6   left        1989      2089        100.0
10  participant_001      6  right        2123      2224        101.0
11  participant_001      6   left        2222      2456        234.0
12  participant_001      7   left           0        97         97.0
15  participant_001      7   left         271       454        183.0
17  participant_001      7   left         500       977        477.0
18  participant_001      7  right         975      1167        192.0
20  participant_001      7  right        1204      1328        124.0
22  participant_001      7  right 

,participant_id,trial,AOI,start_time,end_time,duration_ms
0,participant_001,6,right,0,76,76.0
2,participant_001,6,right,117,418,301.0
3,participant_001,6,left,419,1238,819.0
6,participant_001,6,right,1337,1403,66.0
7,participant_001,6,left,1404,1957,553.0
...,...,...,...,...,...,...
17725,participant_072,46,left,132,1011,879.0
17726,participant_072,47,right,0,954,954.0
17727,participant_072,48,right,0,596,596.0
17728,participant_072,49,left,0,950,950.0


## Analysis

In [95]:
# Summary statistics for episodes (focus on 'duration_ms')
if 'episodes' not in globals():
    raise RuntimeError("episodes DataFrame not found in the notebook environment.")

ep = episodes.copy()
if ep.empty:
    print("episodes is empty")
else:
    # Basic overview
    print("Episodes shape:", ep.shape)
    print("\nOverall duration (ms) descriptive stats:")
    print(ep['duration_ms'].describe(percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]))

    # Additional moments
    print("\nSkewness:", float(ep['duration_ms'].skew()))
    print("Kurtosis:", float(ep['duration_ms'].kurtosis()))

    # Add duration in seconds for convenience
    ep['duration_s'] = ep['duration_ms'] / 1000.0

    # Per-AOI summaries
    print("\nPer-AOI summary (count, mean_ms, median_ms, std_ms, p90_ms):")
    aoi_stats = (
        ep
        .groupby('AOI', observed=True)
        .duration_ms
        .agg(count='count', mean_ms='mean', median_ms='median', std_ms='std')
        .assign(p90_ms=lambda df: ep.groupby('AOI', observed=True)['duration_ms'].quantile(0.9))
        .fillna(0)
        .sort_values('mean_ms', ascending=False)
    )
    print(aoi_stats)

    # Per-participant summary (show top N)
    pp = (
        ep
        .groupby('participant_id', observed=True)
        .duration_ms
        .agg(n_episodes='count', mean_ms='mean', median_ms='median', std_ms='std')
        .sort_values('n_episodes', ascending=False)
    )
    print("\nTop 10 participants by episode count:")
    print(pp.head(10))

    # Outliers: > mean + 3*std (global) and extremely long episodes (top 1%)
    mean_all = ep['duration_ms'].mean()
    std_all = ep['duration_ms'].std()
    outlier_thresh = mean_all + 2 * std_all ########
    top1pct = ep['duration_ms'].quantile(0.99)

    outliers_global = ep[ep['duration_ms'] > outlier_thresh].sort_values('duration_ms', ascending=False)
    outliers_top = ep[ep['duration_ms'] >= top1pct].sort_values('duration_ms', ascending=False)

    print(f"\nGlobal outlier threshold (mean + 3*std): {outlier_thresh:.1f} ms")
    print("Number of global outliers:", len(outliers_global))
    print("Number of episodes in top 1%:", len(outliers_top))

    # Show top longest episodes
    print("\nTop 20 longest episodes:")
    display_cols = ['participant_id', 'trial', 'AOI', 'start_time', 'end_time', 'duration_ms']
    print(outliers_top.head(20)[display_cols])

    # Short episodes (tiny durations)
    short_thresh = 80.0  # ms
    n_short = (ep['duration_ms'] < short_thresh).sum()
    print(f"\nEpisodes shorter than {short_thresh} ms: {n_short}")

    # Save summary tables to CSVs in plots folder if available
    if 'plots' in globals() and plots is not None:
        try:
            # aoi_stats.to_csv(plots / "episodes_aoi_stats.csv")
            # pp.to_csv(plots / "episodes_per_participant.csv")
            # outliers_top.to_csv(plots / "episodes_top1pct.csv", index=False)
            print(f"\nSaved aoi_stats, per-participant and outliers CSVs to: {plots}")
        except Exception as e:
            print("Failed to save CSVs to plots:", e)

    # Plot histogram + KDE of durations (ms) with log-x option
    try:
        import plotly.express as px
        # color map used elsewhere in the notebook
        color_map = {'left': '#5e81ac', 'right': '#a3be8c', 'neutral': dark[3], 'none': dark[3]}
        fig = px.histogram(
            ep,
            x='duration_ms',
            nbins=120,
            marginal='box',
            color='AOI' if 'AOI' in ep.columns else None,
            color_discrete_map=color_map,
            # title='Episode durations (ms) — histogram by AOI',
            labels={'duration_ms': 'duration (ms)'}
        )
        fig.update_layout(bargap=0.02, template='nord_light_paper', width=900, height=500)
        fig.update_xaxes(type='linear')
        fig.show()
        if 'plots' in globals() and plots is not None:
            fig.write_html(plots / "episodes_duration_histogram.html", include_plotlyjs='cdn')
            try:
                fig.write_image(plots / "episodes_duration_histogram.png", scale=3)
            except Exception:
                pass
    except Exception as e:
        print("Plotly histogram failed:", e)

    # Return computed objects to inspect further if run interactively
    summary = {
        'overall': ep['duration_ms'].describe(percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]).to_dict(),
        'skew': float(ep['duration_ms'].skew()),
        'kurtosis': float(ep['duration_ms'].kurtosis()),
        'aoi_stats': aoi_stats,
        'per_participant': pp,
        'outliers_top1pct': outliers_top,
        'global_outliers': outliers_global
    }

    summary

Episodes shape: (11829, 6)

Overall duration (ms) descriptive stats:
count    11829.000000
mean       355.903542
std        373.000647
min         60.000000
1%          60.000000
5%          66.000000
10%         73.000000
25%        110.000000
50%        212.000000
75%        475.000000
90%        834.000000
95%       1090.600000
99%       1714.760000
max       5598.000000
Name: duration_ms, dtype: float64

Skewness: 2.6613882527065256
Kurtosis: 12.934425827237193

Per-AOI summary (count, mean_ms, median_ms, std_ms, p90_ms):
       count     mean_ms  median_ms      std_ms  p90_ms
AOI                                                    
right   6074  375.763582      223.0  388.100499   878.7
left    5755  334.942659      200.0  355.201654   779.6

Top 10 participants by episode count:
                 n_episodes     mean_ms  median_ms      std_ms
participant_id                                                
participant_055         856  351.704439      209.5  456.776773
participant_023 

In [96]:
dur_col = 'duration_ms'
orig_n = len(episodes)
if orig_n == 0:
    print("episodes is empty -> nothing to remove.")
else:
    vals = episodes[dur_col].dropna().astype(float)
    mu = float(vals.mean())
    sigma = float(vals.std(ddof=0))  # population std (ddof=0) for consistency; change to ddof=1 if desired
    thresh = mu + 2.0 * sigma

    # identify outliers, but substitute their duration with the global mean instead of removing rows
    mask = episodes[dur_col] > thresh
    outliers = episodes[mask].copy()
    substituted_n = int(mask.sum())

    if substituted_n > 0:
        episodes.loc[mask, 'duration_ms'] = mu
        # recompute end_time = start_time + round(duration_ms) when start_time exists
        if 'start_time' in episodes.columns:
            episodes.loc[mask, 'end_time'] = (
                episodes.loc[mask, 'start_time'].astype(float)
                + episodes.loc[mask, 'duration_ms'].round().astype(int)
            ).astype(int)

    # keep full dataframe length but reset index for cleanliness
    episodes = episodes.reset_index(drop=True)

    print(f"Original rows: {orig_n}")
    print(f"Rows substituted (> mean + 2σ = {thresh:.1f} ms): {substituted_n} ({(substituted_n / orig_n * 100.0) if orig_n else 0:.2f}%)")
    print(f"Remaining rows: {len(episodes)}")

Original rows: 11829
Rows substituted (> mean + 2σ = 1101.9 ms): 578 (4.89%)
Remaining rows: 11829


In [97]:
# Eye tracking metrics
def summarize_eyetracking(fixations: pd.DataFrame) -> pd.DataFrame:
    required = {'participant_id', 'trial', 'AOI', 'start_time', 'end_time', 'duration_ms'}
    if fixations.empty:
        return pd.DataFrame(columns=[
            'participant_id','trial','dominant','prop_left','prop_right',
            'total_dwell','dwell_left','dwell_right','fixations_left','fixations_right',
            'total_fixation','revisits_left','revisits_right','total_revisits',
            'transitions','ffs','ffd','fvs','fvd',
        ])
    missing = required - set(fixations.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    fx = (fixations
          .dropna(subset=['participant_id','trial'])
          .copy())
    
    # Numeric coercions
    for c in ['duration_ms','start_time','end_time']:
        fx[c] = pd.to_numeric(fx[c], errors='coerce')
    fx = fx.dropna(subset=['start_time','end_time','duration_ms'])
    fx = fx.sort_values(['participant_id','trial','start_time'])

    metrics = []
    for (pid, tr), g in fx.groupby(['participant_id','trial'], sort=True):
        g = g.sort_values('start_time')
        # Dwell (milliseconds)
        dwell_left = g.loc[g['AOI'] == 'left', 'duration_ms'].sum()
        dwell_right = g.loc[g['AOI'] == 'right', 'duration_ms'].sum()
        total_dwell = dwell_left + dwell_right

        if total_dwell > 0:
            prop_left = dwell_left / total_dwell
            prop_right = dwell_right / total_dwell
            dominant = 'left' if dwell_left >= dwell_right else 'right'
        else:
            prop_left = np.nan
            prop_right = np.nan
            dominant = None

        aoi_series = g['AOI'].astype(str)
        durations_ms = g['duration_ms'].to_numpy(dtype=float)
        transitions = int(max((aoi_series != aoi_series.shift()).sum() - 1, 0))
        fix_left = int((aoi_series == 'left').sum())
        fix_right = int((aoi_series == 'right').sum())

        # First visit (consecutive fixations on the same side) — duration in ms
        first_visit_side = None
        first_visit_duration = np.nan
        current_side = None
        current_duration_ms = 0.0
        for aoi, dur in zip(aoi_series, durations_ms):
            if aoi not in ('left', 'right'):
                if current_side is None:
                    continue
                break
            if current_side is None:
                current_side = aoi
                current_duration_ms = dur
            elif aoi == current_side:
                current_duration_ms += dur
            else:
                break
        if current_side is not None:
            first_visit_side = current_side
            first_visit_duration = round(current_duration_ms, 1)  # ms

        # First fixation duration (milliseconds) & location
        if len(g):
            first_aoi = aoi_series.iloc[0]
            first_dur_ms = g.iloc[0]['duration_ms']
            ffd = first_dur_ms if not np.isnan(first_dur_ms) else np.nan
            ffs = first_aoi
        else:
            ffd = np.nan
            ffs = None

        revisits_left = max(0, fix_left - 1)
        revisits_right = max(0, fix_right - 1)

        metrics.append({
            'participant_id': pid,
            'trial': tr,
            'dominant': dominant,
            'prop_left': round(prop_left, 3) if not np.isnan(prop_left) else np.nan,
            'prop_right': round(prop_right, 3) if not np.isnan(prop_right) else np.nan,
            'total_dwell': round(total_dwell, 1),
            'dwell_left': round(dwell_left, 1),
            'dwell_right': round(dwell_right, 1),
            'fixations_left': fix_left,
            'fixations_right': fix_right,
            'total_fixation': fix_left + fix_right,
            'revisits_left': revisits_left,
            'revisits_right': revisits_right,
            'total_revisits': revisits_left + revisits_right,
            'transitions': transitions,
            'ffs': ffs,
            'ffd': round(ffd, 1) if not np.isnan(ffd) else np.nan,
            'fvs': first_visit_side,
            'fvd': first_visit_duration,
        })

    return pd.DataFrame(metrics)

# Build simplified ET summary
et_df = summarize_eyetracking(episodes)
et_df = et_df[~et_df['participant_id'].isin(list_of_participants_to_remove)]
et_df

,participant_id,trial,dominant,prop_left,prop_right,total_dwell,dwell_left,dwell_right,fixations_left,fixations_right,total_fixation,revisits_left,revisits_right,total_revisits,transitions,ffs,ffd,fvs,fvd
0,participant_001,6,left,0.758,0.242,2250.0,1706.0,544.0,4,4,8,3,3,6,5,right,76.0,right,377.0
1,participant_001,7,right,0.341,0.659,2223.0,757.0,1466.0,3,6,9,2,5,7,1,left,97.0,left,757.0
2,participant_001,8,left,0.837,0.163,1268.0,1061.0,207.0,3,2,5,2,1,3,3,right,73.0,right,73.0
3,participant_001,10,right,0.249,0.751,1270.9,316.0,954.9,2,4,6,1,3,4,4,right,199.0,right,199.0
4,participant_001,11,left,0.567,0.433,1396.0,791.0,605.0,5,6,11,4,5,9,3,right,86.0,right,153.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2987,participant_072,46,left,1.000,0.000,961.0,961.0,0.0,2,0,2,1,0,1,0,left,82.0,left,961.0
2988,participant_072,47,right,0.000,1.000,954.0,0.0,954.0,0,1,1,0,0,0,0,right,954.0,right,954.0
2989,participant_072,48,right,0.000,1.000,596.0,0.0,596.0,0,1,1,0,0,0,0,right,596.0,right,596.0
2990,participant_072,49,left,1.000,0.000,950.0,950.0,0.0,1,0,1,0,0,0,0,left,950.0,left,950.0


## Merge ET and behavioral data

In [98]:
# # Overwrite 'trial' values in et_df per participant using the order from region_df/trial_index
# for participant in et_df['participant_id'].unique():
#     # Get the ordered list of trial numbers for this participant from region_df
#     # region_df index: (results_time, participant_id, group, trial, label, no, item, condition, cb, left, right)
#     # Extract trial numbers for this participant
#     idx = region_df.index
#     participant_trials = [i[3] for i in idx if i[1] == participant]
    
#     # Get indices in et_df for this participant
#     mask = et_df['participant_id'] == participant
#     n_trials = mask.sum()
    
#     # Only overwrite if counts match
#     if len(participant_trials) == n_trials:
#         et_df.loc[mask, 'trial'] = participant_trials
#     else:
#         print(f"Warning: trial count mismatch for {participant} (region_df: {len(participant_trials)}, et_df: {n_trials})")

In [99]:
# # Canonical trial order from behavioral data
# trial_reference = (
#     trial_index
#     .reset_index(drop=True)
#     .sort_values(['participant_id', 'no', 'item'])
#     [['participant_id', 'trial']]
# )
# trial_reference['trial_rank'] = trial_reference.groupby('participant_id').cumcount()

# # Actual ET order from episodes (min start_time per trial)
# et_trial_order = (
#     episodes
#     .groupby(['participant_id', 'trial'], as_index=False)['start_time']
#     .min()
#     .rename(columns={'start_time': 'et_t0'})
#     .sort_values(['participant_id', 'et_t0'])
# )
# et_trial_order['trial_rank'] = et_trial_order.groupby('participant_id').cumcount()

# # Map ET trial ids to canonical ids
# trial_map = (
#     et_trial_order
#     .merge(trial_reference, on=['participant_id', 'trial_rank'], how='inner', suffixes=('_et', '_canon'))
#     [['participant_id', 'trial_et', 'trial_canon']]
# )

# et_df = (
#     et_df
#     .merge(trial_map, left_on=['participant_id', 'trial'], right_on=['participant_id', 'trial_et'], how='left')
#     .assign(trial=lambda d: d['trial_canon'].fillna(d['trial']))
#     .drop(columns=['trial_et', 'trial_canon'])
# )

In [100]:
# pid = 'participant_001'

# # ensure relevant dataframes exist
# if 'events_df' not in globals() or 'et_df' not in globals():
#     raise RuntimeError("events_df and/or et_df not found in the notebook environment.")

# # get trial sets (coerce to str/int consistently)
# lf_trials = pd.Series(events_df.loc[events_df['participant_id'] == pid, 'trial'].dropna().unique()).tolist()
# et_trials = pd.Series(et_df.loc[et_df['participant_id'] == pid, 'trial'].dropna().unique()).tolist()

# lf_set = set(lf_trials)
# et_set = set(et_trials)

# only_in_events_df = sorted(lf_set - et_set)
# only_in_et = sorted(et_set - lf_set)
# in_both = sorted(lf_set & et_set)

# print(f"Participant: {pid}")
# print(f"events_df trials: {sorted(lf_trials)} (n={len(lf_trials)})")
# print(f"ET summary trials: {sorted(et_trials)} (n={len(et_trials)})")
# print()
# print("Trials only in events_df (not in et_df):", only_in_events_df)
# print("Trials only in et_df (not in events_df):", only_in_et)
# print("Trials in both:", in_both)

# # show rows for the discrepant trials for inspection
# if only_in_events_df:
#     print("\nRows from events_df for trials only in events_df:")
#     display(events_df[(events_df['participant_id'] == pid) & (events_df['trial'].isin(only_in_events_df))].sort_values('trial'))

# if only_in_et:
#     print("\nRows from et_df for trials only in et_df:")
#     display(et_df[(et_df['participant_id'] == pid) & (et_df['trial'].isin(only_in_et))].sort_values('trial'))

# # quick sanity: show example rows for trials present in both
# if in_both:
#     print("\nExample merged rows for trials present in both (head):")
#     display(
#         pd.merge(
#             events_df[events_df['participant_id'] == pid],
#             et_df[et_df['participant_id'] == pid],
#             on=['participant_id','trial'],
#             how='inner'
#         ).sort_values('trial').head(10)
#     )

In [101]:
# Merge eyetracking data with events_df data on participant_id & trial
et_df = pd.merge(
    events_df,
    et_df,
    how='left',
    left_on=['participant_id', 'trial'],
    right_on=['participant_id', 'trial']
)

# Ensure any timezone-aware datetimes are made timezone-naive before writing to Excel
from pandas import DatetimeTZDtype
import datetime

def _series_is_tz_aware(s, sample_n=20):
    """Return True if Series `s` appears to be timezone-aware.
    First checks dtype (preferred), then falls back to inspecting a small sample of Python datetimes.
    """
    if isinstance(s.dtype, DatetimeTZDtype):
        return True
    # fallback: examine a small sample for tzinfo on python datetimes
    if s.dtype == object:
        sample = s.dropna().head(sample_n)
        for v in sample:
            if isinstance(v, datetime.datetime) and v.tzinfo is not None:
                return True
    return False

for c in et_df.columns:
    try:
        if _series_is_tz_aware(et_df[c]):
            # If dtype is pandas tz-aware datetime, convert to UTC then drop tz
            if isinstance(et_df[c].dtype, DatetimeTZDtype):
                et_df[c] = et_df[c].dt.tz_convert('UTC').dt.tz_localize(None)
            else:
                # fallback: coerce via to_datetime(utc=True) then drop tz
                et_df[c] = pd.to_datetime(et_df[c], utc=True).dt.tz_localize(None)
    except Exception:
        # If conversion fails for any column, leave it as-is but warn for debugging
        print(f"Warning: failed to convert timezone-aware column '{c}' — leaving as-is")

# Sort et_df by participant_id, then no
et_df = et_df.sort_values(by=['participant_id', 'no']).reset_index(drop=True)
et_df

,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,fixations_right,total_fixation,revisits_left,revisits_right,total_revisits,transitions,ffs,ffd,fvs,fvd
0,participant_001,b,34,experiment,2,1,1,unmodified,y,1b,...,5.0,5.0,0.0,4.0,4.0,0.0,right,154.0,right,1228.0
1,participant_001,b,26,experiment,4,2,1,exhaustive,y,2b,...,1.0,1.0,0.0,0.0,0.0,0.0,right,355.9,right,355.9
2,participant_001,b,49,experiment,9,3,1,contrastive,n,3a,...,4.0,10.0,5.0,3.0,8.0,7.0,left,64.0,left,131.0
3,participant_001,b,25,experiment,11,4,1,unmodified,n,4a,...,2.0,5.0,2.0,1.0,3.0,1.0,right,63.0,right,129.0
4,participant_001,b,13,experiment,13,5,1,exhaustive,n,5a,...,4.0,8.0,3.0,3.0,6.0,4.0,left,273.0,left,1037.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1291,participant_072,a,31,experiment,42,14,1,contrastive,y,14b,...,2.0,3.0,0.0,1.0,1.0,1.0,right,718.0,right,783.0
1292,participant_072,a,29,experiment,44,15,1,unmodified,y,15b,...,2.0,2.0,0.0,1.0,1.0,0.0,right,92.0,right,474.0
1293,participant_072,a,40,experiment,46,16,1,exhaustive,y,16b,...,1.0,2.0,0.0,0.0,0.0,1.0,left,86.0,left,86.0
1294,participant_072,a,34,experiment,51,17,1,contrastive,n,17a,...,4.0,7.0,2.0,3.0,5.0,4.0,right,86.0,right,171.0


In [102]:
# Calculate percentage of rows where total_dwell is missing
if 'et_df' not in globals():
    raise RuntimeError("et_df not found in the notebook environment.")

if 'total_dwell' not in et_df.columns:
    print("Column 'total_dwell' not present in et_df.")
else:
    total = len(et_df)
    missing = et_df['total_dwell'].isna().sum()
    pct = (missing / total * 100) if total else float('nan')
    print(f"Missing total_dwell: {missing} / {total} rows ({pct:.2f}%)")

Missing total_dwell: 65 / 1296 rows (5.02%)


In [103]:
# Fill missing values in et_df with column means
import numpy as np

_fill_cols = ['total_dwell', 'ffd', 'fvd', 'total_fixation', 'transitions']
for _col in _fill_cols:
    if _col in et_df.columns:
        missing_mask = et_df[_col].isna()
        n_missing = int(missing_mask.sum())
        if n_missing:
            # compute mean from existing (non-missing) values
            mean_val = float(et_df.loc[~missing_mask, _col].mean()) if (~missing_mask).any() else float(np.nan)
            et_df.loc[missing_mask, _col] = mean_val
            print(f"Filled {n_missing} missing in '{_col}' with mean {mean_val:.3f}")
        else:
            print(f"No missing values in '{_col}'")
    else:
        print(f"Column not present: '{_col}'")

# Quick check
print(et_df[_fill_cols].isna().sum())
et_df

Filled 65 missing in 'total_dwell' with mean 1241.021
Filled 65 missing in 'ffd' with mean 297.478
Filled 65 missing in 'fvd' with mean 460.163
Filled 65 missing in 'total_fixation' with mean 4.197
Filled 65 missing in 'transitions' with mean 1.719
total_dwell       0
ffd               0
fvd               0
total_fixation    0
transitions       0
dtype: int64


,participant_id,group,trial,label,no,item,exp,condition,cb,left,...,fixations_right,total_fixation,revisits_left,revisits_right,total_revisits,transitions,ffs,ffd,fvs,fvd
0,participant_001,b,34,experiment,2,1,1,unmodified,y,1b,...,5.0,5.0,0.0,4.0,4.0,0.0,right,154.0,right,1228.0
1,participant_001,b,26,experiment,4,2,1,exhaustive,y,2b,...,1.0,1.0,0.0,0.0,0.0,0.0,right,355.9,right,355.9
2,participant_001,b,49,experiment,9,3,1,contrastive,n,3a,...,4.0,10.0,5.0,3.0,8.0,7.0,left,64.0,left,131.0
3,participant_001,b,25,experiment,11,4,1,unmodified,n,4a,...,2.0,5.0,2.0,1.0,3.0,1.0,right,63.0,right,129.0
4,participant_001,b,13,experiment,13,5,1,exhaustive,n,5a,...,4.0,8.0,3.0,3.0,6.0,4.0,left,273.0,left,1037.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1291,participant_072,a,31,experiment,42,14,1,contrastive,y,14b,...,2.0,3.0,0.0,1.0,1.0,1.0,right,718.0,right,783.0
1292,participant_072,a,29,experiment,44,15,1,unmodified,y,15b,...,2.0,2.0,0.0,1.0,1.0,0.0,right,92.0,right,474.0
1293,participant_072,a,40,experiment,46,16,1,exhaustive,y,16b,...,1.0,2.0,0.0,0.0,0.0,1.0,left,86.0,left,86.0
1294,participant_072,a,34,experiment,51,17,1,contrastive,n,17a,...,4.0,7.0,2.0,3.0,5.0,4.0,right,86.0,right,171.0


### Summary Statistics fro ET Metrics

In [104]:
import pandas as pd
import numpy as np
import plotly.express as px

# Resolve fixation column name robustly
fix_col = "total_fixations" if "total_fixations" in et_df.columns else "total_fixation"

metrics = ["total_dwell", fix_col]
missing = [c for c in metrics if c not in et_df.columns]
if missing:
    raise ValueError(f"Missing required columns in et_df: {missing}")

# Keep only analysis columns
plot_df = et_df[["condition"] + metrics].copy()
for c in metrics:
    plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")

# Long format
long_df = plot_df.melt(
    id_vars="condition",
    value_vars=metrics,
    var_name="metric",
    value_name="value"
).dropna(subset=["value"])

# Summary statistics
summary = (
    long_df.groupby(["condition", "metric"], observed=True)["value"]
    .agg(
        n="count",
        mean="mean",
        sd="std",
        median="median",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
    )
    .reset_index()
)
display(summary.sort_values(["metric", "condition"]))

# condition order
if pd.api.types.is_categorical_dtype(et_df["condition"]):
    cond_order = list(et_df["condition"].cat.categories)
else:
    cond_order = sorted(summary["condition"].dropna().astype(str).unique())

def make_metric_plot(metric_key: str, title: str, y_label: str):
    d = summary[summary["metric"] == metric_key].copy()
    if d.empty:
        print(f"No data for {metric_key}")
        return None

    fig = px.bar(
        d,
        x="condition",
        y="mean",
        error_y="sd",
        category_orders={"condition": cond_order},
        title=title,
        labels={"mean": y_label, "condition": "Condition"},
        template="nord_light_paper"
    )
    fig.add_scatter(
        x=d["condition"],
        y=d["median"],
        mode="markers",
        marker=dict(color="black", size=8, symbol="diamond"),
        name="Median",
        showlegend=False
    )
    fig.update_layout(height=420, width=750)
    return fig

# Plot 1: total_dwell
fig_dwell = make_metric_plot(
    "total_dwell",
    "Total dwell by condition (mean ± SD)",
    "Total dwell (ms)"
)
if fig_dwell is not None:
    fig_dwell.show()

# Plot 2: total_fixation(s)
fig_fix = make_metric_plot(
    fix_col,
    "Total fixations by condition (mean ± SD)",
    "Total fixations"
)
if fig_fix is not None:
    fig_fix.show()

,condition,metric,n,mean,sd,median,q1,q3
0,exhaustive,total_dwell,432,1063.865072,986.187123,847.0,603.5,1206.75000
2,unmodified,total_dwell,432,1078.946274,865.848237,889.5,589.0,1247.75000
4,contrastive,total_dwell,432,1580.250311,5279.579513,922.5,651.5,1291.25000
1,exhaustive,total_fixation,432,3.622565,3.914468,3.0,2.0,4.04935
3,unmodified,total_fixation,432,3.564238,3.138973,3.0,2.0,4.19740
5,contrastive,total_fixation,432,5.405399,20.208527,3.0,2.0,5.00000


C:\Users\parti\AppData\Local\Temp\ipykernel_10536\3899533259.py:42: DeprecationWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead



In [105]:
# Prepare data for export to be used in R
et_data = et_df.copy()

# Create a new column variable called 'combined' that combines 'condition' and 'chosen_type' and add to the et_data dataframe
et_data['cxt'] = et_data['condition'].astype(str) + '.' + et_data['chosen_type'].astype(str)

# Rename chosen_type to choice
et_data = et_data.rename(columns={'participant_id': 'participant'})
# Save et_df as et_data.xlsx and et_data.csv
et_data.to_excel("et_data.xlsx", index=False)
et_data.to_csv("et_data.csv", index=False)

et_data

,participant,group,trial,label,no,item,exp,condition,cb,left,...,total_fixation,revisits_left,revisits_right,total_revisits,transitions,ffs,ffd,fvs,fvd,cxt
0,participant_001,b,34,experiment,2,1,1,unmodified,y,1b,...,5.0,0.0,4.0,4.0,0.0,right,154.0,right,1228.0,unmodified.A
1,participant_001,b,26,experiment,4,2,1,exhaustive,y,2b,...,1.0,0.0,0.0,0.0,0.0,right,355.9,right,355.9,exhaustive.A
2,participant_001,b,49,experiment,9,3,1,contrastive,n,3a,...,10.0,5.0,3.0,8.0,7.0,left,64.0,left,131.0,contrastive.A
3,participant_001,b,25,experiment,11,4,1,unmodified,n,4a,...,5.0,2.0,1.0,3.0,1.0,right,63.0,right,129.0,unmodified.A
4,participant_001,b,13,experiment,13,5,1,exhaustive,n,5a,...,8.0,3.0,3.0,6.0,4.0,left,273.0,left,1037.0,exhaustive.A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1291,participant_072,a,31,experiment,42,14,1,contrastive,y,14b,...,3.0,0.0,1.0,1.0,1.0,right,718.0,right,783.0,contrastive.B
1292,participant_072,a,29,experiment,44,15,1,unmodified,y,15b,...,2.0,0.0,1.0,1.0,0.0,right,92.0,right,474.0,unmodified.A
1293,participant_072,a,40,experiment,46,16,1,exhaustive,y,16b,...,2.0,0.0,0.0,0.0,1.0,left,86.0,left,86.0,exhaustive.A
1294,participant_072,a,34,experiment,51,17,1,contrastive,n,17a,...,7.0,2.0,3.0,5.0,4.0,right,86.0,right,171.0,contrastive.B


## Plot ET results

### Exploratory plots (Log)

In [106]:
# Plot log(ET) by condition and chosen_type (A / B) for selected ET metrics.
metrics = ['total_dwell', 'total_fixation']

# if 'chosen_type' not in et_data.columns:
#     raise RuntimeError("et_data does not contain 'chosen_type' column. Ensure merge included it.")

for metric in metrics:
    # if metric not in et_data.columns:
    #     print(f"Skipping missing metric: {metric}")
    #     continue

    df = et_data[['condition', 'chosen_type', metric]].copy()
    df = df[df['chosen_type'].isin(['A', 'B'])].copy()

    df[metric] = pd.to_numeric(df[metric], errors='coerce')
    df = df.dropna(subset=[metric, 'condition', 'chosen_type'])
    # if df.empty:
    #     print(f"No data for metric {metric} after filtering.")
    #     continue

    df[f'log_{metric}'] = np.log1p(df[metric].astype(float))

    if isinstance(df['condition'].dtype, pd.CategoricalDtype):
        cond_order = list(df['condition'].cat.categories)
    else:
        cond_order = sorted(df['condition'].dropna().unique())

    title = f"Log({metric}) by condition, split by choice (A or B)"
    fig = px.box(
        df,
        x='condition',
        y=f'log_{metric}',
        color='chosen_type',
        color_discrete_map={'A': '#5e81ac', 'B': '#ebcb8b', 'None': dark[2]},
        points='all',
        template='nord_light_paper',
        category_orders={'condition': cond_order},
        # title=title,
        labels={f'log_{metric}': f'log({metric}) (log1p)', 'condition': 'Condition', 'chosen_type': 'Choice'}
    )

    counts = (
        df.groupby(['condition', 'chosen_type'], observed=True)
          .size()
          .reset_index(name='n')
    )
    y_offsets = {'A': 0.90, 'B': 0.86}
    for _, row in counts.iterrows():
        fig.add_annotation(
            x=row['condition'],
            y=y_offsets.get(row['chosen_type'], 1.0),
            xref='x',
            yref='paper',
            text=f"{row['chosen_type']}: n={int(row['n'])}",
            showarrow=False,
            font=dict(size=14, color='#4c566a')
        )

    fig.update_layout(base_layout, legend_title_text='Choice', legend=dict(orientation='v', y=0.5, yanchor='top', xanchor='center', x=1))
    fig.update_traces(marker=dict(opacity=0.75, size=5), boxmean=True)
    fig.update_xaxes(title_text=None)

    # Resize image to 1200x600 for better readability
    fig.update_layout(width=800, height=400)

    # Remove padding around the plot for better fit
    fig.update_layout(margin=dict(l=40, r=40, t=0, b=20))
    
    fig.show()

    out_html = plots / f'log_{metric}_by_condition_and_choice.html'
    out_png = plots / f'log_{metric}_by_condition_and_choice.png'
    fig.write_html(out_html, include_plotlyjs='cdn')
    try:
        fig.write_image(out_png, scale=3)
    except Exception:
        pass

### Classified samples

#### Transition curves

In [107]:
ORDERED_CONDS = ['exhaustive', 'unmodified', 'contrastive']
COND_COLORS = {
    'exhaustive': '#5e81ac',   # blue
    'unmodified': '#a3be8c',   # green
    'contrastive':'#ebcb8b',  # yellow
}

In [108]:
def build_transition_curves(
    fix_df: pd.DataFrame,
    trials_df: pd.DataFrame,
    step_ms: int = 50,
    max_time_ms: int | None = None,
    drop_na_condition: bool = True,
    pad_to_global: bool = False,      # NEW
    normalize_time: bool = False      # NEW
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Map condition onto fixations (unique per participant×trial)
    cond_map = (trials_df[['participant_id','trial','condition']]
                .dropna(subset=['participant_id','trial'])
                .drop_duplicates())
    fx = fix_df.copy()
    fx['trial'] = pd.to_numeric(fx['trial'], errors='coerce')
    cond_map['trial'] = pd.to_numeric(cond_map['trial'], errors='coerce')

    fx = fx.merge(cond_map, on=['participant_id','trial'], how='left')
    if drop_na_condition:
        fx = fx[fx['condition'].notna()]

    # Keep only needed cols and sanitize
    fx = fx.dropna(subset=['participant_id','trial','AOI','start_time','end_time'])
    fx['start_time'] = pd.to_numeric(fx['start_time'], errors='coerce')
    fx['end_time']   = pd.to_numeric(fx['end_time'], errors='coerce')
    fx = fx.dropna(subset=['start_time','end_time'])
    fx = fx.sort_values(['participant_id','trial','start_time'])

    # Per-trial start/end (relative)
    per_trial = {}
    for (pid, tr), g in fx.groupby(['participant_id','trial'], sort=False):
        t0 = float(g['start_time'].min())
        tend = float(g['end_time'].max()) - t0
        if max_time_ms is not None:
            tend = min(tend, float(max_time_ms))
        per_trial[(pid, tr)] = dict(t0=t0, tend=tend)

    if not per_trial:
        return (pd.DataFrame(columns=['participant_id','trial','condition','time_ms','transitions']),
                pd.DataFrame(columns=['condition','time_ms','mean_transitions','n_trials']))

    global_max = int(max(v['tend'] for v in per_trial.values()))
    rows = []

    for (pid, tr), g in fx.groupby(['participant_id','trial'], sort=False):
        meta = per_trial.get((pid, tr))
        if not meta:
            continue
        t0, tend = meta['t0'], meta['tend']

        if normalize_time:
            if tend <= 0:
                continue
            # Use step_ms as approx bin width in ms; convert to fraction-of-trial step
            frac_step = max(step_ms / max(tend, 1.0), 1.0 / max(int(tend // step_ms), 1))
            frac_grid = np.arange(0.0, 1.0 + 1e-9, frac_step)
            time_grid = (frac_grid * tend).astype(float)
            tcol = 'time_norm'
            tx = frac_grid
        else:
            if pad_to_global:
                time_grid = np.arange(0, int(global_max) + step_ms, step_ms, dtype=int)
            else:
                time_grid = np.arange(0, int(tend) + step_ms, step_ms, dtype=int)
            tcol = 'time_ms'
            tx = time_grid

        # AOI change times (absolute -> relative)
        g = g.sort_values('start_time')
        aoi = g['AOI'].astype(str).to_numpy()
        change = (aoi[1:] != aoi[:-1])
        starts = g['start_time'].to_numpy(dtype=float)
        trans_times_rel = np.maximum(0.0, starts[1:][change] - t0)

        counts = np.searchsorted(np.sort(trans_times_rel), time_grid, side='right') if trans_times_rel.size else np.zeros_like(time_grid, int)

        trial_rows = pd.DataFrame({
            'participant_id': pid,
            'trial': tr,
            'condition': g['condition'].iloc[0] if 'condition' in g.columns else None,
            tcol: tx,
            'transitions': counts,
            'tend_ms': tend
        })
        rows.append(trial_rows)

    trial_curves = pd.concat(rows, ignore_index=True)

    # Mean over available trials at each time bin (no padding -> fewer trials at later bins)
    agg = (trial_curves
           .groupby(['condition', tcol], observed=True)
           .agg(mean_transitions=('transitions', 'mean'),
                n_trials=('transitions', 'size'))
           .reset_index())

    return trial_curves, agg

# Rebuild: clip to each trial’s length (no padding)
trial_curves, mean_curves = build_transition_curves(
    episodes,
    et_df[['participant_id','trial','condition']],
    step_ms=50,
    pad_to_global=False,
    normalize_time=False
)

def plot_transitions_by_condition(
    trial_curves: pd.DataFrame,
    mean_curves: pd.DataFrame,
    show_individual: bool = True,
    template: str = 'nord_light_paper',
    enable_trial_toggle: bool = True,
    show_live_histogram: bool = True,            # NEW
    live_hist_mode: str = 'total'                # 'total' or 'stacked_by_condition'
):
    fig = go.Figure()
    palette = nord if 'nord' in globals() else px.colors.qualitative.Set2
    tcol = 'time_norm' if ('time_norm' in mean_curves.columns) else 'time_ms'

    unique_conds = mean_curves['condition'].dropna().astype(str).unique().tolist()
    conds = [c for c in ORDERED_CONDS if c in unique_conds] + [c for c in unique_conds if c not in ORDERED_CONDS]
    color_map = {c: COND_COLORS.get(c, palette[i % len(palette)]) for i, c in enumerate(conds)}

    # --- Live-trials histogram (bottom panel) ---
    if show_live_histogram:
        if live_hist_mode == 'stacked_by_condition' and len(conds) > 1:
            # Per-condition counts (use mean_curves.n_trials which already encodes “trials present at bin”)
            for cond in conds:
                h = (
                    mean_curves[mean_curves['condition'].astype(str) == cond]
                    .groupby(tcol, observed=True)['n_trials']
                    .sum()
                    .reset_index()
                    .sort_values(tcol)
                )
                fig.add_trace(go.Bar(
                    x=h[tcol], y=h['n_trials'],
                    name=f'{cond} live',
                    marker_color=color_map.get(cond, '#999'),
                    opacity=0.45,
                    yaxis='y2',
                    hovertemplate=f'{cond}: %{y} trials<extra></extra>'
                ))
            hist_barmode = 'stack'
        else:
            # Total counts across all conditions
            h = (
                mean_curves.groupby(tcol, observed=True)['n_trials']
                .sum()
                .reset_index()
                .sort_values(tcol)
            )
            fig.add_trace(go.Bar(
                x=h[tcol], y=h['n_trials'],
                name='trials live',
                marker_color="#d8dee9",
                opacity=0.5,
                yaxis='y2',
                hovertemplate='Live trials: %{y}<extra></extra>'
            ))
            hist_barmode = 'overlay'
    # --- End histogram ---

    if show_individual:
        for cond in conds:
            g = trial_curves[trial_curves['condition'].astype(str) == cond]
            if g.empty:
                continue
            color = color_map.get(cond, '#888')
            group_id = f'{cond}::trials'
            for (pid, tr), gt in g.groupby(['participant_id','trial'], sort=False):
                fig.add_trace(go.Scatter(
                    x=gt[tcol], y=gt['transitions'],
                    mode='lines',
                    line=dict(width=1, color=color),
                    opacity=0.25,
                    name=f'{cond} trial',
                    showlegend=False,
                    hoverinfo='skip',
                    legendgroup=group_id
                ))
            if enable_trial_toggle:
                fig.add_trace(go.Scatter(
                    x=[None], y=[None],
                    mode='lines',
                    line=dict(width=1, color=color, dash='dot'),
                    name=f'{cond} trials',
                    legendgroup=group_id,
                    showlegend=True,
                    visible='legendonly'
                ))

    for cond in conds:
        g = mean_curves[mean_curves['condition'].astype(str) == cond].sort_values(tcol)
        if g.empty:
            continue
        color = color_map.get(cond, '#aaa')
        fig.add_trace(go.Scatter(
            x=g[tcol],
            y=g['mean_transitions'],
            mode='lines',
            line=dict(width=3, color=color),
            name=f'{cond} (mean)',
            customdata=g['n_trials'],
            hovertemplate='t=%{x}<br>mean transitions=%{y:.2f}<br>n(trials)=%{customdata}<extra></extra>'
        ))

    # Layout: split vertical space between main plot (top) and histogram (bottom)
    layout_updates = dict(
        template=template,
        title='Cumulative gaze transitions over time by condition',
        xaxis_title='Time (% of trial)' if tcol=='time_norm' else 'Time (ms)',
        hovermode='x unified',
        legend=dict(groupclick='togglegroup'),
        height=600,
        width=1200
    )
    if show_live_histogram:
        layout_updates.update(dict(
            yaxis=dict(domain=[0.30, 1.0], title='Number of transitions'),
            yaxis2=dict(domain=[0.00, 0.22], title='Live trials', anchor='x', showgrid=False),
            barmode=hist_barmode,
            bargap=0.0
        ))
    else:
        layout_updates.update(dict(
            yaxis_title='Number of transitions'
        ))

    fig.update_layout(**layout_updates)
    return fig

fig = plot_transitions_by_condition(
    trial_curves, mean_curves,
    show_individual=True, template='nord_light_paper', enable_trial_toggle=True,
    show_live_histogram=True, live_hist_mode='total'  # or 'stacked_by_condition'
)

fig.show()

# Save
fig.write_html(plots / "transitions_over_time_by_condition.html", include_plotlyjs='cdn')
try:
    fig.write_image(plots / "transitions_over_time_by_condition.png", scale=3)
except Exception:
    pass

#### All trials

In [109]:
print(et_df.head(10))

    participant_id group  trial       label  no  item exp    condition cb  \
0  participant_001     b     34  experiment   2     1   1   unmodified  y   
1  participant_001     b     26  experiment   4     2   1   exhaustive  y   
2  participant_001     b     49  experiment   9     3   1  contrastive  n   
3  participant_001     b     25  experiment  11     4   1   unmodified  n   
4  participant_001     b     13  experiment  13     5   1   exhaustive  n   
5  participant_001     b     15  experiment  18     6   1  contrastive  y   
6  participant_001     b     37  experiment  20     7   1   unmodified  y   
7  participant_001     b     17  experiment  22     8   1   exhaustive  y   
8  participant_001     b     12  experiment  27     9   1  contrastive  n   
9  participant_001     b     21  experiment  29    10   1   unmodified  n   

  left  ... fixations_right total_fixation  revisits_left  revisits_right  \
0   1b  ...             5.0            5.0            0.0             4.0  

In [110]:
# from plotly.subplots import make_subplots
# import plotly.graph_objects as go
# import pandas as pd
# import os

# transparent_color = 'hsva(0, 0%, 100%, 0)'

# # Facet by condition into rows (one condition per row)
# conditions = list(events_df['condition'].dropna().unique())
# n_rows = max(1, len(conditions))

# fig = make_subplots(
#     rows=n_rows, cols=1,
#     shared_xaxes=True,
#     subplot_titles=conditions,
#     vertical_spacing=0.02
# )

# # Track per-condition ordering and metadata for categorical y-axes
# facet_yticks = {cond: [] for cond in conditions}
# facet_tick_meta = {cond: {} for cond in conditions}

# fixations_for_plot = (
#     episodes
#     .copy()
#     .loc[~episodes['participant_id'].isin(list_of_participants_to_remove)]
# )
# fixations_for_plot['AOI'] = fixations_for_plot['AOI'].astype(str).str.lower()
# fixations_for_plot['start_time'] = pd.to_numeric(fixations_for_plot['start_time'], errors='coerce')
# fixations_for_plot['end_time'] = pd.to_numeric(fixations_for_plot['end_time'], errors='coerce')
# fixations_for_plot['duration_ms'] = pd.to_numeric(fixations_for_plot['duration_ms'], errors='coerce')
# fixations_for_plot = fixations_for_plot.dropna(subset=['participant_id', 'trial', 'start_time', 'end_time'])
# fixations_for_plot['duration_ms'] = fixations_for_plot['duration_ms'].fillna(
#     fixations_for_plot['end_time'] - fixations_for_plot['start_time']
# )
# fixations_for_plot = fixations_for_plot[fixations_for_plot['duration_ms'] > 0]
# fixations_for_plot['start_s'] = fixations_for_plot['start_time'] / 1000.0
# fixations_for_plot['duration_s'] = fixations_for_plot['duration_ms'] / 1000.0
# fixations_for_plot['trial_key'] = fixations_for_plot['trial'].astype(str)

# color_map = {
#     'left': '#5e81ac',
#     'right': '#a3be8c',
#     'none': dark[2],
#     'neutral': dark[3]
# }

# side_label_map = {
#     'left': 'Left',
#     'right': 'Right',
#     'none': 'None',
#     'neutral': 'None'
# }

# for participant in [p for p in et_df['participant_id'].unique() if p not in list_of_participants_to_remove]:
#     df_part = et_df[et_df['participant_id'] == participant]
#     sorted_trials = sorted(
#         df_part['trial'].unique(),
#         key=lambda x: float(x) if str(x).replace('.', '', 1).isdigit() else x
#     )
#     for trial in sorted_trials:
#         lf_row = events_df[(events_df['participant_id'] == participant) & (events_df['trial'] == trial)]
#         if lf_row.empty:
#             continue
#         cond = lf_row['condition'].iloc[0]
#         if pd.isna(cond) or cond not in conditions:
#             continue

#         if 'item' in lf_row.columns and pd.notna(lf_row['item'].iloc[0]):
#             item_label = lf_row['item'].iloc[0]
#         else:
#             item_label = trial
#         label = f"{participant} - Item {item_label}"
#         if label not in facet_yticks[cond]:
#             facet_yticks[cond].append(label)

#         et_row = df_part[df_part['trial'] == trial]
#         dominant = et_row['dominant'].iloc[0] if not et_row.empty else None
#         choice = lf_row['choice'].iloc[0] if not lf_row.empty else None

#         mismatch_flag = False
#         if 'mismatch' in lf_row.columns:
#             try:
#                 mismatch_flag = bool(lf_row['mismatch'].iloc[0])
#             except Exception:
#                 mismatch_flag = False

#         facet_tick_meta[cond][label] = {
#             'pid_short': str(participant)[-3:],
#             'choice': choice if pd.notna(choice) else 'NA',
#             'dominant_matches': (dominant is not None and choice is not None and dominant == choice),
#             'mismatch': mismatch_flag
#         }

#         trial_fix = (
#             fixations_for_plot[
#                 (fixations_for_plot['participant_id'] == participant) &
#                 (fixations_for_plot['trial_key'] == str(trial))
#             ]
#             .sort_values(['start_time', 'end_time'])
#         )

#         if trial_fix.empty:
#             continue

#         row_idx = conditions.index(cond) + 1
#         for _, fix in trial_fix.iterrows():
#             aoi = fix['AOI']
#             color = color_map.get(aoi, "red")
#             side = side_label_map.get(aoi, aoi.title())
#             seg_time = float(fix['duration_s'])
#             seg_start_s = float(fix['start_s'])

#             customdata = [[
#                 participant,
#                 item_label,
#                 trial,
#                 cond,
#                 side,
#                 seg_start_s,
#                 seg_time,
#                 dominant or 'N/A',
#                 choice or 'N/A',
#                 'Yes' if mismatch_flag else 'No'
#             ]]
#             fig.add_trace(
#                 go.Bar(
#                     x=[seg_time],
#                     y=[label],
#                     name=side,
#                     marker_color=color,
#                     orientation='h',
#                     showlegend=False,
#                     customdata=customdata,
#                     hovertemplate=(
#                         "<b>%{customdata[0]}</b><br>"
#                         "Condition: %{customdata[3]}<br>"
#                         "Item: %{customdata[1]}<br>"
#                         "Trial: %{customdata[2]}<br>"
#                         "Side: %{customdata[4]}<br>"
#                         "Segment start: %{customdata[5]:.2f}s<br>"
#                         "Segment duration: %{x:.2f}s<br>"
#                         "Dominant: %{customdata[7]}<br>"
#                         "Choice: %{customdata[8]}<br>"
#                         "Mismatch: %{customdata[9]}<extra></extra>"
#                     )
#                 ),
#                 row=row_idx, col=1
#             )

# # Format y-axis ticks per condition
# for i, cond in enumerate(conditions, start=1):
#     labels = facet_yticks[cond]

#     def format_label(lbl):
#         meta = facet_tick_meta[cond].get(lbl, {})
#         pid_short = meta.get('pid_short', 'NA')
#         choice_val = meta.get('choice', 'NA')
#         base = f"{pid_short} · {choice_val}"
#         bold = meta.get('dominant_matches', False)
#         mismatch = meta.get('mismatch', False)

#         if bold:
#             base = f"<span style='color:#88c0d0'>{base}</span>"
#         if mismatch:
#             inner = f"<s>{base}</s>"
#         else:
#             inner = base
#         return inner

#     ticktext = [format_label(t) for t in labels]
#     fig.update_yaxes(
#         categoryorder='array',
#         categoryarray=labels,
#         tickvals=labels,
#         ticktext=ticktext,
#         showgrid=False,
#         ticks='',
#         row=i, col=1
#     )

# fig.update_xaxes(title_text='Seconds', row=n_rows, col=1)

# fig.update_layout(
#     template='nord_light_paper',
#     barmode='stack',
#     bargap=0.1,
#     bargroupgap=0.0,
#     title='Eye-Tracking: Stacked Dwell Time Segments (Transitions)',
#     height=max(4800, 400 * n_rows),
#     # width=1000,
#     showlegend=True
# )

# fig.show()

# fig.write_html(plots / "dwell_times.html", include_plotlyjs='cdn')
# fig.write_image(plots / "dwell_times.png", scale=3)

#### Per Item

In [111]:
def plot_item_dwell(item_value,
                    events_df_df=None,
                    et_summary_df=None,
                    fixations_df=None,
                    skip_ids=None,
                    template='nord_light_paper',
                    save=True,
                    height_per_row=200):

    lf = events_df_df if events_df_df is not None else globals().get('events_df')
    et_summary = et_summary_df if et_summary_df is not None else globals().get('et_df')
    fixations = fixations_df if fixations_df is not None else globals().get('episodes')
    plots_dir = globals().get('plots')
    skip_ids = set(skip_ids if skip_ids is not None else globals().get('list_of_participants_to_remove', []))

    if lf is None or 'item' not in lf.columns:
        raise ValueError("events_df DataFrame with an 'item' column is required.")
    if et_summary is None or fixations is None:
        raise ValueError("et_summary_df and fixations_df (or globals et_df / episodes) are required.")

    lf_item = lf.loc[lf['item'].astype(str) == str(item_value)].copy()
    lf_item = lf_item[~lf_item['participant_id'].isin(skip_ids)]
    if lf_item.empty:
        raise ValueError(f"No trials found for item {item_value!r}.")

    # --- Force condition order top->bottom: focus, exclusive, contrastive ---
    preferred_order = ['unmodified', 'exhaustive', 'contrastive']

    raw_conds = lf_item['condition'].dropna().astype(str).tolist()
    uniq_conds = list(dict.fromkeys(raw_conds))  # stable unique

    lower_to_orig = {}
    for c in uniq_conds:
        lc = c.strip().lower()
        if lc not in lower_to_orig:
            lower_to_orig[lc] = c  # preserve original spelling

    conditions = (
        [lower_to_orig[c] for c in preferred_order if c in lower_to_orig] +
        [c for c in uniq_conds if c.strip().lower() not in set(preferred_order)]
    )

    if not conditions:
        raise ValueError(f"No conditions available for item {item_value!r}.")
    n_rows = max(1, len(conditions))

    # Tighter facet spacing (was 0.08)
    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=conditions,
        vertical_spacing=0.02
    )

    facet_yticks = {cond: [] for cond in conditions}
    facet_tick_meta = {cond: {} for cond in conditions}

    fix_plot = (
        fixations.copy()
        .loc[~fixations['participant_id'].isin(skip_ids)]
    )
    fix_plot['AOI'] = fix_plot['AOI'].astype(str).str.lower()
    fix_plot['start_time'] = pd.to_numeric(fix_plot['start_time'], errors='coerce')
    fix_plot['end_time'] = pd.to_numeric(fix_plot['end_time'], errors='coerce')
    fix_plot['duration_ms'] = pd.to_numeric(fix_plot['duration_ms'], errors='coerce')
    fix_plot = fix_plot.dropna(subset=['participant_id', 'trial', 'start_time', 'end_time'])
    fix_plot['duration_ms'] = fix_plot['duration_ms'].fillna(fix_plot['end_time'] - fix_plot['start_time'])
    fix_plot = fix_plot[fix_plot['duration_ms'] > 0]
    fix_plot['start_s'] = fix_plot['start_time'] / 1000.0
    fix_plot['duration_s'] = fix_plot['duration_ms'] / 1000.0
    fix_plot['trial_key'] = fix_plot['trial'].astype(str)
    fix_plot['pair_key'] = fix_plot['participant_id'].astype(str) + '||' + fix_plot['trial_key']

    selected_pairs = set(
        (lf_item['participant_id'].astype(str) + '||' + lf_item['trial'].astype(str)).dropna()
    )
    fix_plot = fix_plot[fix_plot['pair_key'].isin(selected_pairs)].copy()
    fix_plot.drop(columns=['pair_key'], inplace=True)

    item_participants = lf_item['participant_id'].unique()

    color_map = {
        'left': '#88c0d0',  # left AOI segments
        'right': "#b48ead", # right AOI segments
        'none': dark[2],
        'neutral': dark[3]
    }
    side_label_map = {
        'left': 'Left',
        'right': 'Right',
        'none': 'None',
        'neutral': 'None'
    }

    # Segment border so stacked bars are visually separated (WHITE EDGES)
    seg_border_color = 'rgba(255,255,255,1.0)'
    seg_border_width = 1.0

    for participant in item_participants:
        lf_part = lf_item[lf_item['participant_id'] == participant]
        trials = sorted(
            lf_part['trial'].dropna().unique(),
            key=lambda x: float(x) if str(x).replace('.', '', 1).isdigit() else x
        )
        df_part = et_summary[et_summary['participant_id'] == participant]

        for trial in trials:
            lf_row = lf_part[lf_part['trial'] == trial]
            if lf_row.empty:
                continue
            cond = lf_row['condition'].iloc[0]
            if pd.isna(cond) or cond not in conditions:
                continue

            item_label = lf_row['item'].iloc[0]
            label = f"{participant} - Item {item_label}"
            if label not in facet_yticks[cond]:
                facet_yticks[cond].append(label)

            et_row = df_part[df_part['trial'] == trial]
            dominant = et_row['dominant'].iloc[0] if not et_row.empty else None
            choice = lf_row['choice'].iloc[0] if not lf_row.empty else None

            mismatch_flag = False
            if 'mismatch' in lf_row.columns:
                try:
                    mismatch_flag = bool(lf_row['mismatch'].iloc[0])
                except Exception:
                    mismatch_flag = False

            facet_tick_meta[cond][label] = {
                'pid_short': str(participant)[-3:],
                'choice': choice if pd.notna(choice) else 'NA',
                'dominant_matches': (dominant is not None and choice is not None and dominant == choice),
                'mismatch': mismatch_flag
            }

            trial_fix = (
                fix_plot[
                    (fix_plot['participant_id'] == participant) &
                    (fix_plot['trial_key'] == str(trial))
                ]
                .sort_values(['start_time', 'end_time'])
            )
            if trial_fix.empty:
                continue

            row_idx = conditions.index(cond) + 1
            for _, fix in trial_fix.iterrows():
                aoi = fix['AOI']
                color = color_map.get(aoi, "red")
                side = side_label_map.get(aoi, aoi.title())
                seg_time = float(fix['duration_s'])
                seg_start_s = float(fix['start_s'])

                customdata = [[
                    participant,
                    item_label,
                    trial,
                    cond,
                    side,
                    seg_start_s,
                    seg_time,
                    dominant or 'N/A',
                    choice or 'N/A',
                    'Yes' if mismatch_flag else 'No'
                ]]
                fig.add_trace(
                    go.Bar(
                        x=[seg_time],
                        y=[label],
                        name=side,
                        marker=dict(
                            color=color,
                            line=dict(color=seg_border_color, width=seg_border_width)
                        ),
                        orientation='h',
                        showlegend=False,
                        customdata=customdata,
                        hovertemplate=(
                            "<b>%{customdata[0]}</b><br>"
                            "Condition: %{customdata[3]}<br>"
                            "Item: %{customdata[1]}<br>"
                            "Trial: %{customdata[2]}<br>"
                            "Side: %{customdata[4]}<br>"
                            "Segment start: %{customdata[5]:.2f}s<br>"
                            "Segment duration: %{x:.2f}s<br>"
                            "Dominant: %{customdata[7]}<br>"
                            "Choice: %{customdata[8]}<br>"
                            "Mismatch: %{customdata[9]}<extra></extra>"
                        )
                    ),
                    row=row_idx,
                    col=1
                )

    for i, cond in enumerate(conditions, start=1):
        labels = facet_yticks[cond]

        def format_label(lbl):
            meta = facet_tick_meta[cond].get(lbl, {})
            pid_short = meta.get('pid_short', 'NA')
            choice_val = meta.get('choice', 'NA')
            base = f"{pid_short} · {choice_val}"
            if meta.get('dominant_matches', False):
                base = f"<span style='color:#a3be8c'>{base}</span>"  # DOMINANT MATCHES
            if meta.get('mismatch', False):
                return f"<s>{base}</s>"
            return base

        ticktext = [format_label(t) for t in labels]
        fig.update_yaxes(
            categoryorder='array',
            categoryarray=labels,
            tickvals=labels,
            ticktext=ticktext,
            showgrid=False,
            ticks='',
            row=i,
            col=1
        )

    # --- Legend inside the grid (middle-right): add legend-only dummy traces ---
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=10, color=color_map.get('left', '#88c0d0')),
            name='Left',
            showlegend=True,
            hoverinfo='skip'
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=10, color=color_map.get('right', '#b48ead')),
            name='Right',
            showlegend=True,
            hoverinfo='skip'
        )
    )

    # Reduce bottom whitespace: small bottom margin + x-axis title standoff
    fig.update_xaxes(title_text='seconds', row=n_rows, col=1, title_standoff=0, automargin=False)

    fig.update_layout(
        template=template,
        barmode='stack',
        bargap=0.0,
        bargroupgap=0.0,
        title=dict(
            text=f"Stacked gaze episode segments (item {item_value})",
            x=0.5,
            xanchor='center',
            pad=dict(t=2, b=2)
        ),
        height=max(800, height_per_row * n_rows),
        width=400,
        showlegend=True,
        legend=dict(
            x=0.98,
            y=0.50,
            xanchor='right',
            yanchor='middle',
            orientation='v',
            bgcolor='rgba(255,255,255,0.75)',
            bordercolor='rgba(0,0,0,0)',
            borderwidth=1,
            itemsizing='constant',
        ),
        margin=dict(l=30, r=20, t=40, b=30),
    )

    # Pull subplot titles closer to their plots (and slightly smaller)
    fig.update_annotations(font=dict(size=12), yshift=-4)

    if save and plots_dir is not None:
        fig.write_html(plots_dir / f"dwell_times_item_{item_value}.html", include_plotlyjs='cdn')
        try:
            fig.write_image(plots_dir / f"dwell_times_item_{item_value}.png", scale=3)
        except Exception:
            pass

    return fig


# Usage
plot_item_dwell(
    item_value='8',
    events_df_df=events_df,
    et_summary_df=et_df,
    fixations_df=episodes,
    skip_ids=list_of_participants_to_remove,
    template='nord_light_paper',
    save=True,
    height_per_row=200
)

In [112]:
def plot_item_dwell_unmodified(item_value,
                               events_df_df=None,
                               et_summary_df=None,
                               fixations_df=None,
                               skip_ids=None,
                               template='nord_light_paper',
                               save=True,
                               height_per_row=200):

    lf = events_df_df if events_df_df is not None else globals().get('events_df')
    et_summary = et_summary_df if et_summary_df is not None else globals().get('et_df')
    fixations = fixations_df if fixations_df is not None else globals().get('episodes')
    plots_dir = globals().get('plots')
    skip_ids = set(skip_ids if skip_ids is not None else globals().get('list_of_participants_to_remove', []))

    if lf is None or 'item' not in lf.columns:
        raise ValueError("events_df DataFrame with an 'item' column is required.")
    if et_summary is None or fixations is None:
        raise ValueError("et_summary_df and fixations_df (or globals et_df / episodes) are required.")

    # Item filter
    lf_item = lf.loc[lf['item'].astype(str) == str(item_value)].copy()
    lf_item = lf_item[~lf_item['participant_id'].isin(skip_ids)]

    # Keep ONLY unmodified
    lf_item = lf_item[lf_item['condition'].astype(str).str.strip().str.lower().eq('unmodified')].copy()
    if lf_item.empty:
        raise ValueError(f"No unmodified trials found for item {item_value!r} (after skip_ids).")

    conditions = ['unmodified']
    n_rows = 1

    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=conditions,
        vertical_spacing=0.02
    )

    facet_yticks = {cond: [] for cond in conditions}
    facet_tick_meta = {cond: {} for cond in conditions}

    fix_plot = (
        fixations.copy()
        .loc[~fixations['participant_id'].isin(skip_ids)]
    )
    fix_plot['AOI'] = fix_plot['AOI'].astype(str).str.lower()
    fix_plot['start_time'] = pd.to_numeric(fix_plot['start_time'], errors='coerce')
    fix_plot['end_time'] = pd.to_numeric(fix_plot['end_time'], errors='coerce')
    fix_plot['duration_ms'] = pd.to_numeric(fix_plot['duration_ms'], errors='coerce')
    fix_plot = fix_plot.dropna(subset=['participant_id', 'trial', 'start_time', 'end_time'])
    fix_plot['duration_ms'] = fix_plot['duration_ms'].fillna(fix_plot['end_time'] - fix_plot['start_time'])
    fix_plot = fix_plot[fix_plot['duration_ms'] > 0]
    fix_plot['start_s'] = fix_plot['start_time'] / 1000.0
    fix_plot['duration_s'] = fix_plot['duration_ms'] / 1000.0
    fix_plot['trial_key'] = fix_plot['trial'].astype(str)
    fix_plot['pair_key'] = fix_plot['participant_id'].astype(str) + '||' + fix_plot['trial_key']

    # Only include fixations from the (participant, trial) pairs in *unmodified* lf_item
    selected_pairs = set(
        (lf_item['participant_id'].astype(str) + '||' + lf_item['trial'].astype(str)).dropna()
    )
    fix_plot = fix_plot[fix_plot['pair_key'].isin(selected_pairs)].copy()
    fix_plot.drop(columns=['pair_key'], inplace=True)

    item_participants = lf_item['participant_id'].unique()

    color_map = {
        'left': '#88c0d0',
        'right': "#b48ead",
        'none': dark[2],
        'neutral': dark[3]
    }
    side_label_map = {
        'left': 'Left',
        'right': 'Right',
        'none': 'None',
        'neutral': 'None'
    }

    seg_border_color = 'rgba(255,255,255,1.0)'
    seg_border_width = 1.0

    for participant in item_participants:
        lf_part = lf_item[lf_item['participant_id'] == participant]
        trials = sorted(
            lf_part['trial'].dropna().unique(),
            key=lambda x: float(x) if str(x).replace('.', '', 1).isdigit() else x
        )
        df_part = et_summary[et_summary['participant_id'] == participant]

        for trial in trials:
            lf_row = lf_part[lf_part['trial'] == trial]
            if lf_row.empty:
                continue

            # cond is always unmodified here, but keep metadata/hover consistent
            cond = 'unmodified'

            item_label = lf_row['item'].iloc[0]
            label = f"{participant} - Item {item_label}"
            if label not in facet_yticks[cond]:
                facet_yticks[cond].append(label)

            et_row = df_part[df_part['trial'] == trial]
            dominant = et_row['dominant'].iloc[0] if not et_row.empty else None
            choice = lf_row['choice'].iloc[0] if not lf_row.empty else None

            mismatch_flag = False
            if 'mismatch' in lf_row.columns:
                try:
                    mismatch_flag = bool(lf_row['mismatch'].iloc[0])
                except Exception:
                    mismatch_flag = False

            facet_tick_meta[cond][label] = {
                'pid_short': str(participant)[-3:],
                'choice': choice if pd.notna(choice) else 'NA',
                'dominant_matches': (dominant is not None and choice is not None and dominant == choice),
                'mismatch': mismatch_flag
            }

            trial_fix = (
                fix_plot[
                    (fix_plot['participant_id'] == participant) &
                    (fix_plot['trial_key'] == str(trial))
                ]
                .sort_values(['start_time', 'end_time'])
            )
            if trial_fix.empty:
                continue

            for _, fix in trial_fix.iterrows():
                aoi = fix['AOI']
                color = color_map.get(aoi, "red")
                side = side_label_map.get(aoi, aoi.title())
                seg_time = float(fix['duration_s'])
                seg_start_s = float(fix['start_s'])

                customdata = [[
                    participant,
                    item_label,
                    trial,
                    cond,
                    side,
                    seg_start_s,
                    seg_time,
                    dominant or 'N/A',
                    choice or 'N/A',
                    'Yes' if mismatch_flag else 'No'
                ]]
                fig.add_trace(
                    go.Bar(
                        x=[seg_time],
                        y=[label],
                        name=side,
                        marker=dict(
                            color=color,
                            line=dict(color=seg_border_color, width=seg_border_width)
                        ),
                        orientation='h',
                        showlegend=False,
                        customdata=customdata,
                        hovertemplate=(
                            "<b>%{customdata[0]}</b><br>"
                            "Condition: %{customdata[3]}<br>"
                            "Item: %{customdata[1]}<br>"
                            "Trial: %{customdata[2]}<br>"
                            "Side: %{customdata[4]}<br>"
                            "Segment start: %{customdata[5]:.2f}s<br>"
                            "Segment duration: %{x:.2f}s<br>"
                            "Dominant: %{customdata[7]}<br>"
                            "Choice: %{customdata[8]}<br>"
                            "Mismatch: %{customdata[9]}<extra></extra>"
                        )
                    ),
                    row=1,
                    col=1
                )

    # y-axis tick formatting (single facet: unmodified)
    cond = 'unmodified'
    labels = facet_yticks[cond]

    def format_label(lbl):
        meta = facet_tick_meta[cond].get(lbl, {})
        pid_short = meta.get('pid_short', 'NA')
        choice_val = meta.get('choice', 'NA')
        base = f"{pid_short} · {choice_val}"
        if meta.get('dominant_matches', False):
            base = f"<span style='color:#a3be8c'>{base}</span>"
        if meta.get('mismatch', False):
            return f"<s>{base}</s>"
        return base

    ticktext = [format_label(t) for t in labels]
    fig.update_yaxes(
        categoryorder='array',
        categoryarray=labels,
        tickvals=labels,
        ticktext=ticktext,
        showgrid=False,
        ticks='',
        row=1,
        col=1
    )

    # Legend-only dummy traces
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=10, color=color_map.get('left', '#88c0d0')),
            name='Left',
            showlegend=True,
            hoverinfo='skip'
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=10, color=color_map.get('right', '#b48ead')),
            name='Right',
            showlegend=True,
            hoverinfo='skip'
        )
    )

    fig.update_xaxes(title_text='seconds', row=1, col=1, title_standoff=0, automargin=False)

    fig.update_layout(
        template=template,
        barmode='stack',
        bargap=0.0,
        bargroupgap=0.0,
        title=dict(
            text=f"Stacked gaze episode segments (item {item_value}) — unmodified only",
            x=0.5,
            xanchor='center',
            pad=dict(t=2, b=2)
        ),
        height=max(800, height_per_row * 1),
        width=400,
        showlegend=True,
        legend=dict(
            x=0.98,
            y=0.50,
            xanchor='right',
            yanchor='middle',
            orientation='v',
            bgcolor='rgba(255,255,255,0.75)',
            bordercolor='rgba(0,0,0,0)',
            borderwidth=1,
            itemsizing='constant',
        ),
        margin=dict(l=30, r=20, t=40, b=30),
    )

    fig.update_annotations(font=dict(size=12), yshift=-4)

    if save and plots_dir is not None:
        fig.write_html(plots_dir / f"dwell_times_item_{item_value}_unmodified.html", include_plotlyjs='cdn')
        try:
            fig.write_image(plots_dir / f"dwell_times_item_{item_value}_unmodified.png", scale=3)
        except Exception:
            pass

    return fig


# Usage
plot_item_dwell_unmodified(
    item_value='8',
    events_df_df=events_df,
    et_summary_df=et_df,
    fixations_df=episodes,
    skip_ids=list_of_participants_to_remove,
    template='nord_light_paper',
    save=True,
    height_per_row=200
)

#### Per participant

In [113]:
def plot_single_participant_dwell(participant_id,
                                  events_df_df=None,
                                  et_summary_df=None,
                                  fixations_df=None,
                                  skip_ids=None,
                                  template='nord_light_paper',
                                  save=True,
                                  height_per_row=220):
    lf = events_df_df if events_df_df is not None else globals().get('events_df')
    et_summary = et_summary_df if et_summary_df is not None else globals().get('et_df')
    fixations = fixations_df if fixations_df is not None else globals().get('episodes')
    plots_dir = globals().get('plots')
    skip_ids = set(skip_ids if skip_ids is not None else globals().get('list_of_participants_to_remove', []))

    if lf is None or et_summary is None or fixations is None:
        raise ValueError("Provide events_df_df, et_summary_df and fixations_df or ensure globals exist.")
    if participant_id in skip_ids:
        raise ValueError(f"Participant {participant_id!r} is flagged for removal.")

    lf_participant = lf[lf['participant_id'] == participant_id].copy()
    if lf_participant.empty:
        raise ValueError(f"No trials for participant {participant_id!r}.")

    conditions = list(lf_participant['condition'].dropna().unique())
    if not conditions:
        raise ValueError(f"No conditions available for participant {participant_id!r}.")
    n_rows = max(1, len(conditions))

    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=conditions,
        vertical_spacing=0.08
    )

    facet_yticks = {cond: [] for cond in conditions}
    facet_tick_meta = {cond: {} for cond in conditions}

    fix_plot = (
        fixations
        .copy()
        .loc[fixations['participant_id'] == participant_id]
    )
    fix_plot['AOI'] = fix_plot['AOI'].astype(str).str.lower()
    fix_plot['start_time'] = pd.to_numeric(fix_plot['start_time'], errors='coerce')
    fix_plot['end_time'] = pd.to_numeric(fix_plot['end_time'], errors='coerce')
    fix_plot['duration_ms'] = pd.to_numeric(fix_plot['duration_ms'], errors='coerce')
    fix_plot = fix_plot.dropna(subset=['start_time', 'end_time', 'trial'])
    fix_plot['duration_ms'] = fix_plot['duration_ms'].fillna(fix_plot['end_time'] - fix_plot['start_time'])
    fix_plot = fix_plot[fix_plot['duration_ms'] > 0]
    fix_plot['start_s'] = fix_plot['start_time'] / 1000.0
    fix_plot['duration_s'] = fix_plot['duration_ms'] / 1000.0
    fix_plot['trial_key'] = fix_plot['trial'].astype(str)

    # nicer color choices and visible white segment borders like other plotting functions
    color_map = {
        'left': '#88c0d0',   # lighter blue (matches item plotting)
        'right': '#b48ead',  # muted purple/pink
        'none': dark[2],
        'neutral': dark[3]
    }
    side_label_map = {
        'left': 'Left',
        'right': 'Right',
        'none': 'None',
        'neutral': 'None'
    }

    # visual segment border
    seg_border_color = 'rgba(255,255,255,1.0)'
    seg_border_width = 1.0

    # font fallbacks from globals if present
    main_font_size = globals().get('FONT_MAIN', 16)
    tick_font_size = globals().get('FONT_TICKS', max(10, main_font_size - 4))
    hover_font_size = globals().get('FONT_HOVER', max(10, main_font_size - 4))
    serif_family = globals().get('SERIF_FAMILY', "Times New Roman, Times, Georgia, serif")

    sorted_trials = sorted(
        lf_participant['trial'].dropna().unique(),
        key=lambda x: float(x) if str(x).replace('.', '', 1).isdigit() else x
    )
    et_part = et_summary[et_summary['participant_id'] == participant_id]

    for trial in sorted_trials:
        lf_row = lf_participant[lf_participant['trial'] == trial]
        if lf_row.empty:
            continue
        cond = lf_row['condition'].iloc[0]
        if pd.isna(cond) or cond not in conditions:
            continue

        item_label = lf_row['item'].iloc[0] if 'item' in lf_row.columns else trial
        label = f"Item {item_label} · Trial {trial}"
        if label not in facet_yticks[cond]:
            facet_yticks[cond].append(label)

        et_row = et_part[et_part['trial'] == trial]
        dominant = et_row['dominant'].iloc[0] if not et_row.empty else None
        choice = lf_row['choice'].iloc[0] if not lf_row.empty else None

        if 'no' in lf_row.columns and pd.notna(lf_row['no'].iloc[0]):
            no_val = lf_row['no'].iloc[0]
        else:
            no_val = 'NA'

        mismatch_flag = False
        if 'mismatch' in lf_row.columns:
            try:
                mismatch_flag = bool(lf_row['mismatch'].iloc[0])
            except Exception:
                mismatch_flag = False

        facet_tick_meta[cond][label] = {
            'choice': choice if pd.notna(choice) else 'NA',
            'dominant_matches': (dominant is not None and choice is not None and dominant == choice),
            'mismatch': mismatch_flag,
            'no': no_val
        }

        trial_fix = (
            fix_plot[fix_plot['trial_key'] == str(trial)]
            .sort_values(['start_time', 'end_time'])
        )
        if trial_fix.empty:
            continue

        row_idx = conditions.index(cond) + 1
        for _, fix in trial_fix.iterrows():
            aoi = fix['AOI']
            color = color_map.get(aoi, "red")
            side = side_label_map.get(aoi, aoi.title())
            seg_time = float(fix['duration_s'])
            seg_start_s = float(fix['start_s'])

            customdata = [[
                participant_id,   # 0
                item_label,       # 1
                trial,            # 2
                cond,             # 3
                no_val,           # 4
                seg_start_s,      # 5
                seg_time,         # 6
                dominant or 'N/A',# 7
                choice or 'N/A',  # 8
                'Yes' if mismatch_flag else 'No'  # 9
            ]]
            fig.add_trace(
                go.Bar(
                    x=[seg_time],
                    y=[label],
                    name=side,
                    marker=dict(color=color, line=dict(color=seg_border_color, width=seg_border_width)),
                    orientation='h',
                    showlegend=False,
                    customdata=customdata,
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Condition: %{customdata[3]}<br>"
                        "Item: %{customdata[1]}<br>"
                        "Trial: %{customdata[2]}<br>"
                        "No: %{customdata[4]}<br>"
                        "Segment start: %{customdata[5]:.2f}s<br>"
                        "Segment duration: %{x:.2f}s<br>"
                        "Dominant: %{customdata[7]}<br>"
                        "Choice: %{customdata[8]}<br>"
                        "Mismatch: %{customdata[9]}<extra></extra>"
                    ),
                    hoverlabel=dict(font=dict(size=hover_font_size, family=serif_family))
                ),
                row=row_idx,
                col=1
            )

    # format y-axis tick labels with metadata (choice/dominant/mismatch)
    for i, cond in enumerate(conditions, start=1):
        labels = facet_yticks[cond]

        def format_label(lbl):
            meta = facet_tick_meta[cond].get(lbl, {})
            base = f"{meta.get('choice', 'NA')}"
            if meta.get('dominant_matches', False):
                base = f"<span style='color:#88c0d0'>{base}</span>"
            if meta.get('mismatch', False):
                return f"<s>{base}</s>"
            return base

        ticktext = [format_label(t) for t in labels]
        fig.update_yaxes(
            categoryorder='array',
            categoryarray=labels,
            tickvals=labels,
            ticktext=ticktext,
            showgrid=False,
            ticks='',
            tickfont=dict(size=tick_font_size, family=serif_family),
            row=i,
            col=1
        )

    fig.update_xaxes(title_text='Seconds', row=n_rows, col=1)
    fig.update_layout(
        template=template,
        barmode='stack',
        bargap=0.0,
        bargroupgap=0.0,
        title=f"Eye-Tracking: Dwell Time Segments · Participant {participant_id}",
        height=max(600, height_per_row * n_rows),
        width=900,
        showlegend=True,
        font=dict(size=main_font_size, family=serif_family),
        margin=dict(l=60, r=20, t=60, b=40),
        legend=dict(
            orientation='v',
            x=0.98,
            y=0.50,
            xanchor='right',
            yanchor='middle',
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='rgba(0,0,0,0)',
            borderwidth=0,
            itemsizing='constant'
        )
    )

    # add legend-only dummy traces for Left/Right to provide a clean legend
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                             marker=dict(size=10, color=color_map.get('left')),
                             name='Left', showlegend=True, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                             marker=dict(size=10, color=color_map.get('right')),
                             name='Right', showlegend=True, hoverinfo='skip'))

    # compact subplot titles / annotations
    if fig.layout.annotations:
        for a in fig.layout.annotations:
            a.font = dict(size=max(10, main_font_size - 2), family=serif_family)
            a.yshift = -6

    if save and plots_dir is not None:
        fig.write_html(plots_dir / f"dwell_times_{participant_id}.html", include_plotlyjs='cdn')
        try:
            fig.write_image(plots_dir / f"dwell_times_{participant_id}.png", scale=3)
        except Exception:
            pass

    return fig

# Usage
plot_single_participant_dwell(
    participant_id='participant_001',
    events_df_df=events_df,
    et_summary_df=et_df,
    fixations_df=episodes,
    skip_ids=list_of_participants_to_remove,
    template='nord_light_paper',
    save=True,
    height_per_row=200
)